# Operazioni preliminari

## Importazione delle Librerie

In questa cella carichiamo tutti gli strumenti necessari per il progetto. Le librerie sono organizzate in gruppi logici per chiarezza.

In [ ]:
# 1. LIBRERIE STANDARD E UTILITIES DI SISTEMA
import os
import sys
import gc
import json
import time
import random
import inspect
import logging
import itertools
import statistics
from math import ceil
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from typing import TypedDict, List, Optional, Final

# 2. DATA SCIENCE, MATEMATICA E METRICHE
import numpy as np
import pandas as pd
import scipy.stats

# Scikit-Learn Metrics
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    ConfusionMatrixDisplay
)

# 3. VISUALIZZAZIONE DATI
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as patches
import seaborn as sns

# 4. ELABORAZIONE IMMAGINI E AUGMENTATION
import cv2
from PIL import Image

# Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 5. DEEP LEARNING (PYTORCH)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Sampler

# Torchvision Models & Weights
from torchvision import models
from torchvision.models import resnet18, ResNet18_Weights

# Torch Utilities
from torchinfo import summary

# 6. PROGRESS BARS E INTERFACCIA
from tqdm import tqdm
from tqdm.auto import tqdm as tqdm_auto  # Alias utile per notebook

# Configurazione base logging (opzionale, per evitare errori se non configurato)
logging.basicConfig(level=logging.INFO)

## Gestione della Riproducibilità

Definiamo funzioni per consentire la riproducibilità:

1.  **`imposta_seed`**: Imposta il seed globale per la generazione di numeri casuali di Python, NumPy e PyTorch. È fondamentale per garantire che l'inizializzazione dei pesi della rete e le scelte casuali siano ripetibili.
2.  **`seed_worker` e `crea_generatore_riproducibile`**: Quando carichiamo le immagini in parallelo (usando più processi), c'è il rischio che l'ordine cambi ogni volta. Queste due funzioni "disciplinano" il caricamento dei dati, assicurando che le immagini vengano pescate sempre nella stessa sequenza.

In [ ]:
# Il type hint Final dichiara che il seed è una costante che non deve essere riassegnata
SEED_GLOBALE: Final[int] = 42

def imposta_seed(seed: int = SEED_GLOBALE, modalita_deterministica: bool = True) -> None:
    """
    Garantisce la riproducibilità dell'esperimento.
    
    Args:
        seed: Il seme per i generatori pseudo-casuali.
        modalita_deterministica: 
            - True: Massima riproducibilità.
            - False: Massima velocità.
    """
    # 1. Environment Variables

    # Disabilitiamo la randomizzazione dell'hash degli oggetti hashable (dizionari, per esempio).    
    os.environ['PYTHONHASHSEED'] = str(seed)
       
    # 2. Python & NumPy
    random.seed(seed)
    np.random.seed(seed)

    # 3. PyTorch (CPU & GPU)
    # Inizializziamo i pesi dei layer
    torch.manual_seed(seed)
    
    # Ci assicuriamo che il seed di CUDA sia propagato a tutte le GPU (usiamo 2 T4)
    torch.cuda.manual_seed_all(seed)

    # 4. Backend CuDNN
    # cuDNN testerebbe  diversi algoritmi di convoluzione
    # Ponendo torch.backends.cudnn.benchmark = False
    # usiamo l'algoritmo predefinito di cuDNN, garantendo 
    # una scelta deterministica dell'algoritmo    

    # torch.backends.cudnn.deterministic = True
    # forza cuDNN a utilizzare algoritmi che sono intrinsecamente deterministici
    
    if torch.cuda.is_available():
        if modalita_deterministica:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
            print(f"Seed {seed} | Modalità deterministica e riproducibile")
        else:
            torch.backends.cudnn.deterministic = False
            torch.backends.cudnn.benchmark = True
            print(f"Seed {seed} | Modalità rapida")

def worker_init_fn(worker_id: int) -> None:
    """
    Garantisce che ogni worker del DataLoader abbia un seed diverso ma deterministico.
    In assenza, ogni worker potrebbe applicare la stessa Data Augmentation.
    """
    # Pytorch genera seed a 64 bit
    # Numpy e Random accettano come seed solo interi unsigned a 32-bit.
   
    # % 2**32 restituisce il resto, portando il numero nell'intervallo [0, 2^32 - 1].
    # in un'operazione di modulo A % B, il valore massimo ottenibile è sempre B - 1.
    worker_seed = torch.initial_seed() % 2**32

    np.random.seed(worker_seed)
    random.seed(worker_seed)

def crea_generatore_riproducibile(seed: int = SEED_GLOBALE) -> torch.Generator:
    """Generatore isolato per DataLoader (evita side-effect sul random state globale)."""
    
    # Creiamo un'istanza separata del generatore PyTorch
    g = torch.Generator()

    g.manual_seed(seed)
    
    return g

# Applicazione immediata del seed
imposta_seed(SEED_GLOBALE, modalita_deterministica=True)

## Configurazione dei Percorsi e del File System

Nella cella seguente, definiamo i percorsi di lavoro su Kaggle in un'unica classe di configurazione `ConfigurazioneProgetto`.

Distinguiamo due aree principali:
1.  **Percorso di Input (`/kaggle/input`)**: È la cartella di **sola lettura** dove Kaggle monta ed estrae automaticamente il dataset LIVECell. Da qui possiamo solo leggere le immagini, non possiamo modificarle.
2.  **Percorso di Lavoro (`/kaggle/working`)**: È la cartella di **scrittura**. Qui salveremo tutto ciò che il nostro codice produce: i file CSV con gli indici, i grafici e i pesi del modello addestrato.

Infine, il comando `os.makedirs` crea preventivamente la cartella per gli indici, evitando che il codice vada in errore più avanti quando proveremo a salvarci dentro i file.

In [ ]:
@dataclass(frozen=True)
class ConfigurazioneProgetto:
    """
    Configurazione immutabile del progetto.
    frozen=True impedisce modifiche a runtime.
    """

    # Percorsi input/output
    PERCORSO_INPUT: Path = Path("/kaggle/input/livecell-dataset")
    PERCORSO_LAVORO: Path = Path("/kaggle/working")
    CARTELLA_INDICI: Path = PERCORSO_LAVORO / "indici"

    ESTENSIONE_TARGET: str = ".png"
    SPLIT: tuple[str, ...] = ("train", "val", "test")

    def __post_init__(self):
        # Creazione automatica cartelle output
        os.makedirs(self.CARTELLA_INDICI, exist_ok=True)

# Istanza globale di configurazione
CONFIG = ConfigurazioneProgetto()

## Indicizzazione del Dataset

Prima di poter addestrare i modelli, è conveniente creare una mappa di tutti i dati a nostra disposizione. Il dataset LIVECell è organizzato in cartelle nidificate (Split $\rightarrow$ Classe $\rightarrow$ Immagine). Abbiamo la necessità di rendere efficiente l'accesso alle immagini.

La funzione `scansiona_dataset` risolve questo problema:
1.  **Esplora** ricorsivamente tutte le cartelle (`train`, `val`, `test`).
2.  **Trova** tutte le immagini `.png` delle varie linee cellulari.
3.  **Crea un Indice**: Costruisce un **DataFrame Pandas** che opera da registro centrale. Per ogni immagine, salviamo il percorso assoluto, la classe di appartenenza (es. "A172") e lo split.

Inoltre, converte le etichette testuali (es. "SHSY5Y") in **numeri interi** (0, 1, 2...) in modo ordinato e deterministico. Questo passaggio è fondamentale perché le reti neurali lavorano con tensori numerici, non con stringhe di testo.

In [ ]:
def scansiona_dataset(percorso_radice: Path) -> tuple[pd.DataFrame, dict[str, int]]:
    """
    Costruisce l'indice del dataset e la mappatura deterministica delle classi.
    Mappa i percorsi assoluti delle immagini alle rispettive etichette e split, 
    garantendo ID numerici consistenti tramite ordinamento lessicografico.
    """
    
    record_trovati: list[dict[str, str | int]] = []

    print(f"Scansione nella directory: {percorso_radice}")

    # 1. Scansione
    for nome_split in CONFIG.SPLIT:
        cartella_split = percorso_radice / nome_split
        
        # Iteriamo sulle cartelle delle classi
        for cartella_classe in tqdm(list(cartella_split.iterdir()), desc=f"Split: {nome_split}"):
            if cartella_classe.is_dir():
                nome_etichetta = cartella_classe.name
                
                # Raccolta percorsi
                file_png = cartella_classe.glob(f"*{CONFIG.ESTENSIONE_TARGET}")
                
                for file_img in file_png:
                    record_trovati.append({
                        "percorso_immagine": str(file_img.absolute()),
                        "etichetta": nome_etichetta,
                        "split": nome_split
                    })

    # 2. DataFrame a partire dalla lista di dizionari
    df = pd.DataFrame(record_trovati)

    # 3. Encoding deterministico
    # Ordiniamo alfabeticamente le classi per garantire che ID=0 sia sempre la stessa classe
    etichette_uniche = sorted(df["etichetta"].unique())
    mappa_classi = {etichetta: idx for idx, etichetta in enumerate(etichette_uniche)}

    print(f"Classi individuate: {json.dumps(mappa_classi, indent=4)}")

    # 4. Applicazione della mappa di conversione
    df["indice_etichetta"] = df["etichetta"].map(mappa_classi).astype(int)

    return df, mappa_classi

## Analisi della Distribuzione dei Dati

Dopo aver indicizzato il dataset, è cruciale verificare che i dati siano stati letti correttamente e comprendere come sono distribuiti.

La funzione `statistiche_dataset` genera un report rapido che mostra:
1.  **Volume Totale**: Il numero complessivo di immagini trovate (per assicurarsi che nessuna sia andata persa).
2.  **Matrice di Distribuzione**: Una tabella pivot (Split $\times$ Etichetta) che ci permette di controllare il **bilanciamento delle classi**.
    *   Possiamo vedere immediatamente quante immagini abbiamo per ogni linea cellulare, con la distribuzione in Training, Validation e Test set.
    *   Questo ci aiuta a identificare eventuali classi sottorappresentate che potrebbero causare problemi di apprendimento.

In [ ]:
def statistiche_dataset(df: pd.DataFrame) -> None:
    """
    Stampa la tabella di contingenza per analizzare il bilanciamento delle classi.
    Usa margins=True per mostrare i totali.
    """
    print("\nAnalisi della distribuzione del dataset")
    print(f"Totale dei campioni indicizzati: {len(df)}")
    
    # pd.crosstab è una funzione specializzata nel conteggio di occorrenze di variabili categoriali
    # margins=True aggiunge la riga/colonna "All" per i totali
    tabella = pd.crosstab(
        index=df["etichetta"], 
        columns=df["split"], 
        margins=True, 
        margins_name="Totale"
    )

    # È equivalente a fare una pivot table
    # tabella = df.pivot_table(index="etichetta", columns="split", aggfunc="size")
    
    print("\nTabella delle frequenze (classi vs split):")
    print(tabella)
    print("-" * 40)

## Salvataggio e Persistenza degli Indici

Una volta completata la scansione, è fondamentale salvare i risultati su disco. Scansionare migliaia di file è un'operazione computazionalmente costosa, mentre leggere un singolo file CSV è immediato.

La funzione `salva_indici` cristallizza il lavoro fatto finora salvando due file nella cartella di lavoro (`/kaggle/working`):
1.  **`indice_livecell.csv`**: L'elenco completo di tutte le immagini con i relativi percorsi e etichette.
2.  **`mappatura_classi.json`**: Il "dizionario" che ci ricorda quale numero corrisponde a quale cellula (es. `0` $\to$ `A172`). Salvarlo è essenziale per poter decodificare le predizioni del modello in futuro.

Infine, viene chiamata la funzione che stampa le statistiche quantitive per avere un riscontro immediato sui dati appena salvati.

In [ ]:
def salva_indici(df: pd.DataFrame, mappa_classi: dict[str, int], cartella_output: Path) -> None:
    """
    Salva il dataset indicizzato e la mappatura classi su disco.
    Include la creazione della cartella per garantire l'atomicità della funzione.
    """

    # parents=True crea tutte le cartelle superiori mancanti
    # exist_ok=True evita che il codice crashi se la cartella esiste già
    cartella_output.mkdir(parents=True, exist_ok=True)

    percorso_csv = cartella_output / "indice_livecell.csv"
    percorso_json = cartella_output / "mappatura_classi.json"
    
    # 1. Salvataggio dell'indice CSV
    df.to_csv(percorso_csv, index=False)
    
    # 2. Salvataggio del JSON
    with open(percorso_json, "w") as f:
        json.dump(mappa_classi, f, indent=4)
        
    print(f"Indici salvati in: {cartella_output}")
    
    # 3. Verifica immediata dei dati salvati
    statistiche_dataset(df)

## Esecuzione e Caching

La cella seguente verifica se i file `indice_livecell.csv` e `mappatura_classi.json` esistono già nella cartella di lavoro. 
1.  Se i file esistono, li carica direttamente in memoria. Questo avviene in una frazione di secondo.
2.  Se i file non esistono viene avviata la scansione completa per salvare i risultati per il futuro.

In [ ]:
percorso_csv = ConfigurazioneProgetto.CARTELLA_INDICI / "indice_livecell.csv"
percorso_json = ConfigurazioneProgetto.CARTELLA_INDICI / "mappatura_classi.json"

if percorso_csv.exists() and percorso_json.exists():
    print("File di indicizzazione già presenti in Kaggle.")
    
    df_dati = pd.read_csv(percorso_csv)
    
    statistiche_dataset(df_dati)
    
else:
    print("File di indicizzazione non trovati.")
    
    df_dati, mappa_classi = scansiona_dataset(ConfigurazioneProgetto.PERCORSO_INPUT)
    
    salva_indici(
        df=df_dati,
        mappa_classi=mappa_classi,
        cartella_output=ConfigurazioneProgetto.CARTELLA_INDICI
    )

# Analisi esplorative

## Visualizzazione esplorativa delle immagini

Prima di progettare la rete neurale, è indispensabile comprendere la natura visiva delle immagini che andremo a processare.

La funzione `visualizza_campioni_per_classe` esegue un controllo qualitativo campionando casualmente **una immagine per ogni classe / linea cellulare**.
Questa visualizzazione ci permette di rispondere a domande cruciali:
1.  **Variabilità**: Quanto sono diverse le cellule tra loro?
2.  **Risoluzione**: Le immagini hanno tutte la stessa dimensione o variano?
3.  **Contenuto**: Quanto spazio spazio vuoto è presente? Questo influenzerà le nostre scelte di Data Augmentation (ad esempio le rotazioni).

In [ ]:
def visualizza_campioni_per_classe(df: pd.DataFrame, colonne: int = 4) -> None:
    """
    Visualizza un campione rappresentativo per ogni classe.
    Usa un seed fisso per garantire che la figura sia identica ad ogni esecuzione.
    """
    
    # 1. Campionamento deterministico con il seed globale
    campioni = df.groupby("etichetta").sample(n=1, random_state=SEED_GLOBALE).sort_values("etichetta")

    n_campioni = len(campioni)
    righe = ceil(n_campioni / colonne)
    
    plt.style.use('default')
    
    # Layout vincolato per evitare sovrapposizioni di testo
    fig, axes = plt.subplots(righe, colonne, figsize=(colonne*3, righe*3.5), constrained_layout=True)
    fig.suptitle(f'LIVECell Dataset: Campioni Rappresentativi (Seed {SEED_GLOBALE})', fontsize=14, y=1.05)
    
    # Gestione assi per griglie non piene
    axes_flat = axes.flatten()
    for ax in axes_flat: 
        ax.axis('off')

    for ax, row in zip(axes_flat, campioni.itertuples()):
        try:
            # Conversione esplicita in RGB per coerenza visuale
            with Image.open(row.percorso_immagine) as img:
                img_rgb = img.convert("RGB")
                w, h = img.size
                
                ax.imshow(img_rgb)
                ax.set_title(f"{row.etichetta}\n{w}x{h} px", fontsize=10, fontweight='bold')
                
        except Exception as e:
            ax.text(0.5, 0.5, "File Corrotto", ha='center', color='red')

    plt.show()
    plt.close(fig)

# Se df_dati non esiste, lo ricarichiamo.
if 'df_dati' not in locals():
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")

visualizza_campioni_per_classe(df_dati)

## Analisi delle immagini

Le immagini LIVECell derivano da microscopia a contrasto di fase, una tecnica che produce acquisizioni monocromatiche.
* la precedente visualizzazione ha mostrato che le immagini presentano un'elevata variabilità morfologica e di risoluzione.

Prima di definire la pipeline di preprocessing, verifichiamo empiricamente il formato dei file analizzando un campione casuale.

**Ipotesi:** le immagini sono memorizzate in formato grayscale (singolo canale L) e non RGB.

In [ ]:
def analizza_campione_microscopia(percorso: str) -> None:
    # Caricamento con OpenCV
    img = cv2.imread(percorso, cv2.IMREAD_UNCHANGED)
 
    # Analisi dinamica dei canali
    # Caso 1: Array 2D (H, W) -> Grayscale puro
    if len(img.shape) == 2:
        n_canali = 1
        modalita = "Grayscale"
    # Caso 2: Array 3D (H, W, C) -> Multicanale
    else:
        n_canali = img.shape[2]
        modalita = "Multichannel/Color"

    # Statistiche descrittive
    mean, std = img.mean(), img.std()
    
    print(f"Analisi del campione {Path(percorso).name}")
    print(f"Shape: {img.shape} | Dtype: {img.dtype}")
    print(f"Configurazione dei canali: {n_canali} ({modalita})")
    print(f"Statistiche pixel: media={mean:.2f}, dev. standard ={std:.2f}")
    print(f"Range intensità: [{img.min()}, {img.max()}]")

analizza_campione_microscopia(df_dati["percorso_immagine"].iloc[0])

In [ ]:
@dataclass(frozen=True)
class ConfigurazionePreprocessing:
    """
    Iperparametri che controllano la pipeline di trasformazione dei dati.
    Separati dalla configurazione fisica per facilitare gli esperimenti (A/B testing).
    """
    
    # Vincoli Architetturali (ResNet)
    DIM_IMMAGINE: int = 224 
    
    # μ e σ per la normalizzazione

    # Standard precisi di ImageNet. Non verranno utilizzati
    MEDIA_IMAGENET: tuple[float, float, float] = (0.485, 0.456, 0.406)
    DEV_STD_IMAGENET: tuple[float, float, float] = (0.229, 0.224, 0.225)

    # Calcolo sul training set di LiveCell (eseguito in locale con un piccolo script di utility)
    MEDIA_LIVECELL: tuple[float, float, float] = (0.188, 0.188, 0.188)
    DEV_STD_LIVECELL: tuple[float, float, float] = (0.250, 0.250, 0.250)

# Istanziazione globale per l'uso nelle pipeline
CONFIG_PREPROC = ConfigurazionePreprocessing()

print(f"Dimensione target: {CONFIG_PREPROC.DIM_IMMAGINE}x{CONFIG_PREPROC.DIM_IMMAGINE} px")
print(f"Normalizzazione con μ pari a {CONFIG_PREPROC.MEDIA_LIVECELL} pMedia) e σ pari a {CONFIG_PREPROC.DEV_STD_LIVECELL} (Dev. Std.)")

# Preprocessing delle immagini

## Pipeline di data augmentation (Albumentations)

In questa sezione definiamo le trasformazioni che vengono applicate alle immagini in tempo reale. Utilizziamo la libreria **Albumentations** per la sua velocità e per la classe `Compose` che combina più operazioni in un'unica pipeline.

In [ ]:
def visualizza_impatto_upscaling(df: pd.DataFrame, soglia_px: int = 25) -> None:
    """
    Cerca un'immagine a bassa risoluzione e visualizza gli artefatti generati dall'upscaling
    forzato a 224x224 (necessario per ResNet).
    """
    print(f"Ricerca immagine < {soglia_px}px per analisi artefatti upscaling...")


    # Facciamo shuffling delle righe senza omissioni
    # frac = frazione di righe da restituire; 1 = 100%
    candidati = df.sample(frac=1, random_state=SEED_GLOBALE)
    path_trovato = None
    dims_orig = (0, 0)

    # Image appartiene a PIL
    for path in candidati["percorso_immagine"]:
        with Image.open(path) as img:
            if img.width < soglia_px:
                path_trovato = path
                dims_orig = img.size
                break

    if not path_trovato:
        print("Nessuna immagine trovata sotto la soglia specificata.")
        return

    print(f"Trovata: {Path(path_trovato).name} ({dims_orig[0]}x{dims_orig[1]} px)")

    # 2. Caricamento e conversione RGB
    img_pil = Image.open(path_trovato).convert("RGB")

    # img_np.shape restituirà (31, 31, 3)
    img_np = np.array(img_pil)

    # 3. Confronto tra le interpolazioni
    target_dim = CONFIG_PREPROC.DIM_IMMAGINE
    metodi = {
        "Scala reale\n(Input originale)": None,
        "Nearest\n": cv2.INTER_NEAREST,
        "Linear\n": cv2.INTER_LINEAR,
        "Cubic\n": cv2.INTER_CUBIC,
    }

    fig, axes = plt.subplots(1, 4, figsize=(16, 5), constrained_layout=True)
    fig.suptitle(f'Impatto Upscaling: {dims_orig} $\\rightarrow$ {target_dim}x{target_dim}', fontsize=14)

    # flag opera come selettore di modalità. Se == None otteniamo l'immagine raw.
    for ax, (nome, flag) in zip(axes, metodi.items()):
        if flag is None:
            # Visualizzazione in scala reale su canvas nero
            canvas = np.zeros((target_dim, target_dim, 3), dtype=np.uint8)
            y_off = (target_dim - dims_orig[1]) // 2
            x_off = (target_dim - dims_orig[0]) // 2
            
            canvas[y_off:y_off+dims_orig[1], x_off:x_off+dims_orig[0]] = img_np
            ax.imshow(canvas)
            
            # Box rosso per evidenziare le dimensioni reali
            rect = patches.Rectangle((x_off - 0.5, y_off - 0.5), dims_orig[0], dims_orig[1], 
                                     linewidth=1, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
        else:
            # Upscaling effettivo
            res = cv2.resize(img_np, (target_dim, target_dim), interpolation=flag)
            ax.imshow(res)

        ax.set_title(nome, fontsize=10)
        ax.axis('off')

    plt.show()

# Esecuzione
if 'df_dati' not in locals():
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")
    
visualizza_impatto_upscaling(df_dati, soglia_px=25)

In [ ]:
def ottieni_trasformazioni(split: str) -> A.Compose:
    """
    Configura la pipeline di data augmentation ottimizzata per LIVECell.
    VERSIONE NO-CLAHE: Mantiene solo trasformazioni geometriche e normalizzazione
    per preservare le intensità originali dei pixel ed evitare rumore.
    """
    
    target_dim = CONFIG_PREPROC.DIM_IMMAGINE
    
    # Definiamo la pipeline differenziata per il training set
    if split == "train":
        # Inizializziamo la lista delle trasformazioni sequenziali
        pipeline = [
            # BLOCCO 1: Invarianza geometrica lossless
            # Applica trasformazioni che non alterano la qualità dei pixel.
            A.OneOf([
                A.HorizontalFlip(p=1.0),
                A.VerticalFlip(p=1.0),
                A.RandomRotate90(p=1.0),
            ], p=0.7),
            
            # BLOCCO 2: Affine
            # Applichiamo qui le deformazioni fini.
            A.Affine(
                scale=(0.8, 1.2),
                translate_percent=(0.1, 0.1),
                rotate=(-30, 30),
                fill=0,
                fill_mask=0,
                border_mode=cv2.BORDER_CONSTANT,
                interpolation=cv2.INTER_LINEAR,
                p=0.3
            ),

            # BLOCCO 3: Resize Deterministico
            A.Resize(
                height=target_dim, 
                width=target_dim,
                interpolation=cv2.INTER_CUBIC 
            ),
            
            # RIMOSSO: BLOCCO 4 (CLAHE)
            # Abbiamo rimosso l'equalizzazione dell'istogramma.
        ]
        
    # Pipeline per Validation e Test (solo deterministica)
    else:
        pipeline = [
            # Ridimensioniamo l'immagine alla dimensione target
            A.Resize(
                height=target_dim, 
                width=target_dim, 
                interpolation=cv2.INTER_CUBIC
            ),
            # RIMOSSO: CLAHE anche qui per coerenza
        ]

    # BLOCCO FINALE: Normalizzazione e Tensore
    pipeline.extend([
        # Normalizziamo i pixel
        A.Normalize(
            mean=CONFIG_PREPROC.MEDIA_LIVECELL,
            std=CONFIG_PREPROC.DEV_STD_LIVECELL,
            max_pixel_value=255.0,
        ),
        # Convertiamo in tensore PyTorch
        ToTensorV2()
    ])

    return A.Compose(pipeline)

# Creazione del Dataset (PyTorch)

In [ ]:
class LIVECellDataset(Dataset):
    """
    Dataset PyTorch ottimizzato.
    Gestisce il caricamento da disco e la gestione degli errori I/O.
    """

    def __init__(self, df: pd.DataFrame, trasformazioni: A.Compose):
        # Convertiamo subito in lista per accesso O(1) rapido
        self.percorsi = df['percorso_immagine'].tolist()
        
        # PyTorch CrossEntropyLoss richiede target di tipo int64 (torch.LongTensor)
        # Convertiamo subito per evitare casting a runtime.
        self.etichette = df['indice_etichetta'].to_numpy(dtype=np.int64)
        
        self.trasformazioni = trasformazioni

    def __len__(self) -> int:
        return len(self.percorsi)

    def __getitem__(self, indice: int) -> tuple[torch.Tensor, int]:
        percorso = self.percorsi[indice]
        etichetta = self.etichette[indice]
        
        # 1. Caricamento Grayscale
        img = cv2.imread(percorso, cv2.IMREAD_GRAYSCALE)       
        
        # 2. Adattamento Dominio (Grayscale -> RGB)
        # ResNet è pre-addestrata su ImageNet (a colori)
        # Replichiamo il canale grigio sui 3 canali RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        # 3. Augmentation & Tensorizzazione
        # Albumentations gestisce la conversione in torch.Tensor e la normalizzazione
        augmented = self.trasformazioni(image=img_rgb)

        # quando usiamo ToTensorV2()
        # il dizionario restituito contiene sotto la chiave "image" un oggetto torch.Tensor
        return augmented["image"], etichetta

# Creazione degli Split e Istanziazione del Dataset

In questa fase trasformiamo il DataFrame contenente tutto l'indice in tre oggetti `Dataset` PyTorch distinti, pronti per essere utilizzati nel loop di training.

L'operazione segue questi passaggi logici:
1.  **Definizione delle pipeline**: Istanziamo le due pipeline di augmentation definite in precedenza.
2.  **Filtraggio del DataFrame**: Dividiamo i dati usando la colonna `split` che avevamo creato durante la scansione. Usiamo `.copy()` per evitare i *SettingWithCopyWarning* di Pandas e garantire che ogni dataset abbia il suo spazio di memoria indipendente.
3.  **Verifica Integrità**: Alla fine stampiamo un breve report esplorativo.

In [ ]:
# 1. Caricamento Dati
if 'df_dati' not in locals():
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")

# 2. Setup delle trasformazioni
pipeline_train = ottieni_trasformazioni("train")
pipeline_eval = ottieni_trasformazioni("val") # Usata sia per Validation che per Test

# 3. Creazione dei subset
df_train = df_dati[df_dati["split"] == "train"].copy()
df_val = df_dati[df_dati["split"] == "val"].copy()
df_test = df_dati[df_dati["split"] == "test"].copy()

# 4. Istanziazione dei Dataset PyTorch
ds_train = LIVECellDataset(df_train, trasformazioni=pipeline_train)
ds_val = LIVECellDataset(df_val, trasformazioni=pipeline_eval)
ds_test = LIVECellDataset(df_test, trasformazioni=pipeline_eval)

# 5. Report e check di integrità 
print("Riepilogo Dataset PyTorch")
print(f"Train: {len(ds_train):>6} campioni")
print(f"Val:   {len(ds_val):>6} campioni")
print(f"Test:  {len(ds_test):>6} campioni")

# Analisi della ResNet-18

Prima di procedere con il fine-tuning, è fondamentale comprendere la struttura interna della rete che utilizzeremo come backbone. Utilizziamo `torchinfo` per generare un sommario dettagliato dei layer e del flusso dei tensori.

In [ ]:
# 1. Carichiamo una ResNet18 "pulita" per analizzarne l'anatomia
# (Impostiamo weights=None perché ci interessa la struttura, non i pesi)
resnet_anatomia = models.resnet18(weights=None)

# 2. Visualizziamo il sommario
# input_size=(Batch_Size, Canali, Altezza, Larghezza)
# depth=3 ci permette di vedere dentro i "BasicBlock" senza scendere troppo nei dettagli
print(f"Analisi Strutturale ResNet18")
print("=" * 80)

summary(
    model=resnet_anatomia, 
    input_size=(1, 3, 224, 224), # Simuliamo un'immagine standard
    col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"],
    depth=3,
    col_width=18,
    row_settings=["var_names"] # Mostra i nomi delle variabili nel codice sorgente
)

In [ ]:
def ispeziona_gerarchico(modulo: nn.Module, indent: int = 0, nome_padre: str = ""):
    """
    Naviga il modello ricorsivamente preservando la gerarchia visiva di PyTorch.
    Sostituisce la rappresentazione standard dei layer Conv/Pool con dettagli geometrici.
    """
    
    # Iteriamo solo sui figli diretti (non su tutti i moduli appiattiti)
    for nome, layer in modulo.named_children():
        
        # Creiamo l'indentazione visiva
        prefix = "  " * indent
        nome_completo = f"({nome})"
        
        # CASO 1: Layer "Foglia" (Conv, Pool) che vogliamo dettagliare
        if isinstance(layer, (nn.Conv2d, nn.MaxPool2d, nn.AdaptiveAvgPool2d)):
            dettagli = ""
            
            if isinstance(layer, nn.Conv2d):
                k = layer.kernel_size if isinstance(layer.kernel_size, tuple) else (layer.kernel_size,)*2
                s = layer.stride if isinstance(layer.stride, tuple) else (layer.stride,)*2
                p = layer.padding if isinstance(layer.padding, tuple) else (layer.padding,)*2
                # Formattazione compatta stile "Datasheet"
                dettagli = f"► Conv2d | In->Out: {layer.in_channels}->{layer.out_channels} | K: {k[0]}x{k[1]} | S: {s[0]}x{s[1]} | Pad: {p[0]}x{p[1]}"
                
            elif isinstance(layer, nn.MaxPool2d):
                k = layer.kernel_size if isinstance(layer.kernel_size, tuple) else (layer.kernel_size,)*2
                s = layer.stride if isinstance(layer.stride, tuple) else (layer.stride,)*2
                p = layer.padding if isinstance(layer.padding, tuple) else (layer.padding,)*2
                dettagli = f"► MaxPool | K: {k[0]}x{k[1]} | S: {s[0]}x{s[1]} | Pad: {p[0]}x{p[1]}"
                
            elif isinstance(layer, nn.AdaptiveAvgPool2d):
                dettagli = f"► AdaptiveAvgPool | Output: {layer.output_size}"

            print(f"{prefix}{nome_completo:<10} {dettagli}")
            
        # CASO 2: Contenitori (Sequential, BasicBlock, Bottleneck)
        else:
            # Stampiamo il nome del blocco (es. "Sequential") e scendiamo di livello
            print(f"{prefix}{nome_completo}: {layer.__class__.__name__}")
            ispeziona_gerarchico(layer, indent + 1)

# Esecuzione
resnet = models.resnet18(weights=None)
print(f"{'STRUTTURA':<30} {'DETTAGLI GEOMETRICI'}")
print("-" * 100)
ispeziona_gerarchico(resnet)

### Punti chiave dall'output:
*   **Backbone:** La rete è composta da 4 stadi principali (`layer1`...`layer4`), ciascuno contenente blocchi residui (`BasicBlock`).
*   **Feature Vector:** Al termine dei layer convoluzionali e dell'`AdaptiveAvgPool2d`, otteniamo un tensore di dimensione **`[Batch, 512, 1, 1]`**. Questo vettore di 512 numeri è la "firma digitale" compressa dell'immagine.
*   **Testa di classificazione (da modificare):** L'ultimo layer visibile è `Linear (fc): [1, 512] -> [1, 1000]`. Questo è configurato per ImageNet (1000 classi).

<img src="https://www.researchgate.net/profile/Muhammed-Enes-Atik-2/publication/349241995/figure/fig2/AS:991139192643586@1613317406497/ResNet-18-architecture-20-The-numbers-added-to-the-end-of-ResNet-represent-the.png"
     width="600">

# Few-Shot Learning con Prototypical Networks

## Configurazione dell'esperimento e degli iperparametri

In [ ]:
# 1. Definizione percorsi e caricamento mappatura
PERCORSO_MAPPATURA = Path("/kaggle/working/indici/mappatura_classi.json")
PERCORSO_JSON_MANUALE = CONFIG.CARTELLA_INDICI / "folds_manuali.json"

# Controllo esistenza del file di mappatura sorgente
if not PERCORSO_MAPPATURA.exists():
    raise FileNotFoundError(f"File di mappatura non trovato in: {PERCORSO_MAPPATURA}")

if PERCORSO_JSON_MANUALE.exists():
    print(f"Configurazione manuale dei fold gia presente: {PERCORSO_JSON_MANUALE}")
else:
    # 2. Caricamento dinamico della mappatura delle classi
    with open(PERCORSO_MAPPATURA, 'r', encoding='utf-8') as f:
        mappa_classi = json.load(f)

    # Definizione delle relazioni semantiche per la validazione Out-of-Distribution
    fold_semantici = [
        {"descrizione": "Seno (BT-474 & SkBr3)", "novel_names": ["BT474", "SkBr3"]},
        {"descrizione": "Cervello (A172 & SHSY5Y)", "novel_names": ["A172", "SHSY5Y"]},
        {"descrizione": "Seno (BT-474 & MCF7)", "novel_names": ["BT474", "MCF7"]},
        {"descrizione": "Cervello (A172 & BV2)", "novel_names": ["A172", "BV2"]},
        {"descrizione": "Ghiandolari (MCF7 & SKOV3)", "novel_names": ["MCF7", "SKOV3"]}
    ]

    # 3. Pipeline di trasformazione
    configurazione_folds = []

    print(f"Inizializzazione folds manuali in {CONFIG.CARTELLA_INDICI}...")
    for i, fold in enumerate(fold_semantici):
        nomi = fold["novel_names"]
        
        # Mapping dei nomi alle label intere tramite il dizionario caricato
        # Esempio: "A172" -> 0, "SHSY5Y" -> 5 => ids = [0, 5]
        try:
            ids = sorted([mappa_classi[nomi[0]], mappa_classi[nomi[1]]])
        except KeyError as e:
            print(f"Errore: La classe {e} definita nei fold non esiste nel file di mappatura.")
            raise

        fold_data = {
            "fold_idx": i + 1,
            "descrizione": fold["descrizione"],
            "novel_classes_names": nomi,
            "novel_classes_ids": ids,
            "fold_id_univoco": f"{ids[0]}-{ids[1]}"
        }
        configurazione_folds.append(fold_data)
        print(f"  [Fold {i+1}] Registrato: {ids} ({nomi[0]}, {nomi[1]})")

    # 4. Serializzazione JSON dei metadati strutturati
    with open(PERCORSO_JSON_MANUALE, "w", encoding='utf-8') as f:
        json.dump(configurazione_folds, f, indent=4)

    print(f"\nGenerazione completata. File salvato in: {PERCORSO_JSON_MANUALE}")

In [ ]:
@dataclass
class ConfigProtoNet:
    """
    Configurazione dinamica dell'esperimento Few-Shot.
    Genera automaticamente nomi e cartelle basandosi su CONFIG e timestamp.
    """
    
    # Meta-Training
    TRAIN_N_WAY: int = 4      
    TRAIN_K_SHOT: int = 5     
    TRAIN_Q_QUERY: int = 15   
    EPISODI_TRAIN: int = 300  
    
    # Meta-Validation
    VAL_N_WAY: int = 2
    VAL_K_SHOT: int = 5
    VAL_Q_QUERY: int = 15
    EPISODI_VAL: int = 100    
    
    # Meta-Testing
    TEST_N_WAY: int = 2       
    TEST_K_SHOT: int = 5      
    TEST_Q_QUERY: int = 15    
    EPISODI_TEST: int = 600  
    
    # Ottimizzazione
    EPOCHE_PER_FOLD: int = 10 
    LEARNING_RATE: float = 1e-4
    
    # Patience indica il numero massimo di epoche consecutive senza miglioramenti nella Validation Loss
    # prima di interrompere il training (Early Stopping)
    PATIENCE: int = 5
    
    # worker = processo parallelo indipendente (figlio del processo principale)
    # dedicato esclusivamente al pre-processing dei dati
    # 2 processi figli aggiuntivi rispetto al processo principale (main process)
    # lavorano in parallelo allo script principale
    NUM_WORKERS: int = 2      

    # PIN_MEMORY velocizza il passaggio dei dati RAM -> GPU eliminando le copie intermedie della CPU.
    PIN_MEMORY: bool = True
    
    # Campi utili al naming delle cartelle
    # init=False perché vengono calcolati nel post_init
    NOME_ESPERIMENTO: str = field(init=False)
    CARTELLA_OUTPUT: Path = field(init=False)
    FILE_CHECKPOINT: Path = field(init=False)

    def __post_init__(self):
        # 1. Timestamp (YYYYMMDD_HHMM) per ordinamento cronologico
        timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")

        
        # 2. Codifica parametri chiave nel nome (Tr=Train, W=Way, S=Shot)
        params_str = f"TTA_8D4_ResNet18_Lin_256_NO_Clahe2.0{self.TRAIN_N_WAY}W_Val{self.VAL_N_WAY}W_Test{self.TEST_N_WAY}W"
        
        # 3. Costruzione nome univoco
        # Es: "20231025_1430_ProtoNet_Tr4W5S"
        self.NOME_ESPERIMENTO = f"{timestamp}_ProtoNet_{params_str}"
        
        # 4. Creazione path (CONFIG.PERCORSO_LAVORO)
        self.CARTELLA_OUTPUT = CONFIG.PERCORSO_LAVORO / "esperimenti" / self.NOME_ESPERIMENTO
        
        # 5. Definizione percorso del modello migliore
        self.FILE_CHECKPOINT = self.CARTELLA_OUTPUT / "best_model.pth"
        
        # Creazione fisica della cartella
        self.CARTELLA_OUTPUT.mkdir(parents=True, exist_ok=True)

# Creiamo l'istanza
PROTO_CONFIG = ConfigProtoNet()

print(f"Nome Esperimento: {PROTO_CONFIG.NOME_ESPERIMENTO}")
print(f"Salvataggio in: {PROTO_CONFIG.CARTELLA_OUTPUT}")
print(f"Checkpoint:     {PROTO_CONFIG.FILE_CHECKPOINT}")

## Analisi Episodica del Batch (Few-Shot Learning)

Il `CampionatoreEpisodico` non estrae dati in modo casuale, ma organizza ogni batch come un mini task di apprendimento. 
Secondo la configurazione `PROTO_CONFIG` (4-Way, 5-Shot, 15-Query), ecco come i dati vengono strutturati.

### 1. Selezione delle Classi (N-Way)
Supponiamo che il sampler scelga casualmente 4 linee cellulari:
* **Classe 4**: MCF7
* **Classe 0**: A172
* **Classe 6**: SKOV3
* **Classe 3**: Huh7

### 2. Struttura del Batch Indici (DataLoader)
Il batch restituito è una sequenza deterministica di **80 indici** (4 classi × 20 immagini per classe). L'ordine è fondamentale per permettere alla rete di separare il **Support Set** (per creare i prototipi) dal **Query Set** (per il calcolo della loss).

| Segmento Indici | ID Classe | Nome Linea Cellulare | Ruolo Tecnico | Campioni |
| :--- | :---: | :--- | :--- | :---: |
| **[0 : 5]** | **4** | MCF7 | **Support Set** (K-Shot) | 5 |
| **[5 : 20]** | **4** | MCF7 | **Query Set** (Q-Query) | 15 |
| **[20 : 25]** | **0** | A172 | **Support Set** (K-Shot) | 5 |
| **[25 : 40]** | **0** | A172 | **Query Set** (Q-Query) | 15 |
| **[40 : 45]** | **6** | SKOV3 | **Support Set** (K-Shot) | 5 |
| **[45 : 60]** | **6** | SKOV3 | **Query Set** (Q-Query) | 15 |
| **[60 : 65]** | **3** | Huh7 | **Support Set** (K-Shot) | 5 |
| **[65 : 80]** | **3** | Huh7 | **Query Set** (Q-Query) | 15 |

---

### 3. Rappresentazione nello Spazio Latente (Embedding Space)
Dopo il passaggio nella **ResNet18**, ogni immagine diventa un vettore numerico di dimensione 512. La ProtoNet opera su questi vettori:



| Componente | Shape del Tensore | Descrizione Operativa |
| :--- | :---: | :--- |
| **Support Embeddings** | `(20, 512)` | I 5 vettori per classe usati come "esempi guida". |
| **Prototypes ($c_k$)** | `(4, 512)` | La **media aritmetica** dei 5 vettori di supporto per ogni classe. |
| **Query Embeddings** | `(60, 512)` | Le immagini "test" da classificare in base alla distanza dai prototipi. |

---

### 4. Classificazione via Distanza Euclidea
Per ogni immagine Query, la rete calcola la distanza dai 4 prototipi generati. La classe predetta è quella col prototipo più vicino.

| Esempio Query | Distanza da $c_{MCF7}$ | Distanza da $c_{A172}$ | Distanza da $c_{SKOV3}$ | Distanza da $c_{Huh7}$ | Predizione Finale |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Query #1** (MCF7) | **0.84** (Min) | 12.40 | 15.10 | 11.20 | **4 (Correct)** |
| **Query #25** (A172) | 10.20 | **1.15** (Min) | 9.80 | 14.50 | **0 (Correct)** |

> **Obiettivo del Training**: Minimizzare la distanza tra le Query e il loro prototipo corrispondente, massimizzando al contempo la distanza dai prototipi delle altre classi.

In [ ]:
class CampionatoreEpisodico(Sampler):
    """
    Sampler per Few-Shot Learning.
    Genera batch strutturati (N-Way, K-Shot) per l'addestramento episodico.
    
    Struttura Batch:
    [Classe A (Support+Query), Classe B (Support+Query), ...]
    """
    def __init__(self, etichette: np.ndarray, n_way: int, k_shot: int, q_query: int, numero_episodi: int):
        """
        Args:
            etichette: Array numpy con le label di tutto il dataset.
            n_way: Numero di classi per episodio.
            k_shot: Immagini di supporto per classe.
            q_query: Immagini di query per classe.
            numero_episodi: Totale episodi da generare (lunghezza epoch).
        """
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.samples_per_class = k_shot + q_query
        self.numero_episodi = numero_episodi
        
        # Troviamo le classi e i loro indici velocemente
        self.classi_uniche = np.unique(etichette)
        
        # Mappa pre-calcolata: Classe -> array di indici
        # Questo evita di fare ricerche costose a ogni iterazione
        self.indici_per_classe = {}
        for c in self.classi_uniche:
            # np.where restituisce una tupla, prendiamo il primo elemento
            self.indici_per_classe[c] = np.where(etichette == c)[0]

        """Otteniamo ad esempio self.indici_per_classe = {
                                        4: array([0, 2, 4]),  # Posizioni delle cellule MCF7
                                        0: array([1]),        # Posizione della cellula A172
                                        6: array([3])         # Posizione della cellula SKOV3 }"""

    def __len__(self):
        return self.numero_episodi

    def __iter__(self):
        for _ in range(self.numero_episodi):
            batch = []
            
            # 1. Campionamento classi (N-Way) senza reinserimento
            # random.choice ha parametri posizionali:
            # a: l'array da cui estrarre i campioni
            # size: N-Way
            # replace=False: garantisce classi distinte nell'episodio
            classi_scelte = np.random.choice(self.classi_uniche, self.n_way, replace=False)
            
            # 2. Campionamento immagini per classe (K+Q)
            for c in classi_scelte:
                indici_disponibili = self.indici_per_classe[c]
                            
                indici_scelti = np.random.choice(
                    indici_disponibili, 
                    self.samples_per_class, # k_shot + q_query
                    replace=False
                )
                batch.extend(indici_scelti)
            
            yield batch

def filtra_dataset(df: pd.DataFrame, classi_target: tuple[int, ...]) -> pd.DataFrame:
    """
    Restituisce un sottoinsieme del DataFrame contenente solo le classi specificate.
    Essenziale per dividere Base Classes (Train) e Novel Classes (Test).
    """
    return df[df["indice_etichetta"].isin(classi_target)].copy()

# Configurazione degli esperimenti tramite nn.Module

## Esperimenti A e B

In [ ]:
'''class PrototypicalResNet(nn.Module):
    """
    ESPERIMENTO A E B
    
    Encoder di embedding per Prototypical Networks su LIVECell.
    CONFIGURAZIONE BEST RUN:
    - Backbone: ResNet18
    - Head: Linear Projector (512 -> 256)
    - Output: RAW (Nessuna Normalizzazione L2, Nessuno Scaling)
    """
    def __init__(self, pretrained: bool = True, output_dim: int = 256):
        super().__init__()
        
        # Carichiamo i pesi della rete pretrained su ImageNet
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None

        # Inizializziamo la ResNet18 standard
        self.backbone = resnet18(weights=weights)
        
        # Sostituiamo il layer Fully Connected finale con Identity
        self.backbone.fc = nn.Identity()

        # 2. Definizione del layer di proiezione lineare
        # Proietta da 512 (feature ResNet) a 256 (spazio metrico)
        self.projection_layer = nn.Linear(in_features=512, out_features=output_dim)

        # RIMOSSO: self.scale (Non serve senza normalizzazione)

        
    def forward(self, x):
        # 1. Backbone
        out = self.backbone(x)
        
        # 2. Projection Head (Lineare)
        out_refined = self.projection_layer(out)
        
        # 3. Output RAW
        # Ritorniamo direttamente il vettore non normalizzato.
        # La rete utilizzerà la distanza Euclidea pura (inclusa la magnitudo).
        return out_refined
'''

## Esperimento C

In [ ]:
# ==============================================================================
# ESPERIMENTO C: L2 NORM + LEARNABLE SCALING
# ==============================================================================
class PrototypicalResNet(nn.Module):
    """
    Encoder di embedding per Prototypical Networks.
    VARIANTE: L2 Normalization + Learnable Scaling (Alpha).
    """
    def __init__(self, pretrained: bool = True, output_dim: int = 256):
        super().__init__()
        
        # 1. Backbone
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = resnet18(weights=weights)
        self.backbone.fc = nn.Identity()

        # 2. Projection Head
        self.projection_layer = nn.Linear(in_features=512, out_features=output_dim)

        # 3. Learnable Scaling Factor (Alpha)
        # Inizializziamo a 10.0. Questo valore verrà ottimizzato durante il training.
        # Serve a gestire il range dinamico della Softmax dopo la normalizzazione.
        self.scale = nn.Parameter(torch.tensor(10.0))

    def forward(self, x):
        # Feature extraction
        out = self.backbone(x)
        
        # Projezione nello spazio metrico
        out = self.projection_layer(out)
        
        # A. L2 Normalization
        # Proietta tutti i vettori su una sfera unitaria (lunghezza = 1)
        out = F.normalize(out, p=2, dim=1)
        
        # B. Scaling
        # Moltiplichiamo per alpha. 
        # Matematicamente: ||a*x - a*y||^2 = a^2 * ||x - y||^2
        # La rete impara quanto devono essere "distanti" i cluster per minimizzare la loss.
        return out * self.scale
        

## Esperimento D

In [ ]:
'''
# ==============================================================================
# ESPERIMENTO D: MLP HEAD (NON-LINEAR) - NO L2 NORM
# ==============================================================================
#class PrototypicalResNet(nn.Module):
    """
    Encoder di embedding per Prototypical Networks.
    VARIANTE: MLP Head (Linear -> ReLU -> Dropout -> Linear).
    OUTPUT: RAW (Euclidean Distance pura, come nella Best Run).
    """
    def __init__(self, pretrained: bool = True, output_dim: int = 256):
        super().__init__()
        
        # 1. Backbone
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = resnet18(weights=weights)
        self.backbone.fc = nn.Identity()

        # 2. Projection Head: MLP (Multi-Layer Perceptron)
        # Sostituiamo il singolo layer lineare con una sequenza non-lineare.
        # Questo permette alla rete di modellare relazioni più complesse tra le feature
        # prima di proiettarle nello spazio metrico.
        self.projection_layer = nn.Sequential(
            nn.Linear(512, 512),        # Mantiene la dimensionalità
            nn.ReLU(inplace=True),      # Non-linearità
            nn.Dropout(p=0.2),          # Leggera regolarizzazione per evitare overfitting sulle classi base
            nn.Linear(512, output_dim)  # Compressione finale a 256
        )

    def forward(self, x):
        # Feature extraction
        out = self.backbone(x)
        
        # Projezione Non-Lineare
        out = self.projection_layer(out)
        
        # OUTPUT RAW
        # Nessuna normalizzazione L2, Nessuno scaling.
        # Lasciamo che la distanza Euclidea lavori anche sulle magnitudo.
        return out
'''

In [ ]:
'''class PrototypicalResNet(nn.Module):
    """
    ESPERIMENTO E: NO PROJECTION HEAD (512-dim Embedding)
    
    In questa variante rimuoviamo il layer lineare di proiezione.
    L'output è il feature vector grezzo (avgpool) della ResNet18.
    
    Vantaggi ipotetici:
    - Nessuna perdita di informazione dovuta alla compressione (512 -> 256).
    - Minore rischio di overfitting sulle feature specifiche delle classi Base.
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        
        # 1. Caricamento pesi ImageNet
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = resnet18(weights=weights)
        
        # 2. Rimozione del FC layer originale
        # Sostituiamo l'ultimo layer con Identity.
        # In questo modo, l'output della backbone sarà direttamente il pool5 features (512 dim)
        self.backbone.fc = nn.Identity()
        
        # NOTA: Non c'è self.projection_layer qui.
        
    def forward(self, x):
        # L'output ha dimensione [Batch, 512]
        return self.backbone(x)

'''

In [ ]:
def protonet_loss(embeddings: torch.Tensor, label_globali: torch.Tensor, n_way: int, k_shot: int, q_query: int):
    """
    Calcola loss, logits e target locali.
    """
    # In PyTorch tutte le operazioni devono avvenire sullo stesso dispositivo
    # ci assicuriamo che i successivi tensori vengano generati sullo stesso device degli embeddings
    device = embeddings.device
    
    # 1. Reshape & sanity check delle etichette
    
    # Il tensore label_globali è un vettore 1D di lunghezza 80
    # Contiene gli ID di encoding delle classi
    # Il reshape forza il vettore piatto a diventare una matrice con n righe (n-way) e k+q colonne (k-shot+query)
    labels_check = label_globali.view(n_way, k_shot + q_query)
    
    # Controlliamo se la label è uguale per ciascuna classe sfruttando la deviazione standard
    # dato che la deviazione standard misura quanto i valori si discostano dalla media
    if labels_check.float().std(dim=1).sum() > 0:
        raise RuntimeError("ERRORE: Batch non ordinato per classe!")

    # 2. Struttura Dati
    
    # L'embedding parte come matrice [80, 256] 
    # con numero righe == batch size =  4*(5 + 15) = 80 
    # e numero colonne == dimensioni del feature vector
    
    # Riorganizziamo i dati in 3 dimensioni:
    # Dimensione 0 (n_way = 4): Suddivide il batch in 4 linee cellulari.
    # Dimensione 1 (k_shot + q_query = 20): All'interno di ogni classe, raggruppa le 20 immagini (le 5 di supporto e le 15 di query).
    # Dimensione 2 (-1): PyTorch calcola in modo automatico e flessibile la lunghezza dell'embedding.
    z = embeddings.view(n_way, k_shot + q_query, -1)
    
    # 3. Prototipi (Support) vs Query
    
    # Prototipi è un tensore di forma [4, 256], contiene 4 punti ideali nello spazio
    prototipi = z[:, :k_shot].mean(dim=1)

    # Query_flat contiene gli embedding delle 60 query da classificare rispetto ai prototipi
    # contiguous() ordina i dati in modo sequenziale in memoria per evitare errori
    # view effettua il reshape per ottenere un tensore [60, 256]
    query_flat = z[:, k_shot:].contiguous().view(n_way * q_query, -1)
    
    # 4. Calcoliamo le distanze sfruttando l'efficienza della formula alternativa x^2 + y^2 - 2xy

    # Eleviamo al quadrato ogni singola feature e poi sommiamo lungo la riga (dim=1)
    # keepdim=True mantiene il risultato come colonna invece di appiattirlo in un vettore
    # Questo è essenziale per il broadcasting
    query_sq = query_flat.pow(2).sum(dim=1, keepdim=True)

    # Eleviamo al quadrato ogni singola feature e poi sommiamo lungo la riga (dim=1)
    proto_sq = prototipi.pow(2).sum(dim=1, keepdim=True).t()

    # Moltiplicazione matriciale tra query_flat [60, 256] e prototipi trasposti [256, 4]
    # Calcola il prodotto scalare tra ogni query e ogni prototipo
    product = torch.mm(query_flat, prototipi.t())

    # Il .clamp(min=0) è una tecnica di stabilità numerica
    # Senza di esso, un singolo pixel quasi identico al prototipo potrebbe generare un valore negativo    
    dist_sq = (query_sq + proto_sq - 2 * product).clamp(min=0)

    # Calcoliamo i logits
    logits = -dist_sq
    
    # 5. Calculo della loss tramite CrossEntropy

    # Creiamo un tensore di etichette target di lunghezza [60] (4x15)
    # usiamo repeat_interleave (che ripete i singoli elementi 0,0,1,1) invece di repeat (che farebbe 0,1,0,1)
    target_locale = torch.arange(n_way, device=device).repeat_interleave(q_query)

    # nn.CrossEntropyLoss() crea un'istanza della classe.
    # Le parentesi tonde subito dopo invocano il dunder method __call__ di quell'istanza
    # La classe si comporta così come una funzione
    # Esegue softmax e negative log likelihood
    loss = nn.CrossEntropyLoss()(logits, target_locale)
    
    # Restituiamo i tensori grezzi necessari per le metriche, senza calcolarle qui
    return loss, logits, target_locale

In [ ]:
def calcola_metriche(logits, target_locale, label_globali, n_way, k_shot, q_query):
    """
    METRICS FUNCTION: Calcola Acc, Precision, Recall, F1 Score e Mapping Globale.
    """
    with torch.no_grad():
        device = logits.device
        
        # 1. Predizioni locali

        # Per ogni query (riga), cerca l'indice della colonna che ha il valore massimo (il logit più vicino allo zero).
        preds_locali = logits.argmax(dim=1)

        # Confrontiamo il vettore delle predizioni [60] con il vettore dei target corretti [60]
        # Risultato: Un vettore booleano di 60 elementi.
        # True se il modello ha indovinato (predizione == target).False se il modello ha sbagliato.
        # item() fa il casting a numero standard della media delle predizioni corrette
        acc = (preds_locali == target_locale).float().mean().item()

        # 2. Metriche avanzate
        
        # epsilon serve a prevenire il crash del programma dovuto alla divisione per zero
        # Esempio: Se il modello non predice mai la classe "A172",
        # il denominatore della Precision (TP + FP) sarebbe 0.
        epsilon = 1e-8

        # torch.arange(n_way) crea un vettore degli indici LOCALI delle classi presenti nell'episodio.
        # Shape finale: [1, 4]. Contenuto: [[0, 1, 2, 3]].
        classes = torch.arange(n_way, device=device).view(1, -1)

        # oh = one-hot 
        
        # Trasformiamo il vettore delle predizioni [60] in una colonna [60, 1]
        # Ogni riga ha 1.0 nella colonna della classe predetta per la query i
        pred_oh = (preds_locali.view(-1, 1) == classes).float()

        # Trasformiamo il vettore dei target locali in colonna di forma [60, 1].
        # ogni riga i ha 1.0 nella colonna della classe reale (ground truth) per la query i
        target_oh = (target_locale.view(-1, 1) == classes).float()

        # dim = 0 somma verticalmente lungo le colonne
        tp = (pred_oh * target_oh).sum(dim=0)
        fp = (pred_oh * (1 - target_oh)).sum(dim=0)
        fn = ((1 - pred_oh) * target_oh).sum(dim=0)
        
        precision = (tp / (tp + fp + epsilon)).mean().item()
        recall = (tp / (tp + fn + epsilon)).mean().item()
        f1 = (2 * tp / (2 * tp + fp + fn + epsilon)).mean().item()
        
        # 3. Mapping globale

        # Organizziamo le label globali in una matrice [4, 20] per isolare i nomi reali di ogni classe
        lbls = label_globali.view(n_way, k_shot + q_query)
        
        # Estraiamo l'ID reale per ciascuna delle 4 classi locali prendendo il primo elemento di ogni riga
        ids_classi_episodio = lbls[:, 0]

        # Sostituisce ogni indice locale (0,1,2,3) con l'ID reale della cellula (es. 4,0,6,7) 
        # usando ids_classi_episodio come una tabella di conversione.
        preds_globali = ids_classi_episodio[preds_locali]

        # Estraiamo i target reali per le query e appiattiamo in un vettore [60] 
        target_globali = lbls[:, k_shot:].contiguous().view(-1)
        
    return acc, precision, recall, f1, preds_globali, target_globali

In [ ]:
def visualizza_training_fold(storia: dict, fold_idx: int, path_out: Path):
    """
    Dashboard completa del training.
    
    Compatibilità:
    Visualizza le metriche salvate nel dizionario 'storia'.
    Si aspetta che le chiavi siano popolate con i valori restituiti da `calcola_metriche`:
    - 'train_loss'
    - 'val_acc'
    - 'val_f1', 'val_precision', 'val_recall' (Opzionali ma raccomandati)
    """
    # Creiamo una figura con 3 subplot in fila
    fig, ax = plt.subplots(1, 3, figsize=(18, 5), facecolor='#f5f7f9')
    
    epochs = range(1, len(storia['train_loss']) + 1)

    # 1. Andamento della loss (Curva di apprendimento)
    ax[0].plot(epochs, storia['train_loss'], label='Train Loss', color='tab:blue', linewidth=2)
    # Se avessi anche la validation loss:
    # ax[0].plot(epochs, storia.get('val_loss', []), label='Val Loss', color='tab:red', linestyle='--')
    ax[0].set_title(f"Fold {fold_idx}: Loss Curve", fontsize=12, fontweight='bold')
    ax[0].set_xlabel("Epoca")
    ax[0].set_ylabel("Loss")
    ax[0].grid(True, alpha=0.3, linestyle='--')
    ax[0].legend()

    # 2. Accuracy plot
    ax[1].plot(epochs, storia['val_acc'], label='Val Accuracy', color='tab:green', linewidth=2)
    ax[1].set_title(f"Fold {fold_idx}: Accuracy Trend", fontsize=12, fontweight='bold')
    ax[1].set_xlabel("Epoca")
    ax[1].set_ylabel("Accuracy")
    ax[1].set_ylim([0, 1.05]) 
    ax[1].grid(True, alpha=0.3, linestyle='--')
    ax[1].legend()

    # 3. Plot delle metriche avanzate (F1, Precision, Recall)
    if 'val_f1' in storia and len(storia['val_f1']) > 0:
        ax[2].plot(epochs, storia['val_f1'], label='F1 Score', color='purple', linewidth=2)
        ax[2].plot(epochs, storia['val_precision'], label='Precision', color='cyan', linestyle='--', alpha=0.7)
        ax[2].plot(epochs, storia['val_recall'], label='Recall', color='magenta', linestyle='--', alpha=0.7)
        ax[2].set_title(f"Fold {fold_idx}: Quality Metrics", fontsize=12, fontweight='bold')
        ax[2].set_xlabel("Epoca")
        ax[2].set_ylabel("Score")
        ax[2].set_ylim([0, 1.05])
        ax[2].grid(True, alpha=0.3, linestyle='--')
        ax[2].legend()
    else:
        ax[2].text(0.5, 0.5, "Metriche F1/P/R non disponibili\n(Usa calcola_metriche nel loop)", 
                   ha='center', va='center', fontsize=10, color='gray')

    plt.suptitle(f"Training Dashboard - Fold {fold_idx}", fontsize=16)
    plt.tight_layout()
    plt.savefig(path_out / f"fold_{fold_idx}_dashboard.png", dpi=300)
    plt.close()

In [ ]:
def visualizza_confusion_matrix_fewshot(y_true, y_pred, nomi_classi, fold_idx, path_out):
    """
    Confusion Matrix professionale con doppia annotazione.
    
    Upgrade:
    Accetta automaticamente sia numpy arrays che torch.Tensor (output di calcola_metriche).
    """
    
    # Se arrivano tensori GPU da calcola_metriche, li portiamo su CPU/Numpy automaticamente
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.detach().cpu().numpy()
    if isinstance(y_pred, torch.Tensor):
        y_pred = y_pred.detach().cpu().numpy()

    cm = confusion_matrix(y_true, y_pred)
    
    # Calcolo percentuali per riga (Recall per classe)
    with np.errstate(divide='ignore', invalid='ignore'):
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm) 

    # Creazione delle annotazioni composte
    annotazioni = []
    for i in range(cm.shape[0]):
        row_annot = []
        for j in range(cm.shape[1]):
            count = cm[i, j]
            percent = cm_norm[i, j]
            if count > 0:
                row_annot.append(f"{count}\n({percent:.1%})")
            else:
                row_annot.append("0")
        annotazioni.append(row_annot)
    annotazioni = np.array(annotazioni)

    plt.figure(figsize=(12, 10), facecolor='#f5f7f9')
    sns.heatmap(cm_norm, 
                annot=annotazioni, 
                fmt="", 
                cmap="Blues", 
                cbar_kws={'label': 'Recall (Probabilità Predizione)'},
                xticklabels=nomi_classi, 
                yticklabels=nomi_classi,
                square=True, 
                linewidths=0.5, 
                linecolor='gray')
    
    plt.title(f"Confusion Matrix (Test Set) - Fold {fold_idx}\n(Righe=Reale, Colonne=Predetto)", fontsize=14, pad=20)
    plt.ylabel("Classe Reale (Ground Truth)", fontsize=12, fontweight='bold')
    plt.xlabel("Classe Predetta (Prediction)", fontsize=12, fontweight='bold')
    
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig(path_out / f"fold_{fold_idx}_cm_advanced.png", dpi=300)
    plt.close()

In [ ]:
# FUNZIONE DI SCHEDULING CHE AVEVAMO INIZIALMENTE ADOTTATO. NON PIÚ IN USO
'''
def get_scheduler_with_warmup(optimizer, warmup_epochs: int, total_epochs: int):
    """
    Crea uno scheduler ibrido:
    1. Warmup Lineare: LR sale da ~0 al target per 'warmup_epochs'.
       Serve a stabilizzare i gradienti iniziali senza distruggere i pesi ImageNet.
    2. Cosine Decay: LR scende dolcemente fino alla fine.
    """
    # Fase 1: Warmup (Start factor = 0.01 significa partire dall'1% del LR)
    scheduler_warmup = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs
    )
    
    # Fase 2: Annealing (Decadimento)
    scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=(total_epochs - warmup_epochs), eta_min=1e-6
    )
    
    # Combinazione
    return optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_epochs]
    )
'''

In [ ]:
def esegui_fold(fold_idx, classi_base, classi_novel, df_dati, config, device):
    # Per riproducibilità resettiamo il seed all'inizio di ogni fold. 
    # Usiamo SEED_GLOBALE + fold_idx per avere variazioni deterministiche tra i fold.
    seed_corrente = SEED_GLOBALE + fold_idx
    imposta_seed(seed_corrente, modalita_deterministica=True)
    
    # Creiamo il generatore per il DataLoader
    g = crea_generatore_riproducibile(seed_corrente)

    print(f"\n{'='*30} FOLD {fold_idx} {'='*30}")
    print(f"Seed attivo: {seed_corrente}")
    
    print(f"Train Classes: {classi_base}")
    print(f"Test Classes: {classi_novel} (Solo Novel - Few-Shot Puro)")
    
    # 1. Dataset Setup
    df_base = filtra_dataset(df_dati, classi_base)
    
    # Filtriamo solo le classi novel per il test
    df_test_novel = filtra_dataset(df_dati, classi_novel)
    df_test_novel = df_test_novel[df_test_novel["split"] == "test"].copy()
    
    ds_train = LIVECellDataset(df_base[df_base["split"] == "train"], ottieni_trasformazioni("train"))
    ds_val = LIVECellDataset(df_base[df_base["split"] == "val"], ottieni_trasformazioni("val"))
    ds_test = LIVECellDataset(df_test_novel, ottieni_trasformazioni("val"))

    print(f"Dataset Sizes: Train={len(ds_train)} | Val={len(ds_val)} | Test={len(ds_test)} (Tutte le classi)")
    
    # 2. Sampler & Loader
    s_train = CampionatoreEpisodico(ds_train.etichette, config.TRAIN_N_WAY, config.TRAIN_K_SHOT, config.TRAIN_Q_QUERY, config.EPISODI_TRAIN)
    s_val = CampionatoreEpisodico(ds_val.etichette, config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY, config.EPISODI_VAL)
    s_test = CampionatoreEpisodico(ds_test.etichette, config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY, config.EPISODI_TEST)
    
    # Worker init per riproducibilità
    l_train = DataLoader(ds_train, batch_sampler=s_train, num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY, 
                         worker_init_fn=worker_init_fn, generator=g, persistent_workers=True, prefetch_factor=2)
    l_val = DataLoader(ds_val, batch_sampler=s_val, num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY, 
                        worker_init_fn=worker_init_fn, generator=g, persistent_workers=True, prefetch_factor=2)
    l_test = DataLoader(ds_test, batch_sampler=s_test, num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY, 
                         worker_init_fn=worker_init_fn, generator=g)

    # 3. Model, Optimizer & Scheduler (MULTI-GPU SETUP)
    modello = PrototypicalResNet(pretrained=True).to(device)
    
    # MULTI-GPU KAGGLE
    if torch.cuda.device_count() > 1:
        print(f" -> Attivazione DataParallel su {torch.cuda.device_count()} GPU T4")
        modello = nn.DataParallel(modello)

    # Optimizer AdamW
    optimizer = optim.AdamW(modello.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
    
    # Scheduler non più in uso scheduler = get_scheduler_with_warmup(optimizer, warmup_epochs=3, total_epochs=config.EPOCHE_PER_FOLD)
    
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    
    # 4. Training Loop
    best_acc = 0.0
    best_weights = None
    
    # Setup storia per tutte le metriche
    storia = {'train_loss': [], 'val_acc': [], 'val_f1': [], 'val_precision': [], 'val_recall': []}
    
    for ep in range(config.EPOCHE_PER_FOLD):
        # TRAIN
        modello.train()
        loss_cum = 0.0
        pbar = tqdm(l_train, desc=f"Ep {ep+1}/{config.EPOCHE_PER_FOLD}", leave=False)
        
        for batch_img, batch_lbl in pbar:
            batch_img = batch_img.to(device)
            batch_lbl = batch_lbl.to(device)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                emb = modello(batch_img)
                loss, _, _ = protonet_loss(
                    emb, batch_lbl, 
                    config.TRAIN_N_WAY, config.TRAIN_K_SHOT, config.TRAIN_Q_QUERY
                )
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            loss_cum += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.3f}"})
            
        storia['train_loss'].append(loss_cum / len(l_train))
        
        # VAL
        modello.eval()
        metrics_sum = {'acc': 0.0, 'p': 0.0, 'r': 0.0, 'f1': 0.0}
        
        with torch.no_grad():
            for batch_img, batch_lbl in l_val:
                batch_img = batch_img.to(device)
                batch_lbl = batch_lbl.to(device)
                
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    emb = modello(batch_img)
                    
                    # Step A: Logits
                    _, logits, target_locale = protonet_loss(
                        emb, batch_lbl,
                        config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY
                    )
                    
                    # Step B: Metriche
                    acc, p, r, f1, _, _ = calcola_metriche(
                        logits, target_locale, batch_lbl,
                        config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY
                    )
                
                metrics_sum['acc'] += acc
                metrics_sum['p'] += p
                metrics_sum['r'] += r
                metrics_sum['f1'] += f1
        
        # Calcolo medie epoch
        n_val = len(l_val)
        val_acc = metrics_sum['acc'] / n_val
        val_f1 = metrics_sum['f1'] / n_val
        
        # Aggiornamento storia
        storia['val_acc'].append(val_acc)
        storia['val_f1'].append(val_f1)
        storia['val_precision'].append(metrics_sum['p'] / n_val)
        storia['val_recall'].append(metrics_sum['r'] / n_val)
        
        # scheduler.step()
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Ep {ep+1} | Loss: {storia['train_loss'][-1]:.4f} | Val Acc: {val_acc:.2%} | Val F1: {val_f1:.4f} | LR: {curr_lr:.2e}")
        
        # Save Best Model
        if val_acc > best_acc:
            best_acc = val_acc
            
            if isinstance(modello, nn.DataParallel):
                state_dict = modello.module.state_dict()
            else:
                state_dict = modello.state_dict()

            best_weights = {k: v.cpu() for k, v in state_dict.items()}
            
            path_ckpt = config.CARTELLA_OUTPUT / f"fold_{fold_idx}_best_model.pth"
            torch.save({
                'epoch': ep,
                'model_state_dict': best_weights,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_metrics': {'acc': best_acc, 'f1': val_f1},
                'config': asdict(config)
            }, path_ckpt)
            print(f"  -> Checkpoint salvato: {path_ckpt.name}")
            
    # Plotting Dashboard
    visualizza_training_fold(storia, fold_idx, config.CARTELLA_OUTPUT)
    
    # 5. Testing (Novel Classes)
    print("Caricamento migliori pesi per il test...")
    
    modello_test = PrototypicalResNet(pretrained=False).to(device)
    modello_test.load_state_dict(best_weights)
    
    if torch.cuda.device_count() > 1:
        modello_test = nn.DataParallel(modello_test)
        
    modello_test.eval()
    
    # Inizializziamo tutte le metriche da accumulare
    metrics_test_sum = {'acc': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    preds_totali = []
    targets_totali = []
    
    with torch.no_grad():
        for batch_img, batch_lbl in tqdm(l_test, desc="Testing Novel"):
            batch_img = batch_img.to(device)
            batch_lbl = batch_lbl.to(device)
            
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                emb = modello_test(batch_img)
                
                # Step A: Logits
                _, logits, target_locale = protonet_loss(
                    emb, batch_lbl,
                    config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY
                )
                
                # Step B: Metriche
                acc, p, r, f1, preds_glob, targs_glob = calcola_metriche(
                    logits, target_locale, batch_lbl,
                    config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY
                )

            # ACCUMULO DI TUTTE LE METRICHE 
            metrics_test_sum['acc'] += acc
            metrics_test_sum['f1'] += f1
            metrics_test_sum['precision'] += p
            metrics_test_sum['recall'] += r
            
            preds_totali.extend(preds_glob.cpu().numpy())
            targets_totali.extend(targs_glob.cpu().numpy())
            
    # Calcolo delle medie finali
    n_batches = len(l_test)
    final_metrics = {k: v / n_batches for k, v in metrics_test_sum.items()}
    
    print(f"--> Fold {fold_idx} Result: Acc={final_metrics['acc']:.2%} | F1={final_metrics['f1']:.4f}")
    
    # 6. Cleanup
    del modello, modello_test, optimizer, scaler, l_train, l_val, l_test
    gc.collect()
    torch.cuda.empty_cache()
    
    # Restituiamo il dizionario completo delle metriche invece di solo acc
    return final_metrics, preds_totali, targets_totali

# Esecuzione degli esperimenti su fold difficili/realistici

In [ ]:
# 1. CONFIGURAZIONE PATH E PARAMETRI
PROTO_CONFIG.EPOCHE_PER_FOLD = 10 

# Opzione Resume: Inserire il path se si vuole continuare un esperimento interrotto
PATH_RESUME = None 
# Esempio: PATH_RESUME = Path("/kaggle/working/esperimenti/2026_01_22_15_08_ProtoNet_...")

if PATH_RESUME:
    OUTPUT_DIR = Path(PATH_RESUME)
    PROTO_CONFIG.CARTELLA_OUTPUT = OUTPUT_DIR 
    print(f"-> RESUME MODE: Proseguimento esperimento in {OUTPUT_DIR}")
else:
    OUTPUT_DIR = PROTO_CONFIG.CARTELLA_OUTPUT
    print(f"-> NEW EXPERIMENT: Creazione directory {OUTPUT_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_RISULTATI = OUTPUT_DIR / "risultati_parziali.csv"

# 2. SETUP HARDWARE E DATI
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

# Caricamento Indici (se non presenti)
if 'df_dati' not in locals():
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")

# Caricamento Mapping Classi
with open(CONFIG.CARTELLA_INDICI / "mappatura_classi.json") as f:
    mappatura_classi = json.load(f)
ind_to_class = {v: k for k, v in mappatura_classi.items()}

# 3. GESTIONE STATO E CARICAMENTO FOLD
folds_gia_fatti = []
results = []

if FILE_RISULTATI.exists():
    print("Recupero risultati parziali esistenti...")
    df_esistente = pd.read_csv(FILE_RISULTATI)
    folds_gia_fatti = df_esistente["Fold_ID_Univoco"].astype(str).tolist()
    results = df_esistente.to_dict('records')
    print(f"-> Fold completati: {len(folds_gia_fatti)}")

# Caricamento configurazione semantica manuale
PATH_MANUAL_FOLDS = CONFIG.CARTELLA_INDICI / "folds_manuali.json"
if not PATH_MANUAL_FOLDS.exists():
    raise FileNotFoundError(f"File configurazione fold non trovato: {PATH_MANUAL_FOLDS}")

with open(PATH_MANUAL_FOLDS, "r") as f:
    dati_manuali = json.load(f)

# Parsing della struttura fold
folds_target = []
descrizioni_fold = {} 

for item in dati_manuali:
    t_ids = tuple(item["novel_classes_ids"])
    folds_target.append(t_ids)
    # Mapping indice 0-based -> descrizione
    descrizioni_fold[item["fold_idx"] - 1] = item["descrizione"]

NUMERO_FOLD_DA_ESEGUIRE = len(folds_target)
print(f"\nPiano di Esecuzione ({NUMERO_FOLD_DA_ESEGUIRE} Scenari):")
for k, f in enumerate(folds_target):
    status = "[COMPLETATO]" if f"{sorted(f)[0]}-{sorted(f)[1]}" in folds_gia_fatti else "[DA ESEGUIRE]"
    print(f"  Fold {k+1}: {descrizioni_fold.get(k)} {status}")

# Backup della configurazione usata
with open(OUTPUT_DIR / "configurazione_folds_usata.json", "w") as f:
    json.dump({"modalita": "MANUALE", "folds": dati_manuali}, f, indent=4)

# 4. LOOP DI TRAINING
all_classes = list(range(8))

for i, novel_idx in enumerate(folds_target):
    # Identificativi
    novel_sorted = sorted(novel_idx)
    fold_id = f"{novel_sorted[0]}-{novel_sorted[1]}"
    descrizione_corrente = descrizioni_fold.get(i, "Generico")
    fold_progressivo = i + 1  # Usiamo l'indice naturale del fold semantico

    # Skip se già processato
    if fold_id in folds_gia_fatti:
        continue

    # Definizione Base Classes (Tutte tranne le novel)
    base_idx = tuple(c for c in all_classes if c not in novel_idx)
    nomi_novel = [ind_to_class[idx] for idx in novel_sorted]

    # Garbage Collection preventiva
    gc.collect()
    torch.cuda.empty_cache()

    try:
        print(f"\n{'='*80}")
        print(f"RUNNING FOLD {fold_progressivo}/{NUMERO_FOLD_DA_ESEGUIRE} | ID: {fold_id}")
        print(f"Scenario: {descrizione_corrente}")
        print(f"Novel Classes: {nomi_novel} (IDs: {novel_idx})")
        print(f"{'='*80}")
        
        # ESECUZIONE 
        # Restituisce: Dizionario Metriche, Predizioni, Target
        metriche_test, preds, targs = esegui_fold(
            fold_progressivo, base_idx, novel_idx, df_dati, PROTO_CONFIG, device
        )
        
        # VISUALIZZAZIONE 
        visualizza_confusion_matrix_fewshot(
            targs, preds, nomi_novel, 
            fold_idx=f"{fold_progressivo}_{fold_id}", 
            path_out=OUTPUT_DIR
        )
        
        # REGISTRAZIONE 
        nuovo_record = {
            "Fold_Progressivo": fold_progressivo,
            "Fold_ID_Univoco": fold_id,
            "Descrizione_Scenario": descrizione_corrente,
            "Novel_Classes_IDs": str(list(novel_idx)),
            "Novel_Classes_Names": str(nomi_novel),
            # Metriche dettagliate
            "Test_Accuracy": metriche_test['acc'],
            "Test_F1": metriche_test['f1'],
            "Test_Precision": metriche_test['precision'],
            "Test_Recall": metriche_test['recall']
        }
        results.append(nuovo_record)
        
        # Salvataggio incrementale
        df_log = pd.DataFrame(results)
        df_log.to_csv(FILE_RISULTATI, index=False)
        
        print(f"--> Fold Completato. Acc: {metriche_test['acc']:.2%} | F1: {metriche_test['f1']:.4f}")
        
    except Exception as e:
        print(f"[ERRORE CRITICO] Fold {fold_id} fallito: {e}")
        # Tentativo di recovery memoria
        torch.cuda.empty_cache()
        gc.collect()
        raise e 

    # Cleanup fine ciclo
    del preds, targs
    gc.collect()
    torch.cuda.empty_cache()

# 5. REPORT FINALE AGGREGATO
if results:
    df_res = pd.DataFrame(results)
    
    # Calcolo statistiche per tutte le metriche numeriche
    metriche_cols = ["Test_Accuracy", "Test_F1", "Test_Precision", "Test_Recall"]
    summary = df_res[metriche_cols].agg(['mean', 'std'])
    
    print("\n" + "="*80)
    print(f"REPORT FINALE - {PROTO_CONFIG.NOME_ESPERIMENTO}")
    print("="*80)
    
    # Tabella riassuntiva per fold
    cols_display = ["Fold_ID_Univoco", "Descrizione_Scenario", "Test_Accuracy", "Test_F1"]
    print(df_res[cols_display].to_string(index=False))
    
    print("-" * 80)
    print("MEDIA GENERALE:")
    print(f"Accuracy:  {summary.loc['mean', 'Test_Accuracy']:.2%} ± {summary.loc['std', 'Test_Accuracy']:.2%}")
    print(f"F1-Score:  {summary.loc['mean', 'Test_F1']:.4f} ± {summary.loc['std', 'Test_F1']:.4f}")
    print(f"Precision: {summary.loc['mean', 'Test_Precision']:.4f} ± {summary.loc['std', 'Test_Precision']:.4f}")
    print(f"Recall:    {summary.loc['mean', 'Test_Recall']:.4f} ± {summary.loc['std', 'Test_Recall']:.4f}")
    print("="*80)
    
    # Salvataggio report testuale
    with open(OUTPUT_DIR / "final_report.txt", "w") as f:
        f.write(f"Esperimento: {PROTO_CONFIG.NOME_ESPERIMENTO}\n")
        f.write(f"Data completamento: {pd.Timestamp.now()}\n\n")
        f.write("RISULTATI PER FOLD:\n")
        f.write(df_res.to_string(index=False))
        f.write("\n\nAGGREGATI GLOBALI:\n")
        f.write(summary.to_string())

else:
    print("Nessun risultato generato.")

# Esperimento A vs Esperimento B: cella focalizzata sul testing dell'esperimento A senza TTA dopo aver svolto l'esperimento B per ablation study

In [ ]:
# 1. CONFIGURAZIONE PATH ESISTENTI
# La cartella dove hai i pesi addestrati (quella che hai indicato)
DIR_PESI_SORGENTE = Path("/kaggle/working/esperimenti/2026_01_26_20_44_ProtoNet_TTA_8D4_ResNet18_Lin256_NO_Clahe2.0_NoNorm4W_Val2W_Test2W")

# La nuova cartella dove salveremo i risultati di questo test comparativo
TIMESTAMP_NOW = pd.Timestamp.now().strftime("%Y_%m_%d_%H_%M")
DIR_OUTPUT_TEST = CONFIG.PERCORSO_LAVORO / "esperimenti" / f"{TIMESTAMP_NOW}_TEST_ONLY_NO_TTA_Standard"
DIR_OUTPUT_TEST.mkdir(parents=True, exist_ok=True)

print(f"SORGENTE PESI: {DIR_PESI_SORGENTE}")
print(f"OUTPUT REPORT: {DIR_OUTPUT_TEST}")

# 2. FUNZIONE DI TESTING PURO (SENZA TTA)
def esegui_test_no_tta(fold_idx, novel_classes, df_dati, config, device, path_checkpoint):
    """
    Carica un checkpoint ed esegue il test STANDARD (Singola vista, NO TTA).
    """
    # A. Setup Dataset (Solo Test set delle classi Novel)
    df_test_novel = filtra_dataset(df_dati, novel_classes)
    df_test_novel = df_test_novel[df_test_novel["split"] == "test"].copy()
    
    # Usa le trasformazioni di validazione (Resize + Normalize standard)
    ds_test = LIVECellDataset(df_test_novel, ottieni_trasformazioni("val"))
    
    s_test = CampionatoreEpisodico(
        ds_test.etichette, 
        config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY, config.EPISODI_TEST
    )
    
    l_test = DataLoader(
        ds_test, batch_sampler=s_test, num_workers=config.NUM_WORKERS, pin_memory=True
    )

    # B. Setup Modello e Caricamento Pesi
    modello = PrototypicalResNet(pretrained=False).to(device)
    
    if not path_checkpoint.exists():
        raise FileNotFoundError(f"Checkpoint non trovato: {path_checkpoint}")
        
    print(f"  -> Caricamento pesi da: {path_checkpoint.name}")
    checkpoint = torch.load(path_checkpoint, map_location=device, weights_only=False)
    
    # Gestione compatibilità nomi layer (rimozione prefisso 'module.' se salvato con DataParallel)
    state_dict = checkpoint['model_state_dict']
    new_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    modello.load_state_dict(new_state_dict)
    
    if torch.cuda.device_count() > 1:
        modello = nn.DataParallel(modello)
        
    modello.eval()
    
    # C. Loop di Test (NO TTA)
    metrics_sum = {'acc': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    preds_totali = []
    targets_totali = []
    
    with torch.no_grad():
        for batch_img, batch_lbl in tqdm(l_test, desc=f"Test Fold {fold_idx} (NO TTA)", leave=False):
            batch_img = batch_img.to(device)
            batch_lbl = batch_lbl.to(device)
            
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                
                # --- INFERENCE STANDARD (Singola vista) ---
                # Nessuna chiamata a forward_tta_all, passiamo l'immagine direttamente
                emb = modello(batch_img)
                # ------------------------------------------
                
                _, logits, target_locale = protonet_loss(
                    emb, batch_lbl,
                    config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY
                )
                
                acc, p, r, f1, preds_glob, targs_glob = calcola_metriche(
                    logits, target_locale, batch_lbl,
                    config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY
                )
            
            metrics_sum['acc'] += acc
            metrics_sum['f1'] += f1
            metrics_sum['precision'] += p
            metrics_sum['recall'] += r
            
            preds_totali.extend(preds_glob.cpu().numpy())
            targets_totali.extend(targs_glob.cpu().numpy())
            
    n_batches = len(l_test)
    final_metrics = {k: v / n_batches for k, v in metrics_sum.items()}
    
    del modello, l_test
    gc.collect()
    torch.cuda.empty_cache()
    
    return final_metrics, preds_totali, targets_totali

# 3. MAIN LOOP DI ESECUZIONE

# Setup Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Caricamento Dati
if 'df_dati' not in locals():
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")

# Caricamento definizione Folds Manuali
PATH_MANUAL_FOLDS = CONFIG.CARTELLA_INDICI / "folds_manuali.json"
with open(PATH_MANUAL_FOLDS, "r") as f:
    dati_manuali = json.load(f)

results = []

print(f"\nAVVIO TEST COMPARATIVO: STANDARD (NO TTA) usando pesi addestrati.")
print("-" * 80)

for item in dati_manuali:
    fold_idx = item["fold_idx"]
    descrizione = item["descrizione"]
    novel_ids = tuple(item["novel_classes_ids"])
    novel_names = item["novel_classes_names"]
    fold_id_univoco = item["fold_id_univoco"]
    
    print(f"\nProcessing Fold {fold_idx}: {descrizione} (Classi: {novel_names})")
    
    # Costruzione percorso checkpoint previsto
    # Nota: Assicurati che il nome file corrisponda a quello salvato nel training precedente
    path_ckpt = DIR_PESI_SORGENTE / f"fold_{fold_idx}_best_model.pth"
    
    if not path_ckpt.exists():
        print(f"[ERRORE] Checkpoint non trovato: {path_ckpt}")
        continue
        
    # Esecuzione Test
    try:
        metrics, preds, targs = esegui_test_no_tta(
            fold_idx, novel_ids, df_dati, PROTO_CONFIG, device, path_ckpt
        )
        
        print(f"  -> Risultato NO TTA: Acc={metrics['acc']:.2%} | F1={metrics['f1']:.4f}")
        
        # 1. Visualizzazione Confusion Matrix
        visualizza_confusion_matrix_fewshot(
            targs, preds, novel_names, 
            fold_idx=f"{fold_idx}_NO_TTA", 
            path_out=DIR_OUTPUT_TEST
        )
        
        # 2. Salvataggio record
        results.append({
            "Fold_Progressivo": fold_idx,
            "Fold_ID_Univoco": fold_id_univoco,
            "Descrizione_Scenario": descrizione,
            "Modalita": "Standard (NO TTA)",
            "Test_Accuracy": metrics['acc'],
            "Test_F1": metrics['f1'],
            "Test_Precision": metrics['precision'],
            "Test_Recall": metrics['recall']
        })
        
    except Exception as e:
        print(f"Errore durante il test del fold {fold_idx}: {e}")
        import traceback
        traceback.print_exc()

# 4. REPORT FINALE E SALVATAGGIO
if results:
    df_res = pd.DataFrame(results)
    path_csv = DIR_OUTPUT_TEST / "risultati_no_tta.csv"
    df_res.to_csv(path_csv, index=False)
    
    print("\n" + "="*80)
    print(f"REPORT FINALE (NO TTA) - Salvato in {path_csv}")
    print("="*80)
    
    # Mostra tabella
    cols = ["Descrizione_Scenario", "Test_Accuracy", "Test_F1"]
    print(df_res[cols].to_string(index=False))
    
    # Statistiche aggregate
    mean_acc = df_res["Test_Accuracy"].mean()
    std_acc = df_res["Test_Accuracy"].std()
    mean_f1 = df_res["Test_F1"].mean()
    
    print("-" * 80)
    print(f"MEDIA GENERALE (NO TTA):")
    print(f"Accuracy: {mean_acc:.2%} ± {std_acc:.2%}")
    print(f"F1-Score: {mean_f1:.4f}")
    print("="*80)
    
    # Creazione file report testuale
    with open(DIR_OUTPUT_TEST / "report_summary.txt", "w") as f:
        f.write(f"Sorgente Pesi: {DIR_PESI_SORGENTE}\n")
        f.write(f"Modalità: Inference Standard (NO TTA)\n\n")
        f.write(df_res.to_string(index=False))
        f.write(f"\n\nMean Acc: {mean_acc:.4f}\nMean F1: {mean_f1:.4f}")

else:
    print("Nessun risultato generato.")

# Test-time Augmentation per l'esperimento B

In [ ]:
# HELPER: TEST TIME AUGMENTATION (TTA) - D4 DIHEDRAL GROUP (8 Viste)
def forward_tta_all(modello, batch_img, n_way, k_shot, q_query):
    """
    Esegue il forward pass applicando TTA completo (8 viste).
    
    Secondo il paper di Moshkov et al.: Le cellule sono invarianti alla rotazione.
    Usiamo il gruppo diedrale D4 che copre tutte le 8 simmetrie su griglia quadrata:
    - 4 Rotazioni (0, 90, 180, 270)
    - 4 Rotazioni dell'immagine flippata orizzontalmente (che include il flip verticale)
    
    Ispirazione Paper 2 (Wang et al.): L'embedding finale è la media (Expectation) 
    delle 8 viste per ridurre l'incertezza aleatorica.
    """
    # batch_img ha shape originale [Totale_Img, C, H, W]
    b, c, h, w = batch_img.size()
    
    # 1. Generazione Viste TTA (D4 Group - 8 Viste)
    # Creiamo una lista di tensori trasformati
    viste = []
    
    # Gruppo 1: Rotazioni dell'immagine originale
    viste.append(batch_img)                                     # 0° (Originale)
    viste.append(torch.rot90(batch_img, 1, [2, 3]))             # 90°
    viste.append(torch.rot90(batch_img, 2, [2, 3]))             # 180°
    viste.append(torch.rot90(batch_img, 3, [2, 3]))             # 270°
    
    # Gruppo 2: Rotazioni dell'immagine flippata (Horizontal Flip)
    img_flip = torch.flip(batch_img, [3])
    viste.append(img_flip)                                      # Flip H
    viste.append(torch.rot90(img_flip, 1, [2, 3]))              # Flip H + 90°  (Equivale a Transpose)
    viste.append(torch.rot90(img_flip, 2, [2, 3]))              # Flip H + 180° (Equivale a Flip V)
    viste.append(torch.rot90(img_flip, 3, [2, 3]))              # Flip H + 270° (Equivale a Anti-Transpose)
    
    # Concateniamo in un mega-batch [8 * B, C, H, W]
    # Nota: Con Batch standard (20 img) -> 160 immagini. ResNet18 su T4 regge bene.
    aug_batch = torch.cat(viste, dim=0)
    
    # 2. Forward Pass Unico
    # Passiamo tutte le 8 varianti contemporaneamente per efficienza GPU
    feat_aug = modello(aug_batch) # Output: [8*B, 256]
    
    # 3. Aggregazione (Monte Carlo approximation of Expectation)
    # Reshape: [8 (viste), B (immagini originali), 256 (dim)]
    feat_aug = feat_aug.view(8, b, -1)
    
    # Calcoliamo la media lungo la dimensione delle viste (dim=0)
    # Secondo Wang et al., questo riduce l'incertezza aleatorica dovuta alla posa
    feat_avg = feat_aug.mean(dim=0) # [B, 256]
    
    # L'output ha la stessa shape che avrebbe avuto senza TTA
    return feat_avg

# FUNZIONE PRINCIPALE: ESEGUI FOLD
def esegui_fold(fold_idx, classi_base, classi_novel, df_dati, config, device):
    seed_corrente = SEED_GLOBALE + fold_idx
    imposta_seed(seed_corrente, modalita_deterministica=True)
    g = crea_generatore_riproducibile(seed_corrente)

    print(f"\n{'='*30} FOLD {fold_idx} {'='*30}")
    print(f"Seed attivo: {seed_corrente}")
    print(f"Train Classes: {classi_base}")
    print(f"Test Classes: {classi_novel} (Solo Novel - Few-Shot Puro)")
    
    # 1. Dataset Setup
    df_base = filtra_dataset(df_dati, classi_base)
    df_test_novel = filtra_dataset(df_dati, classi_novel)
    df_test_novel = df_test_novel[df_test_novel["split"] == "test"].copy()
    
    ds_train = LIVECellDataset(df_base[df_base["split"] == "train"], ottieni_trasformazioni("train"))
    ds_val = LIVECellDataset(df_base[df_base["split"] == "val"], ottieni_trasformazioni("val"))
    ds_test = LIVECellDataset(df_test_novel, ottieni_trasformazioni("val"))

    print(f"Dataset Sizes: Train={len(ds_train)} | Val={len(ds_val)} | Test={len(ds_test)}")
    
    # 2. Sampler & Loader
    s_train = CampionatoreEpisodico(ds_train.etichette, config.TRAIN_N_WAY, config.TRAIN_K_SHOT, config.TRAIN_Q_QUERY, config.EPISODI_TRAIN)
    s_val = CampionatoreEpisodico(ds_val.etichette, config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY, config.EPISODI_VAL)
    s_test = CampionatoreEpisodico(ds_test.etichette, config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY, config.EPISODI_TEST)
    
    l_train = DataLoader(ds_train, batch_sampler=s_train, num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY, 
                         worker_init_fn=worker_init_fn, generator=g, persistent_workers=True, prefetch_factor=2)
    l_val = DataLoader(ds_val, batch_sampler=s_val, num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY, 
                        worker_init_fn=worker_init_fn, generator=g, persistent_workers=True, prefetch_factor=2)
    l_test = DataLoader(ds_test, batch_sampler=s_test, num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY, 
                         worker_init_fn=worker_init_fn, generator=g)

    # 3. Model Setup
    modello = PrototypicalResNet(pretrained=True).to(device)
    if torch.cuda.device_count() > 1:
        modello = nn.DataParallel(modello)

    optimizer = optim.AdamW(modello.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
    
    # scheduler rimosso scheduler = get_scheduler_with_warmup(optimizer, warmup_epochs=3, total_epochs=config.EPOCHE_PER_FOLD)
    
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
    
    # 4. Training Loop
    best_acc = 0.0
    best_weights = None
    storia = {'train_loss': [], 'val_acc': [], 'val_f1': [], 'val_precision': [], 'val_recall': []}
    
    for ep in range(config.EPOCHE_PER_FOLD):
        # --- TRAIN ---
        modello.train()
        loss_cum = 0.0
        pbar = tqdm(l_train, desc=f"Ep {ep+1}/{config.EPOCHE_PER_FOLD}", leave=False)
        
        for batch_img, batch_lbl in pbar:
            batch_img, batch_lbl = batch_img.to(device), batch_lbl.to(device)
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                emb = modello(batch_img)
                loss, _, _ = protonet_loss(emb, batch_lbl, config.TRAIN_N_WAY, config.TRAIN_K_SHOT, config.TRAIN_Q_QUERY)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            loss_cum += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.3f}"})
            
        storia['train_loss'].append(loss_cum / len(l_train))
        
        # VAL 
        modello.eval()
        metrics_sum = {'acc': 0.0, 'p': 0.0, 'r': 0.0, 'f1': 0.0}
        
        with torch.no_grad():
            for batch_img, batch_lbl in l_val:
                batch_img, batch_lbl = batch_img.to(device), batch_lbl.to(device)
                
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    # Val senza TTA per velocità e per selezionare il modello che generalizza meglio senza "aiutini"
                    emb = modello(batch_img)
                    _, logits, target_locale = protonet_loss(emb, batch_lbl, config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY)
                    acc, p, r, f1, _, _ = calcola_metriche(logits, target_locale, batch_lbl, config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY)
                
                metrics_sum['acc'] += acc; metrics_sum['p'] += p; metrics_sum['r'] += r; metrics_sum['f1'] += f1
        
        n_val = len(l_val)
        val_acc = metrics_sum['acc'] / n_val
        val_f1 = metrics_sum['f1'] / n_val
        
        storia['val_acc'].append(val_acc)
        storia['val_f1'].append(val_f1)
        storia['val_precision'].append(metrics_sum['p'] / n_val)
        storia['val_recall'].append(metrics_sum['r'] / n_val)
        
        # scheduler.step()
        
        print(f"Ep {ep+1} | Loss: {storia['train_loss'][-1]:.4f} | Val Acc: {val_acc:.2%} | Val F1: {val_f1:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            if isinstance(modello, nn.DataParallel): state_dict = modello.module.state_dict()
            else: state_dict = modello.state_dict()
            best_weights = {k: v.cpu() for k, v in state_dict.items()}
            
            torch.save({'model_state_dict': best_weights, 'val_metrics': {'acc': best_acc}}, 
                       config.CARTELLA_OUTPUT / f"fold_{fold_idx}_best_model.pth")
            print(f"  -> Checkpoint salvato.")
            
    visualizza_training_fold(storia, fold_idx, config.CARTELLA_OUTPUT)
    
    # 5. Testing (Novel Classes) con D4 TTA (8 Views)
    print("Caricamento migliori pesi per il test...")
    modello_test = PrototypicalResNet(pretrained=False).to(device)
    modello_test.load_state_dict(best_weights)
    if torch.cuda.device_count() > 1: modello_test = nn.DataParallel(modello_test)
    modello_test.eval()
    
    metrics_test_sum = {'acc': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    preds_totali, targets_totali = [], []
    
    with torch.no_grad():
        for batch_img, batch_lbl in tqdm(l_test, desc="Testing Novel (D4 TTA)"):
            batch_img, batch_lbl = batch_img.to(device), batch_lbl.to(device)
            
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                
                # APPLICAZIONE TTA D4 (8 VISTE) 
                emb = forward_tta_all(
                    modello_test, 
                    batch_img, 
                    config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY
                )
                
                _, logits, target_locale = protonet_loss(emb, batch_lbl, config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY)
                acc, p, r, f1, preds_glob, targs_glob = calcola_metriche(logits, target_locale, batch_lbl, config.TEST_N_WAY, config.TEST_K_SHOT, config.TEST_Q_QUERY)
            
            metrics_test_sum['acc'] += acc; metrics_test_sum['f1'] += f1
            metrics_test_sum['precision'] += p; metrics_test_sum['recall'] += r
            
            preds_totali.extend(preds_glob.cpu().numpy())
            targets_totali.extend(targs_glob.cpu().numpy())
            
    n_batches = len(l_test)
    final_metrics = {k: v / n_batches for k, v in metrics_test_sum.items()}
    print(f"--> Fold {fold_idx} Result (D4 TTA): Acc={final_metrics['acc']:.2%} | F1={final_metrics['f1']:.4f}")
    
    del modello, modello_test, optimizer, scaler, l_train, l_val, l_test
    gc.collect()
    torch.cuda.empty_cache()
    
    return final_metrics, preds_totali, targets_totali

In [ ]:
# 1. CONFIGURAZIONE PATH E PARAMETRI
PROTO_CONFIG.EPOCHE_PER_FOLD = 10 

# Opzione Resume: Inserire il path se si vuole continuare un esperimento interrotto
PATH_RESUME = None 
# Esempio: PATH_RESUME = Path("/kaggle/working/esperimenti/2026_01_22_15_08_ProtoNet_...")

if PATH_RESUME:
    OUTPUT_DIR = Path(PATH_RESUME)
    PROTO_CONFIG.CARTELLA_OUTPUT = OUTPUT_DIR 
    print(f"-> RESUME MODE: Proseguimento esperimento in {OUTPUT_DIR}")
else:
    OUTPUT_DIR = PROTO_CONFIG.CARTELLA_OUTPUT
    print(f"-> NEW EXPERIMENT: Creazione directory {OUTPUT_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_RISULTATI = OUTPUT_DIR / "risultati_parziali.csv"

# 2. SETUP HARDWARE E DATI
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

# Caricamento Indici (se non presenti)
if 'df_dati' not in locals():
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")

# Caricamento Mapping Classi
with open(CONFIG.CARTELLA_INDICI / "mappatura_classi.json") as f:
    mappatura_classi = json.load(f)
ind_to_class = {v: k for k, v in mappatura_classi.items()}

# 3. GESTIONE STATO E CARICAMENTO FOLD
folds_gia_fatti = []
results = []

if FILE_RISULTATI.exists():
    print("Recupero risultati parziali esistenti...")
    df_esistente = pd.read_csv(FILE_RISULTATI)
    folds_gia_fatti = df_esistente["Fold_ID_Univoco"].astype(str).tolist()
    results = df_esistente.to_dict('records')
    print(f"-> Fold completati: {len(folds_gia_fatti)}")

# Caricamento configurazione semantica manuale
PATH_MANUAL_FOLDS = CONFIG.CARTELLA_INDICI / "folds_manuali.json"
if not PATH_MANUAL_FOLDS.exists():
    raise FileNotFoundError(f"File configurazione fold non trovato: {PATH_MANUAL_FOLDS}")

with open(PATH_MANUAL_FOLDS, "r") as f:
    dati_manuali = json.load(f)

# Parsing della struttura fold
folds_target = []
descrizioni_fold = {} 

for item in dati_manuali:
    t_ids = tuple(item["novel_classes_ids"])
    folds_target.append(t_ids)
    # Mapping indice 0-based -> descrizione
    descrizioni_fold[item["fold_idx"] - 1] = item["descrizione"]

NUMERO_FOLD_DA_ESEGUIRE = len(folds_target)
print(f"\nPiano di Esecuzione ({NUMERO_FOLD_DA_ESEGUIRE} Scenari):")
for k, f in enumerate(folds_target):
    status = "[COMPLETATO]" if f"{sorted(f)[0]}-{sorted(f)[1]}" in folds_gia_fatti else "[DA ESEGUIRE]"
    print(f"  Fold {k+1}: {descrizioni_fold.get(k)} {status}")

# Backup della configurazione usata
with open(OUTPUT_DIR / "configurazione_folds_usata.json", "w") as f:
    json.dump({"modalita": "MANUALE", "folds": dati_manuali}, f, indent=4)

# 4. LOOP DI TRAINING
all_classes = list(range(8))

for i, novel_idx in enumerate(folds_target):
    # Identificativi
    novel_sorted = sorted(novel_idx)
    fold_id = f"{novel_sorted[0]}-{novel_sorted[1]}"
    descrizione_corrente = descrizioni_fold.get(i, "Generico")
    fold_progressivo = i + 1  # Usiamo l'indice naturale del fold semantico

    # Skip se già processato
    if fold_id in folds_gia_fatti:
        continue

    # Definizione Base Classes (Tutte tranne le novel)
    base_idx = tuple(c for c in all_classes if c not in novel_idx)
    nomi_novel = [ind_to_class[idx] for idx in novel_sorted]

    # Garbage Collection preventiva
    gc.collect()
    torch.cuda.empty_cache()

    try:
        print(f"\n{'='*80}")
        print(f"RUNNING FOLD {fold_progressivo}/{NUMERO_FOLD_DA_ESEGUIRE} | ID: {fold_id}")
        print(f"Scenario: {descrizione_corrente}")
        print(f"Novel Classes: {nomi_novel} (IDs: {novel_idx})")
        print(f"{'='*80}")
        
        # --- ESECUZIONE ---
        # Restituisce: Dizionario Metriche, Predizioni, Target
        metriche_test, preds, targs = esegui_fold(
            fold_progressivo, base_idx, novel_idx, df_dati, PROTO_CONFIG, device
        )
        
        # --- VISUALIZZAZIONE ---
        visualizza_confusion_matrix_fewshot(
            targs, preds, nomi_novel, 
            fold_idx=f"{fold_progressivo}_{fold_id}", 
            path_out=OUTPUT_DIR
        )
        
        # --- REGISTRAZIONE ---
        nuovo_record = {
            "Fold_Progressivo": fold_progressivo,
            "Fold_ID_Univoco": fold_id,
            "Descrizione_Scenario": descrizione_corrente,
            "Novel_Classes_IDs": str(list(novel_idx)),
            "Novel_Classes_Names": str(nomi_novel),
            # Metriche dettagliate
            "Test_Accuracy": metriche_test['acc'],
            "Test_F1": metriche_test['f1'],
            "Test_Precision": metriche_test['precision'],
            "Test_Recall": metriche_test['recall']
        }
        results.append(nuovo_record)
        
        # Salvataggio incrementale
        df_log = pd.DataFrame(results)
        df_log.to_csv(FILE_RISULTATI, index=False)
        
        print(f"--> Fold Completato. Acc: {metriche_test['acc']:.2%} | F1: {metriche_test['f1']:.4f}")
        
    except Exception as e:
        print(f"[ERRORE CRITICO] Fold {fold_id} fallito: {e}")
        # Tentativo di recovery memoria
        torch.cuda.empty_cache()
        gc.collect()
        raise e 

    # Cleanup fine ciclo
    del preds, targs
    gc.collect()
    torch.cuda.empty_cache()

# 5. REPORT FINALE AGGREGATO
if results:
    df_res = pd.DataFrame(results)
    
    # Calcolo statistiche per tutte le metriche numeriche
    metriche_cols = ["Test_Accuracy", "Test_F1", "Test_Precision", "Test_Recall"]
    summary = df_res[metriche_cols].agg(['mean', 'std'])
    
    print("\n" + "="*80)
    print(f"REPORT FINALE - {PROTO_CONFIG.NOME_ESPERIMENTO}")
    print("="*80)
    
    # Tabella riassuntiva per fold
    cols_display = ["Fold_ID_Univoco", "Descrizione_Scenario", "Test_Accuracy", "Test_F1"]
    print(df_res[cols_display].to_string(index=False))
    
    print("-" * 80)
    print("MEDIA GENERALE:")
    print(f"Accuracy:  {summary.loc['mean', 'Test_Accuracy']:.2%} ± {summary.loc['std', 'Test_Accuracy']:.2%}")
    print(f"F1-Score:  {summary.loc['mean', 'Test_F1']:.4f} ± {summary.loc['std', 'Test_F1']:.4f}")
    print(f"Precision: {summary.loc['mean', 'Test_Precision']:.4f} ± {summary.loc['std', 'Test_Precision']:.4f}")
    print(f"Recall:    {summary.loc['mean', 'Test_Recall']:.4f} ± {summary.loc['std', 'Test_Recall']:.4f}")
    print("="*80)
    
    # Salvataggio report testuale
    with open(OUTPUT_DIR / "final_report.txt", "w") as f:
        f.write(f"Esperimento: {PROTO_CONFIG.NOME_ESPERIMENTO}\n")
        f.write(f"Data completamento: {pd.Timestamp.now()}\n\n")
        f.write("RISULTATI PER FOLD:\n")
        f.write(df_res.to_string(index=False))
        f.write("\n\nAGGREGATI GLOBALI:\n")
        f.write(summary.to_string())

else:
    print("Nessun risultato generato.")

# Analisi di sensibilità al variare del numero di shot con i migliori pesi senza TTA E con TTA

In [ ]:
# 0. CONFIGURAZIONE LOGGING E PATH
# Configurazione Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)
logger = logging.getLogger()

# Percorso Pesi (La tua Best Run No-L2 No-Norm)
DIR_PESI_SORGENTE = Path("/kaggle/working/RETRAINING_2026_01_30_14_39/2026_01_30_14_39_B_STANDARD_BEST")

# Output Folder
TIMESTAMP = datetime.now().strftime("%Y_%m_%d_%H_%M")
DIR_OUTPUT_ANALISI = CONFIG.PERCORSO_LAVORO / "esperimenti" / f"{TIMESTAMP}_COMPARATIVA_KSHOT_TTA_vs_NOTTA_600EP"
DIR_OUTPUT_ANALISI.mkdir(parents=True, exist_ok=True)

logger.info(f"SORGENTE PESI: {DIR_PESI_SORGENTE}")
logger.info(f"OUTPUT DUMP:   {DIR_OUTPUT_ANALISI}")
logger.info("IMPOSTAZIONE: 600 EPISODI PER TEST")

# K-Shots da testare
K_SHOT_VALUES = [20, 15, 10, 5, 3, 1]

# 1. RIDEFINIZIONE MODELLO (Compatibile con Lin256_NoNorm)
class PrototypicalResNet(nn.Module):
    """
    Versione Standard (Linear Head, No L2, No Scale).
    Deve corrispondere all'architettura usata per addestrare i pesi in DIR_PESI_SORGENTE.
    """
    def __init__(self, pretrained: bool = False, output_dim: int = 256):
        super().__init__()
        self.backbone = resnet18(weights=None) 
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(in_features=512, out_features=output_dim)

    def forward(self, x):
        out = self.backbone(x)
        out = self.projection_layer(out)
        return out # RAW output

# 2. HELPER TTA
def forward_tta_all(modello, batch_img):
    b, c, h, w = batch_img.size()
    viste = [batch_img]
    viste.append(torch.rot90(batch_img, 1, [2, 3]))
    viste.append(torch.rot90(batch_img, 2, [2, 3]))
    viste.append(torch.rot90(batch_img, 3, [2, 3]))
    img_flip = torch.flip(batch_img, [3])
    viste.append(img_flip)
    viste.append(torch.rot90(img_flip, 1, [2, 3]))
    viste.append(torch.rot90(img_flip, 2, [2, 3]))
    viste.append(torch.rot90(img_flip, 3, [2, 3]))
    
    aug_batch = torch.cat(viste, dim=0) 
    feat_aug = modello(aug_batch)       
    feat_aug = feat_aug.view(8, b, -1)
    return feat_aug.mean(dim=0)

# 3. MAIN EXECUTION LOOP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device attivo: {device}")

# Caricamento Dati e Fold
if 'df_dati' not in locals():
    logger.info("Caricamento indice CSV...")
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")

with open(CONFIG.CARTELLA_INDICI / "folds_manuali.json", "r") as f:
    dati_manuali = json.load(f)

results_list = []

logger.info(f"Inizio analisi su {len(dati_manuali)} Folds.")

# Loop Folds
for item in dati_manuali:
    fold_idx = item["fold_idx"]
    fold_uid = item["fold_id_univoco"]
    desc = item["descrizione"]
    novel_ids = tuple(item["novel_classes_ids"])
    
    logger.info(f"--- Processing Fold {fold_idx}: {desc} ---")
    
    # 1. Verifica Checkpoint
    path_ckpt = DIR_PESI_SORGENTE / f"fold_{fold_idx}_best_model.pth"
    if not path_ckpt.exists():
        logger.warning(f"Checkpoint mancante: {path_ckpt}. Skipping.")
        continue
        
    # 2. Setup Modello
    try:
        modello = PrototypicalResNet(pretrained=False).to(device)
        ckpt = torch.load(path_ckpt, map_location=device, weights_only=False)
        state_dict = {k.replace("module.", ""): v for k, v in ckpt['model_state_dict'].items()}
        modello.load_state_dict(state_dict)
        modello.eval()
        logger.info(f"Modello caricato correttamente per Fold {fold_idx}")
    except Exception as e:
        logger.error(f"Errore caricamento modello Fold {fold_idx}: {e}")
        continue

    # 3. Setup Dataset (Una volta per fold)
    df_test = filtra_dataset(df_dati, novel_ids)
    df_test = df_test[df_test["split"] == "test"].copy()
    ds_test = LIVECellDataset(df_test, ottieni_trasformazioni("val"))
    logger.info(f"Dataset Test creato: {len(ds_test)} immagini.")

    # 4. Loop K-Shots
    for k in K_SHOT_VALUES:
        logger.info(f"  > Testing K={k} (600 Episodi)...")
        
        # --- MODIFICA CRUCIALE: 600 EPISODI ---
        NUM_EPISODI = 600
        
        # Sampler & Loader
        s_test = CampionatoreEpisodico(ds_test.etichette, PROTO_CONFIG.TEST_N_WAY, k, PROTO_CONFIG.TEST_Q_QUERY, NUM_EPISODI)
        l_test = DataLoader(ds_test, batch_sampler=s_test, num_workers=2, pin_memory=True)
        
        acc_no_sum = 0.0
        acc_tta_sum = 0.0
        n_batches = 0
        
        with torch.no_grad():
            for batch_img, batch_lbl in l_test:
                batch_img = batch_img.to(device)
                batch_lbl = batch_lbl.to(device)
                
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    # A. NO TTA
                    emb = modello(batch_img)
                    _, logits, targs = protonet_loss(emb, batch_lbl, PROTO_CONFIG.TEST_N_WAY, k, PROTO_CONFIG.TEST_Q_QUERY)
                    acc_no, _, _, _, _, _ = calcola_metriche(logits, targs, batch_lbl, PROTO_CONFIG.TEST_N_WAY, k, PROTO_CONFIG.TEST_Q_QUERY)
                    
                    # B. WITH TTA
                    emb_tta = forward_tta_all(modello, batch_img)
                    _, logits_tta, _ = protonet_loss(emb_tta, batch_lbl, PROTO_CONFIG.TEST_N_WAY, k, PROTO_CONFIG.TEST_Q_QUERY)
                    acc_tta, _, _, _, _, _ = calcola_metriche(logits_tta, targs, batch_lbl, PROTO_CONFIG.TEST_N_WAY, k, PROTO_CONFIG.TEST_Q_QUERY)
                
                acc_no_sum += acc_no
                acc_tta_sum += acc_tta
                n_batches += 1
        
        mean_no = acc_no_sum / n_batches
        mean_tta = acc_tta_sum / n_batches
        
        logger.info(f"    Result K={k}: NoTTA={mean_no:.1%} | TTA={mean_tta:.1%}")
        
        results_list.append({
            "fold_id": fold_uid,
            "descrizione": desc,
            "k_shot": k,
            "acc_no_tta": mean_no,
            "acc_tta": mean_tta
        })

    # Pulizia memoria tra fold
    del modello, l_test, ds_test
    gc.collect()
    torch.cuda.empty_cache()

# 4. SALVATAGGIO OUTPUT
if len(results_list) > 0:
    df_all = pd.DataFrame(results_list)
    
    # Split in due dataframe per compatibilità con il codice grafico successivo
    df_tta = df_all[["fold_id", "descrizione", "k_shot", "acc_tta"]].rename(columns={"acc_tta": "accuracy"})
    df_no = df_all[["fold_id", "descrizione", "k_shot", "acc_no_tta"]].rename(columns={"acc_no_tta": "accuracy"})
    
    path_tta = DIR_OUTPUT_ANALISI / "risultati_tta.csv"
    path_no = DIR_OUTPUT_ANALISI / "risultati_no_tta.csv"
    
    df_tta.to_csv(path_tta, index=False)
    df_no.to_csv(path_no, index=False)
    
    logger.info("="*60)
    logger.info(f"ANALISI 600 EPISODI COMPLETATA. CSV SALVATI IN: {DIR_OUTPUT_ANALISI}")
    logger.info("È necessario eseguire la cella successiva per ottenere il grafico.")
else:
    logger.error("Nessun risultato generato.")

# Cella di visualizzazione per l'analisi di sensibilità. Viene mostrata la differenza tra l'esperimento A e B

In [ ]:
# RECUPERO DEL PERCORSO 
if 'DIR_OUTPUT_ANALISI' not in locals():
    # DIR_OUTPUT_ANALISI = Path("/kaggle/working/esperimenti/...") 
    raise ValueError("Variabile DIR_OUTPUT_ANALISI non trovata. Inserisci il percorso manualmente.")

path_tta = DIR_OUTPUT_ANALISI / "risultati_tta.csv"
path_no_tta = DIR_OUTPUT_ANALISI / "risultati_no_tta.csv"

# 1. Caricamento
df_tta = pd.read_csv(path_tta)
df_no = pd.read_csv(path_no_tta)

# 2. Aggregazione Statistica
stats_tta = df_tta.groupby('k_shot')['accuracy'].agg(['mean', 'std']).reset_index().sort_values('k_shot', ascending=False)
stats_no = df_no.groupby('k_shot')['accuracy'].agg(['mean', 'std']).reset_index().sort_values('k_shot', ascending=False)

# 3. Configurazione Stile "Slide Presentation"
BG_COLOR = "#f5f7f9"       # Sfondo richiesto
TEXT_COLOR = "#2c3e50"     # Dark Slate (ottimo contrasto su f5f7f9)
GRID_COLOR = "#bdc3c7"     # Grigio chiaro per la griglia
C_TTA = "#27ae60"          # Verde vivido
C_NO = "#c0392b"           # Rosso spento

# Setup Figura
fig, ax = plt.subplots(figsize=(12, 8), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)

# TRACCIAMENTO CURVE 

# 1. Standard No TTA
ax.plot(stats_no['k_shot'], stats_no['mean'], color=C_NO, marker='o', 
        linestyle='--', linewidth=2.5, label='Standard (No TTA)', zorder=2)
# Area di confidenza
ax.fill_between(stats_no['k_shot'], stats_no['mean']-stats_no['std'], stats_no['mean']+stats_no['std'], 
                color=C_NO, alpha=0.1, zorder=1)

# 2. TTA (Highlight)
ax.plot(stats_tta['k_shot'], stats_tta['mean'], color=C_TTA, marker='s', 
        linestyle='-', linewidth=3.5, label='Con TTA (8-View)', zorder=4)
# Area di confidenza
ax.fill_between(stats_tta['k_shot'], stats_tta['mean']-stats_tta['std'], stats_tta['mean']+stats_tta['std'], 
                color=C_TTA, alpha=0.15, zorder=3)

# ANNOTAZIONI 
for k in stats_tta['k_shot']:
    val_t = stats_tta[stats_tta['k_shot']==k]['mean'].values[0]
    val_n = stats_no[stats_no['k_shot']==k]['mean'].values[0]
    delta = val_t - val_n
    
    # Delta (Guadagno)
    ax.annotate(f"+{delta:.1%}", xy=(k, val_t), xytext=(k, val_t + 0.03),
                arrowprops=dict(arrowstyle='->', color=TEXT_COLOR, lw=1.2),
                ha='center', color=C_TTA, fontweight='bold', fontsize=11, zorder=5)
    
    # Valore puntuale Baseline
    ax.text(k, val_n - 0.04, f"{val_n:.1%}", ha='center', color=C_NO, fontsize=9, fontweight='bold', zorder=5)

# STYLING DEGLI ASSI 
ax.set_title("Robustezza Few-Shot: Impatto della Test-Time Augmentation", 
             fontsize=18, pad=25, fontweight='bold', color=TEXT_COLOR)

ax.set_xlabel("Numero di Shot (K)", fontsize=13, fontweight='bold', color=TEXT_COLOR, labelpad=10)
ax.set_ylabel("Accuratezza Media (5-Fold CV)", fontsize=13, fontweight='bold', color=TEXT_COLOR, labelpad=10)

# Gestione limiti e tick
ax.set_xlim(21, 0)
ax.set_xticks([20, 15, 10, 5, 3, 1])
ax.set_ylim(0.5, 1.0)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

# Colore dei tick
ax.tick_params(axis='x', colors=TEXT_COLOR, labelsize=11)
ax.tick_params(axis='y', colors=TEXT_COLOR, labelsize=11)

# Gestione Bordi (Spines) - Rimuoviamo Top e Right per pulizia
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color(TEXT_COLOR)
ax.spines['bottom'].set_color(TEXT_COLOR)

# Griglia
ax.grid(True, axis='y', linestyle='--', alpha=0.5, color=GRID_COLOR)
ax.grid(False, axis='x') # Rimuoviamo griglia verticale per pulizia

# Legenda con sfondo coordinato
legend = ax.legend(loc='lower left', frameon=True, facecolor=BG_COLOR, edgecolor=GRID_COLOR, fontsize=11)
for text in legend.get_texts():
    text.set_color(TEXT_COLOR)

# Salvataggio
plt.tight_layout()
plt.savefig(DIR_OUTPUT_ANALISI / "grafico_comparativo_slide_style.png", dpi=300, facecolor=BG_COLOR)
plt.show()

# Cella di visualizzazione per rappresentare l'andamento della loss

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from torchvision import models

# 1. CONFIGURAZIONE STILE GRAFICO
STYLE = {
    "bg_color": "#f5f7f9",      # Sfondo Slide
    "text_color": "#4a749d",    # Testo, Assi, Griglia
    "color_loss": "#e74c3c",    # Rosso (Loss Corrente)
    "color_ref": "#95a5a6",     # Grigio (Loss di Riferimento/Baseline)
    "color_acc": "#27ae60",     # Verde (Accuracy)
    "color_f1": "#8e44ad",      # Viola (F1 Score)
    "font_title": 16,
    "font_subtitle": 14,
    "font_label": 11
}

# 2. DEFINIZIONE ARCHITETTURE
class ProtoNet_Standard(nn.Module): # EXP A: Baseline
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, 256)
    def forward(self, x):
        return self.projection_layer(self.backbone(x))

class ProtoNet_L2Scale(nn.Module): # EXP C: Proposed Method
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, 256)
        self.scale = nn.Parameter(torch.tensor(10.0))
    def forward(self, x):
        out = self.projection_layer(self.backbone(x))
        out = F.normalize(out, p=2, dim=1)
        return out * self.scale

# 3. MOTORE DI TRAINING & TRACKING 
def train_and_track_full(model_class, exp_name, device):
    print(f"\n>>> Avvio Training & Validation: {exp_name}")
    
    # Selezione Dati (Fold 1 per analisi dinamiche)
    FOLD_IDX = 1
    fold_data = next(f for f in dati_manuali if f['fold_idx'] == FOLD_IDX)
    base_ids = tuple(c for c in range(8) if c not in fold_data['novel_classes_ids'])
    df_base = filtra_dataset(df_dati, base_ids)
    
    ds_train = LIVECellDataset(df_base[df_base["split"] == "train"], ottieni_trasformazioni_config("train", usa_clahe=False))
    ds_val = LIVECellDataset(df_base[df_base["split"] == "val"], ottieni_trasformazioni_config("val", usa_clahe=False))
    
    loader_train = DataLoader(ds_train, batch_sampler=CampionatoreEpisodico(ds_train.etichette, PROTO_CONFIG.TRAIN_N_WAY, PROTO_CONFIG.TRAIN_K_SHOT, PROTO_CONFIG.TRAIN_Q_QUERY, PROTO_CONFIG.EPISODI_TRAIN), num_workers=2)
    loader_val = DataLoader(ds_val, batch_sampler=CampionatoreEpisodico(ds_val.etichette, PROTO_CONFIG.VAL_N_WAY, PROTO_CONFIG.VAL_K_SHOT, PROTO_CONFIG.VAL_Q_QUERY, PROTO_CONFIG.EPISODI_VAL), num_workers=2)
    
    model = model_class().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=PROTO_CONFIG.LEARNING_RATE, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler()
    
    history = {"loss": [], "val_acc": [], "val_f1": []}
    
    for ep in range(PROTO_CONFIG.EPOCHE_PER_FOLD):
        # Training Loop
        model.train()
        ep_loss = 0.0
        pbar = tqdm(loader_train, desc=f"{exp_name} Ep {ep+1}", leave=False)
        
        for bx, by in pbar:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                emb = model(bx)
                loss, _, _ = protonet_loss(emb, by, PROTO_CONFIG.TRAIN_N_WAY, PROTO_CONFIG.TRAIN_K_SHOT, PROTO_CONFIG.TRAIN_Q_QUERY)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ep_loss += loss.item()
        
        # Validation Loop
        model.eval()
        acc_sum = 0.0
        f1_sum = 0.0
        with torch.no_grad():
            for bx, by in loader_val:
                bx, by = bx.to(device), by.to(device)
                with torch.amp.autocast('cuda'):
                    emb = model(bx)
                    _, logits, t_loc = protonet_loss(emb, by, PROTO_CONFIG.VAL_N_WAY, PROTO_CONFIG.VAL_K_SHOT, PROTO_CONFIG.VAL_Q_QUERY)
                    acc, _, _, f1, _, _ = calcola_metriche(logits, t_loc, by, PROTO_CONFIG.VAL_N_WAY, PROTO_CONFIG.VAL_K_SHOT, PROTO_CONFIG.VAL_Q_QUERY)
                acc_sum += acc
                f1_sum += f1
        
        # Logging
        avg_loss = ep_loss / len(loader_train)
        avg_acc = acc_sum / len(loader_val)
        avg_f1 = f1_sum / len(loader_val)
        
        history["loss"].append(avg_loss)
        history["val_acc"].append(avg_acc)
        history["val_f1"].append(avg_f1)
        print(f"  Ep {ep+1}: Loss={avg_loss:.4f} | Val Acc={avg_acc:.2%} | F1={avg_f1:.4f}")
        
    return history

# 4. FUNZIONE DI VISUALIZZAZIONE FINALE (ITALIANO + CONFRONTO)
def genera_dashboard_finale(history, exp_title, filename, ref_history=None, ref_label="Baseline"):
    epochs = range(1, len(history["loss"]) + 1)
    
    # Creazione Figura
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor=STYLE["bg_color"])
    
    # Titolo Principale
    fig.suptitle(f"{exp_title}: Dinamiche di Apprendimento", 
                 fontsize=STYLE["font_title"], fontweight='bold', color=STYLE["text_color"], y=0.98)
    
    # SUBPLOT 1: LOSS 
    ax1.set_facecolor(STYLE["bg_color"])
    
    # Linea di Riferimento (Opzionale, per confronto)
    if ref_history:
        ax1.plot(epochs, ref_history["loss"], marker='', linestyle='--', linewidth=2, 
                 color=STYLE["color_ref"], alpha=0.7, label=f"Loss {ref_label}")

    # Linea Loss Corrente
    ax1.plot(epochs, history["loss"], marker='o', linestyle='-', linewidth=2.5, 
             color=STYLE["color_loss"], label="Loss di Training")
    
    ax1.set_title("Convergenza della Loss", fontsize=STYLE["font_subtitle"], 
                  fontweight='bold', color=STYLE["text_color"], pad=10)
    ax1.set_xlabel("Epoche", fontsize=STYLE["font_label"], fontweight='bold', color=STYLE["text_color"])
    ax1.set_ylabel("Valore Loss", fontsize=STYLE["font_label"], fontweight='bold', color=STYLE["text_color"])
    
    # Limite fisso per comparazione diretta
    ax1.set_ylim(0.0, 1.2) 
    
    ax1.grid(True, linestyle='--', alpha=0.4, color=STYLE["text_color"])
    ax1.legend(facecolor=STYLE["bg_color"], edgecolor=STYLE["text_color"], 
               labelcolor=STYLE["text_color"], loc='upper right')

    # SUBPLOT 2: METRICHE 
    ax2.set_facecolor(STYLE["bg_color"])
    ax2.plot(epochs, history["val_acc"], marker='s', linestyle='-', linewidth=2.5, 
             color=STYLE["color_acc"], label="Accuratezza (Validazione)")
    ax2.plot(epochs, history["val_f1"], marker='^', linestyle='--', linewidth=2.5, 
             color=STYLE["color_f1"], label="F1 Score (Validazione)")
    
    ax2.set_title("Metriche di Generalizzazione", fontsize=STYLE["font_subtitle"], 
                  fontweight='bold', color=STYLE["text_color"], pad=10)
    ax2.set_xlabel("Epoche", fontsize=STYLE["font_label"], fontweight='bold', color=STYLE["text_color"])
    ax2.set_ylabel("Punteggio (0.0 - 1.0)", fontsize=STYLE["font_label"], fontweight='bold', color=STYLE["text_color"])
    
    # Limite fisso
    ax2.set_ylim(0.0, 1.05)
    
    ax2.grid(True, linestyle='--', alpha=0.4, color=STYLE["text_color"])
    ax2.legend(facecolor=STYLE["bg_color"], edgecolor=STYLE["text_color"], 
               labelcolor=STYLE["text_color"], loc='lower right')

    # Styling Comune
    for ax in [ax1, ax2]:
        ax.tick_params(colors=STYLE["text_color"])
        for spine in ax.spines.values():
            spine.set_color(STYLE["text_color"])

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    # Salvataggio
    out_path = CONFIG.PERCORSO_LAVORO / filename
    plt.savefig(out_path, dpi=300, facecolor=STYLE["bg_color"])
    print(f"Dashboard salvata in: {out_path}")
    plt.show()

# 5. ESECUZIONE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Training Exp A (Baseline)
hist_a = train_and_track_full(ProtoNet_Standard, "Exp A (Raw Euclidean)", device)

# 2. Training Exp C (Proposed)
hist_c = train_and_track_full(ProtoNet_L2Scale, "Exp C (L2 Norm + Scaling)", device)

# 3. Generazione Grafico Exp A (Standard)
genera_dashboard_finale(hist_a, "Esperimento A (Raw Euclidean)", "final_dashboard_A.png")

# 4. Generazione Grafico Exp C (Con confronto Loss rispetto ad A)
genera_dashboard_finale(
    history=hist_c, 
    exp_title="Esperimento C (L2 + Scale)", 
    filename="final_dashboard_C_confronto.png",
    ref_history=hist_a,       # Passiamo lo storico di A per il confronto
    ref_label="Exp A (Raw)"   # Etichetta per la linea grigia
)

# Analisi della complessità computazionale (esperimento A vs esperimento B) 

In [ ]:
# 0. CONFIGURAZIONE BENCHMARK
DIR_PESI = Path("/kaggle/working/esperimenti/2026_01_26_20_44_ProtoNet_TTA_8D4_ResNet18_Lin256_NO_Clahe2.0_NoNorm4W_Val2W_Test2W")

NUM_REPEAT = 5        
NUM_EPISODI = 100     
K_SHOT_BENCH = 5      
BATCH_SIZE_REAL = (PROTO_CONFIG.TEST_N_WAY * (K_SHOT_BENCH + PROTO_CONFIG.TEST_Q_QUERY)) 

logging.basicConfig(level=logging.INFO, format='%(message)s', handlers=[logging.StreamHandler(sys.stdout)], force=True)
logger = logging.getLogger()

# 1. MODELLO E TTA
class PrototypicalResNet(nn.Module):
    def __init__(self, pretrained=False, output_dim=256):
        super().__init__()
        self.backbone = resnet18(weights=None) 
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(in_features=512, out_features=output_dim)

    def forward(self, x):
        out = self.backbone(x)
        out = self.projection_layer(out)
        return out 

def forward_tta_all(modello, batch_img):
    b, c, h, w = batch_img.size()
    viste = [batch_img]
    viste.append(torch.rot90(batch_img, 1, [2, 3]))
    viste.append(torch.rot90(batch_img, 2, [2, 3]))
    viste.append(torch.rot90(batch_img, 3, [2, 3]))
    img_flip = torch.flip(batch_img, [3])
    viste.append(img_flip)
    viste.append(torch.rot90(img_flip, 1, [2, 3]))
    viste.append(torch.rot90(img_flip, 2, [2, 3]))
    viste.append(torch.rot90(img_flip, 3, [2, 3]))
    
    aug_batch = torch.cat(viste, dim=0) 
    feat_aug = modello(aug_batch)       
    feat_aug = feat_aug.view(8, b, -1)
    return feat_aug.mean(dim=0)

# 2. FUNZIONE DI TIMING E MEMORIA
def benchmark_session(loader, modello, device, use_tta: bool, desc: str):
    modello.eval()
    
    # Pulizia memoria e reset statistiche
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    
    # Warmup
    dummy_img = torch.randn(BATCH_SIZE_REAL, 3, 224, 224).to(device)
    for _ in range(5):
        _ = modello(dummy_img)
    torch.cuda.synchronize() 
    
    start_time = time.perf_counter()
    
    with torch.no_grad():
        for batch_img, batch_lbl in tqdm(loader, desc=desc, leave=False):
            batch_img = batch_img.to(device)
            batch_lbl = batch_lbl.to(device)
            
            with torch.amp.autocast('cuda'):
                if use_tta:
                    emb = forward_tta_all(modello, batch_img)
                else:
                    emb = modello(batch_img)
                
                _, logits, targs = protonet_loss(emb, batch_lbl, PROTO_CONFIG.TEST_N_WAY, K_SHOT_BENCH, PROTO_CONFIG.TEST_Q_QUERY)
                
    torch.cuda.synchronize() 
    end_time = time.perf_counter()
    
    # Picco di memoria allocata in MB
    peak_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    
    return end_time - start_time, peak_mem

# 3. ESECUZIONE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if 'df_dati' not in locals(): df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")
with open(CONFIG.CARTELLA_INDICI / "folds_manuali.json", "r") as f: dati_manuali = json.load(f)

fold_data = dati_manuali[0] 
df_bench = filtra_dataset(df_dati, tuple(fold_data["novel_classes_ids"]))
df_bench = df_bench[df_bench["split"] == "test"].copy()
ds_bench = LIVECellDataset(df_bench, ottieni_trasformazioni("val"))
s_bench = CampionatoreEpisodico(ds_bench.etichette, PROTO_CONFIG.TEST_N_WAY, K_SHOT_BENCH, PROTO_CONFIG.TEST_Q_QUERY, NUM_EPISODI)
l_bench = DataLoader(ds_bench, batch_sampler=s_bench, num_workers=2, pin_memory=True)

path_ckpt = DIR_PESI / "fold_1_best_model.pth"
modello = PrototypicalResNet(pretrained=False).to(device)
ckpt = torch.load(path_ckpt, map_location=device)
state_dict = {k.replace("module.", ""): v for k, v in ckpt['model_state_dict'].items()}
modello.load_state_dict(state_dict)

res_no = []
res_tta = []

print("\n" + "="*70)
print(f"AVVIO BENCHMARK COMPLETO (5 Ripetizioni)")
print(f"Scenario: {fold_data['descrizione']} | K={K_SHOT_BENCH}")
print("="*70)

for i in range(NUM_REPEAT):
    logger.info(f"Run {i+1}/{NUM_REPEAT}...")
    
    t_no, m_no = benchmark_session(l_bench, modello, device, False, "Standard")
    res_no.append({'t': t_no, 'm': m_no})
    
    t_tta, m_tta = benchmark_session(l_bench, modello, device, True, "TTA")
    res_tta.append({'t': t_tta, 'm': m_tta})
    
    logger.info(f"  Standard: {t_no:.2f}s, {m_no:.1f}MB | TTA: {t_tta:.2f}s, {m_tta:.1f}MB")

# 4. REPORT
def get_stats(data):
    ts = [x['t'] for x in data]
    ms = [x['m'] for x in data]
    return statistics.mean(ts), statistics.stdev(ts), statistics.mean(ms)

avg_t_no, std_t_no, avg_m_no = get_stats(res_no)
avg_t_tta, std_t_tta, avg_m_tta = get_stats(res_tta)

print("\n" + "="*75)
print("RISULTATI FINALI BENCHMARK (TEMPO E MEMORIA)")
print("="*75)
print(f"{'MODALITÀ':<20} | {'TEMPO (100 Ep)':<18} | {'VRAM PICCO':<12} | {'VELOCITÀ'}")
print("-" * 75)
print(f"{'Standard (No TTA)':<20} | {avg_t_no:.3f}s ± {std_t_no:.2f} | {avg_m_no:>7.1f} MB | {NUM_EPISODI/avg_t_no:.1f} ep/s")
print(f"{'TTA (8-View)':<20} | {avg_t_tta:.3f}s ± {std_t_tta:.2f} | {avg_m_tta:>7.1f} MB | {NUM_EPISODI/avg_t_tta:.1f} ep/s")
print("-" * 75)
print(f"OVERHEAD MEMORIA: La TTA occupa {avg_m_tta/avg_m_no:.2f}x più memoria VRAM.")
print(f"DIFFERENZA TEMPO: {(avg_t_tta - avg_t_no)/avg_t_no:+.1%} (Differenza trascurabile grazie a I/O Bound)")
print("="*75)

# Comparativa finale su 5 test run per ciascun esperimento per riportare risultati statisticamente solidi

In [ ]:
import sys
import json
import pandas as pd
import torch
import logging
from pathlib import Path

# --- 1. CARICAMENTO DATI MANCANTI (Fix NameError) ---
print("Caricamento configurazioni e dati...")

# Percorsi assoluti per sicurezza
path_json = Path("/kaggle/working/indici/folds_manuali.json")
path_csv = Path("/kaggle/working/indici/indice_livecell.csv")

# 1. Caricamento dati_manuali
if path_json.exists():
    with open(path_json, "r") as f:
        dati_manuali = json.load(f)
    print("Variabile 'dati_manuali' ricaricata.")
else:
    raise FileNotFoundError(f"File non trovato: {path_json}. Esegui le celle di setup iniziali.")

# 2. Caricamento df_dati
if 'df_dati' not in locals():
    df_dati = pd.read_csv(path_csv)
    print("Variabile 'df_dati' ricaricata.")

# 3. Setup Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. CONFIGURAZIONE RECUPERO ---
BASE_PATH = Path("/kaggle/working/RETRAINING_2026_01_30_14_39")
TARGET_DIR = BASE_PATH / "2026_01_30_14_39_E_CLAHE"
FOLD_TARGET_IDX = 5

# Setup Logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', force=True, handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger()

if not TARGET_DIR.exists():
    # Se la cartella non esiste (magari era parziale), la creiamo
    TARGET_DIR.mkdir(parents=True, exist_ok=True)
    logger.info(f"Creata cartella mancante: {TARGET_DIR}")

# --- 3. PREPARAZIONE FOLD ---
fold_data = next((f for f in dati_manuali if f['fold_idx'] == FOLD_TARGET_IDX), None)
novel_ids = tuple(fold_data['novel_classes_ids'])
base_ids = tuple(c for c in range(8) if c not in novel_ids)

logger.info(f"RECUPERO FOLD {FOLD_TARGET_IDX} | Target: {TARGET_DIR}")
logger.info(f"Classi Novel: {novel_ids} | Classi Base: {base_ids}")

# --- 4. ESECUZIONE TRAINING ---
print(f"\nAVVIO TRAINING CHIRURGICO: Fold {FOLD_TARGET_IDX} (E_CLAHE)...")

try:
    esegui_training_fold(
        fold_idx=FOLD_TARGET_IDX, 
        classi_base=base_ids, 
        df_dati=df_dati, 
        config=PROTO_CONFIG, # Si assume che PROTO_CONFIG sia definito nella cella configurazione
        device=device,
        ModelloClasse=ProtoNet_Standard, # EXP E usa Standard
        cartella_output_exp=TARGET_DIR,
        usa_clahe_fold=True # EXP E usa CLAHE
    )
    logger.info(f"\nRECUPERO COMPLETATO. File salvato in: {TARGET_DIR}")

except NameError as e:
    print(f"\nERRORE DI DIPENDENZA: {e}")
    print("Sembra che manchino le funzioni di training (esegui_training_fold, ProtoNet_Standard, etc.).")
    print("Per favore, esegui la cella grande che contiene le definizioni delle funzioni e riprova.")

In [ ]:
import sys
import gc
import json
import time
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm  # Importazione corretta
from torch.utils.data import DataLoader
from torchvision import models
from dataclasses import asdict

# --- 0. CONFIGURAZIONE SESSIONE DI RETRAINING ---
TIMESTAMP = pd.Timestamp.now().strftime("%Y_%m_%d_%H_%M")
OUTPUT_SESSION_DIR = CONFIG.PERCORSO_LAVORO / f"RETRAINING_{TIMESTAMP}"
OUTPUT_SESSION_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout),
                              logging.FileHandler(OUTPUT_SESSION_DIR / "log_addestramento.txt")], force=True)
logger = logging.getLogger()
logger.info(f"Avvio sessione di retraining. Output in: {OUTPUT_SESSION_DIR}")

# --- 1. RIDEFINIZIONE ARCHITETTURE ---
class ProtoNet_Standard(nn.Module):
    def __init__(self, pretrained=True, output_dim=256):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, output_dim)
    def forward(self, x): return self.projection_layer(self.backbone(x))

class ProtoNet_L2Scale(nn.Module):
    def __init__(self, pretrained=True, output_dim=256):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, output_dim)
        self.scale = nn.Parameter(torch.tensor(10.0))
    def forward(self, x):
        out = self.projection_layer(self.backbone(x))
        out = F.normalize(out, p=2, dim=1)
        return out * self.scale

class ProtoNet_MLP(nn.Module):
    def __init__(self, pretrained=True, output_dim=256):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(inplace=True),
            nn.Dropout(p=0.2), nn.Linear(512, output_dim)
        )
    def forward(self, x): return self.projection_layer(self.backbone(x))

# --- 2. PIPELINE DATI CONFIGURABILE ---
def ottieni_trasformazioni_config(split: str, usa_clahe: bool = False) -> A.Compose:
    target_dim = CONFIG_PREPROC.DIM_IMMAGINE
    pipeline = []
    
    if split == "train":
        pipeline.extend([
            A.OneOf([A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0), A.RandomRotate90(p=1.0)], p=0.7),
            A.Affine(scale=(0.8, 1.2), translate_percent=(0.1, 0.1), rotate=(-30, 30),
                     border_mode=cv2.BORDER_CONSTANT, interpolation=cv2.INTER_LINEAR, p=0.3)
        ])
    
    pipeline.append(A.Resize(height=target_dim, width=target_dim, interpolation=cv2.INTER_CUBIC))
    
    if usa_clahe:
        pipeline.append(A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), always_apply=True))

    pipeline.extend([
        A.Normalize(mean=CONFIG_PREPROC.MEDIA_LIVECELL, std=CONFIG_PREPROC.DEV_STD_LIVECELL, max_pixel_value=255.0),
        ToTensorV2()
    ])
    return A.Compose(pipeline)

# --- 3. FUNZIONE DI TRAINING AGGIORNATA ---
def esegui_training_fold(fold_idx, classi_base, df_dati, config, device, ModelloClasse, cartella_output_exp, usa_clahe_fold):
    seed_corrente = SEED_GLOBALE + fold_idx
    imposta_seed(seed_corrente, modalita_deterministica=True)
    g = crea_generatore_riproducibile(seed_corrente)

    df_base = filtra_dataset(df_dati, classi_base)
    ds_train = LIVECellDataset(df_base[df_base["split"] == "train"], ottieni_trasformazioni_config("train", usa_clahe=usa_clahe_fold))
    ds_val = LIVECellDataset(df_base[df_base["split"] == "val"], ottieni_trasformazioni_config("val", usa_clahe=usa_clahe_fold))
    
    s_train = CampionatoreEpisodico(ds_train.etichette, config.TRAIN_N_WAY, config.TRAIN_K_SHOT, config.TRAIN_Q_QUERY, config.EPISODI_TRAIN)
    s_val = CampionatoreEpisodico(ds_val.etichette, config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY, config.EPISODI_VAL)
    l_train = DataLoader(ds_train, batch_sampler=s_train, num_workers=config.NUM_WORKERS, pin_memory=True, worker_init_fn=worker_init_fn, generator=g)
    l_val = DataLoader(ds_val, batch_sampler=s_val, num_workers=config.NUM_WORKERS, pin_memory=True, worker_init_fn=worker_init_fn, generator=g)

    modello = ModelloClasse(pretrained=True).to(device)
    optimizer = torch.optim.AdamW(modello.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    
    best_f1 = 0.0
    path_ckpt = cartella_output_exp / f"fold_{fold_idx}_best_model.pth"

    for ep in range(config.EPOCHE_PER_FOLD):
        modello.train()
        pbar_train = tqdm(l_train, desc=f"Ep {ep+1} Train", leave=False)
        for batch_img, batch_lbl in pbar_train:
            batch_img, batch_lbl = batch_img.to(device), batch_lbl.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                emb = modello(batch_img)
                loss, _, _ = protonet_loss(emb, batch_lbl, config.TRAIN_N_WAY, config.TRAIN_K_SHOT, config.TRAIN_Q_QUERY)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            pbar_train.set_postfix({"loss": f"{loss.item():.4f}"})

        modello.eval()
        val_acc_sum, val_f1_sum = 0.0, 0.0
        pbar_val = tqdm(l_val, desc=f"Ep {ep+1} Val", leave=False)
        with torch.no_grad():
            for batch_img, batch_lbl in pbar_val:
                batch_img, batch_lbl = batch_img.to(device), batch_lbl.to(device)
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    emb = modello(batch_img)
                    _, logits, target_locale = protonet_loss(emb, batch_lbl, config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY)
                    acc, _, _, f1, _, _ = calcola_metriche(logits, target_locale, batch_lbl, config.VAL_N_WAY, config.VAL_K_SHOT, config.VAL_Q_QUERY)
                val_acc_sum += acc
                val_f1_sum += f1
        
        val_acc = val_acc_sum / len(l_val)
        val_f1 = val_f1_sum / len(l_val)
        print(f"  Ep {ep+1}/{config.EPOCHE_PER_FOLD} -> Val Acc: {val_acc:.2%} | Val F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_acc_at_best_f1 = val_acc
            state_dict = modello.state_dict()
            torch.save({'model_state_dict': state_dict, 'val_f1': best_f1, 'val_acc': best_acc_at_best_f1}, path_ckpt)
            print(f"    -> New best model saved (F1: {best_f1:.4f})")
    
    # Verifica post-salvataggio
    if path_ckpt.exists():
        size_mb = path_ckpt.stat().st_size / (1024 * 1024)
        if size_mb < 40:
            logger.error(f"SALVATAGGIO CORROTTO! Fold {fold_idx} pesa solo {size_mb:.2f}MB")
        else:
            logger.info(f"Fold {fold_idx} salvato correttamente. Best F1: {best_f1:.4f} (Acc: {best_acc_at_best_f1:.2%}) | Dim: {size_mb:.2f}MB")
    else:
        logger.error(f"SALVATAGGIO FALLITO! Il file per il Fold {fold_idx} non è stato creato.")
    
    del modello, optimizer, scaler, l_train, l_val
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# --- 4. CONFIGURAZIONE ESECUZIONE ---
# Ordine strategico: dal più veloce/semplice al più complesso
EXPERIMENTS_TO_RUN = [
    {"id": "EXP_B", "name": "B_STANDARD_BEST", "arch_class": ProtoNet_Standard, "use_clahe": False},
    {"id": "EXP_C", "name": "C_L2_SCALE", "arch_class": ProtoNet_L2Scale, "use_clahe": False},
    {"id": "EXP_D", "name": "D_MLP", "arch_class": ProtoNet_MLP, "use_clahe": False},
    {"id": "EXP_E", "name": "E_CLAHE", "arch_class": ProtoNet_Standard, "use_clahe": True},
]

if 'df_dati' not in locals(): df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")
with open(CONFIG.CARTELLA_INDICI / "folds_manuali.json", "r") as f: dati_manuali = json.load(f)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_classes = list(range(8))

# --- 5. LOOP DI ADDESTRAMENTO PRINCIPALE ---
for exp_config in EXPERIMENTS_TO_RUN:
    exp_name = exp_config['name']
    Modello = exp_config['arch_class']
    use_clahe = exp_config['use_clahe']
    
    exp_dir = OUTPUT_SESSION_DIR / f"{TIMESTAMP}_{exp_name}"
    exp_dir.mkdir(exist_ok=True)
    
    logger.info(f"\n{'='*60}\nINIZIO ADDESTRAMENTO: {exp_name}\n{'='*60}")
    
    for fold_item in dati_manuali:
        fold_idx = fold_item['fold_idx']
        novel_idx = tuple(fold_item['novel_classes_ids'])
        base_idx = tuple(c for c in all_classes if c not in novel_idx)
        
        print(f"\nTraining Fold {fold_idx}/{len(dati_manuali)} (Novel: {novel_idx})...")
        
        esegui_training_fold(
            fold_idx, base_idx, df_dati,
            PROTO_CONFIG, device,
            ModelloClasse=Modello,
            cartella_output_exp=exp_dir,
            usa_clahe_fold=use_clahe
        )
        
logger.info("\n\nSESSIONE DI ADDESTRAMENTO COMPLETATA.")
logger.info("I pesi sono stati salvati nelle rispettive cartelle all'interno di:")
logger.info(f"{OUTPUT_SESSION_DIR}")

In [ ]:
import sys
import gc
import json
import logging
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm 
from torch.utils.data import DataLoader
from torchvision import models
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix

SOURCE_DIR = Path("/kaggle/working/RETRAINING_2026_01_30_14_39")

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"ERRORE: La cartella {SOURCE_DIR} non esiste.")

TIMESTAMP_TEST = pd.Timestamp.now().strftime("%Y_%m_%d_%H_%M")
BENCHMARK_DIR = CONFIG.PERCORSO_LAVORO / "benchmark_finale" / f"{TIMESTAMP_TEST}_FINAL_RESULTS_WITH_PLOTS"
PLOTS_DIR = BENCHMARK_DIR / "confusion_matrices"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

# Configurazione Logger per scrivere sia su file che su console
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout),
                              logging.FileHandler(BENCHMARK_DIR / "log_testing.txt")], force=True)
logger = logging.getLogger()
logger.info(f"AVVIO BENCHMARK + GRAFICI SU: {SOURCE_DIR}")

# --- 1. UTILITIES ---

def plot_save_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    """Genera CM con sfondo trasparente."""
    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(divide='ignore', invalid='ignore'):
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm)

    annot = [[f"{val}\n({pct:.1%})" for val, pct in zip(row_c, row_p)] for row_c, row_p in zip(cm, cm_norm)]

    plt.figure(figsize=(8, 7))
    sns.heatmap(cm_norm, annot=np.array(annot), fmt='', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Recall'}, square=True, linewidths=1, linecolor='white')
    
    plt.title(title, fontsize=12, fontweight='bold', pad=15)
    plt.ylabel('Ground Truth', fontsize=10, fontweight='bold')
    plt.xlabel('Prediction', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, transparent=True, bbox_inches='tight')
    plt.close()

# Ridefinizione Architetture
class ProtoNet_Standard(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        self.backbone = models.resnet18(weights=None); self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, output_dim)
    def forward(self, x): return self.projection_layer(self.backbone(x))

class ProtoNet_L2Scale(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        self.backbone = models.resnet18(weights=None); self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, output_dim)
        self.scale = nn.Parameter(torch.tensor(10.0))
    def forward(self, x): return F.normalize(self.projection_layer(self.backbone(x)), p=2, dim=1) * self.scale

class ProtoNet_MLP(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        self.backbone = models.resnet18(weights=None); self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Sequential(nn.Linear(512, 512), nn.ReLU(True), nn.Dropout(0.2), nn.Linear(512, output_dim))
    def forward(self, x): return self.projection_layer(self.backbone(x))

def forward_tta_all(modello, batch_img):
    b = batch_img.size(0)
    viste = [batch_img, torch.rot90(batch_img, 1, [2, 3]), torch.rot90(batch_img, 2, [2, 3]), torch.rot90(batch_img, 3, [2, 3])]
    img_flip = torch.flip(batch_img, [3])
    viste.extend([img_flip, torch.rot90(img_flip, 1, [2, 3]), torch.rot90(img_flip, 2, [2, 3]), torch.rot90(img_flip, 3, [2, 3])])
    aug_batch = torch.cat(viste, dim=0)
    feat_aug = modello(aug_batch)
    return feat_aug.view(8, b, -1).mean(dim=0)

def carica_modello(arch, path, device):
    if arch == "Standard": m = ProtoNet_Standard()
    elif arch == "L2Scale": m = ProtoNet_L2Scale()
    elif arch == "MLP": m = ProtoNet_MLP()
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m.load_state_dict({k.replace("module.", ""): v for k, v in ckpt['model_state_dict'].items()})
    m.to(device).eval()
    return m

# --- 2. CONFIGURAZIONE ESPERIMENTI ---
TEST_CONFIG = [
    {"id": "EXP_A", "label": "A_NO_TTA", "folder_suffix": "B_STANDARD_BEST", "arch": "Standard", "tta": False, "clahe": False},
    {"id": "EXP_B", "label": "B_STANDARD_TTA", "folder_suffix": "B_STANDARD_BEST", "arch": "Standard", "tta": True, "clahe": False},
    {"id": "EXP_C", "label": "C_L2_SCALE", "folder_suffix": "C_L2_SCALE", "arch": "L2Scale", "tta": True, "clahe": False},
    {"id": "EXP_D", "label": "D_MLP", "folder_suffix": "D_MLP", "arch": "MLP", "tta": True, "clahe": False},
    {"id": "EXP_E", "label": "E_CLAHE", "folder_suffix": "E_CLAHE", "arch": "Standard", "tta": True, "clahe": True},
]

# Dati
if 'df_dati' not in locals(): df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")
with open(CONFIG.CARTELLA_INDICI / "folds_manuali.json", "r") as f: dati_manuali = json.load(f)
with open(CONFIG.CARTELLA_INDICI / "mappatura_classi.json", "r") as f: mappa_classi = json.load(f)
id_to_name = {v: k for k, v in mappa_classi.items()}

NUM_RUNS = 5
NUM_EPISODI = 600
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark_results = [] # Lista globale per il CSV

# --- 3. MAIN TESTING LOOP ---
print(f"{'='*80}\nSTART BENCHMARK (5 Runs x {NUM_EPISODI} Ep) - LOGGING ATTIVO\n{'='*80}")

for cfg in TEST_CONFIG:
    exp_label = cfg['label']
    folder_suffix = cfg['folder_suffix']
    
    # Ricerca Cartella
    candidates = list(SOURCE_DIR.glob(f"*{folder_suffix}"))
    if not candidates: 
        logger.error(f"Cartella non trovata per suffix '{folder_suffix}'. Skipping.")
        continue
    exp_path = candidates[0]
    
    logger.info(f"Processing: {exp_label}...")
    
    exp_plot_dir = PLOTS_DIR / exp_label
    exp_plot_dir.mkdir(exist_ok=True)

    for fold_item in dati_manuali:
        fold_idx = fold_item['fold_idx']
        novel_ids = tuple(fold_item['novel_classes_ids'])
        fold_desc = fold_item['descrizione']
        sorted_novel_ids = sorted(novel_ids)
        class_names_fold = [id_to_name[idx] for idx in sorted_novel_ids]
        
        path_weights = exp_path / f"fold_{fold_idx}_best_model.pth"
        if not path_weights.exists(): 
            logger.warning(f"  [SKIP] Pesi mancanti per Fold {fold_idx}")
            continue
        
        modello = carica_modello(cfg['arch'], path_weights, device)
        ds_test = LIVECellDataset(filtra_dataset(df_dati, novel_ids)[filtra_dataset(df_dati, novel_ids)["split"] == "test"], 
                                  ottieni_trasformazioni_config("val", usa_clahe=cfg['clahe']))
        
        all_preds_global = []
        all_targets_global = []
        
        # Accumulatori per calcolare la media del fold corrente
        current_fold_accs = []
        current_fold_f1s = []
        
        # Loop 5 Run
        for run_i in range(NUM_RUNS):
            g = crea_generatore_riproducibile(42 + run_i)
            loader = DataLoader(ds_test, batch_sampler=CampionatoreEpisodico(ds_test.etichette, 
                                PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY, NUM_EPISODI),
                                num_workers=2, worker_init_fn=worker_init_fn, generator=g)
            
            acc_run = 0.0; f1_run = 0.0
            
            pbar = tqdm(loader, desc=f"{exp_label} F{fold_idx} R{run_i+1}", leave=False)
            
            with torch.no_grad():
                for bx, by in pbar:
                    bx, by = bx.to(device), by.to(device)
                    with torch.amp.autocast('cuda'):
                        emb = forward_tta_all(modello, bx) if cfg['tta'] else modello(bx)
                        _, logits, t_loc = protonet_loss(emb, by, PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY)
                        acc, _, _, f1, _, _ = calcola_metriche(logits, t_loc, by, PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY)
                    
                    acc_run += acc; f1_run += f1
                    
                    # Accumulo per Plot
                    labels_matrix = by.view(PROTO_CONFIG.TEST_N_WAY, -1)
                    classi_episodio_globali = labels_matrix[:, 0]
                    preds_loc = logits.argmax(dim=1)
                    preds_glob = classi_episodio_globali[preds_loc]
                    targets_glob = labels_matrix[:, PROTO_CONFIG.TEST_K_SHOT:].contiguous().view(-1)
                    all_preds_global.extend(preds_glob.cpu().numpy())
                    all_targets_global.extend(targets_glob.cpu().numpy())

            # Calcolo metriche singola run
            final_acc_run = acc_run/len(loader)
            final_f1_run = f1_run/len(loader)
            
            # Salvo nei risultati globali
            benchmark_results.append({
                "Experiment": exp_label, "Fold": fold_idx, "Run": run_i+1, 
                "Acc": final_acc_run, "F1": final_f1_run, "Desc": fold_desc
            })
            
            # Salvo per la media del fold
            current_fold_accs.append(final_acc_run)
            current_fold_f1s.append(final_f1_run)
        
        # --- FINE FOLD: CALCOLO MEDIE E SALVATAGGIO PARZIALE ---
        mean_acc_fold = np.mean(current_fold_accs)
        mean_f1_fold = np.mean(current_fold_f1s)
        
        # 1. Logging Informativo
        logger.info(f"  5 run eseguite.con l'EXP {exp_label} | Fold {fold_idx} | Acc: {mean_acc_fold:.2%} | F1: {mean_f1_fold:.4f}")
        
        # 2. Salvataggio CSV Incrementale
        pd.DataFrame(benchmark_results).to_csv(BENCHMARK_DIR / "detailed_results.csv", index=False)
        
        # 3. Plot Confusion Matrix
        plot_save_confusion_matrix(
            y_true=all_targets_global,
            y_pred=all_preds_global,
            class_names=class_names_fold,
            title=f"{exp_label} - Fold {fold_idx}\n(Avg 5 Runs - Acc: {mean_acc_fold:.2%})",
            save_path=exp_plot_dir / f"fold_{fold_idx}_cm.png"
        )
        
        del modello, ds_test
        gc.collect(); torch.cuda.empty_cache()
    print("-" * 40)

# --- 4. REPORT FINALE ---
if benchmark_results:
    summary = pd.DataFrame(benchmark_results).groupby("Experiment")[["Acc", "F1"]].agg(['mean', 'std'])
    summary.to_csv(BENCHMARK_DIR / "summary_final.csv")
    print("\n\nRISULTATI FINALI:\n", summary)
else:
    print("Nessun risultato.")

In [ ]:
import sys
import gc
import json
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import confusion_matrix

# --- 0. SETUP GLOBALE EXP Z ---
TIMESTAMP = pd.Timestamp.now().strftime("%Y_%m_%d_%H_%M")
EXP_NAME = "Z_MLP_L2_HYBRID"
BASE_DIR = CONFIG.PERCORSO_LAVORO / "ESPERIMENTO_Z" / f"{TIMESTAMP}_{EXP_NAME}"
TRAIN_DIR = BASE_DIR / "weights"
TEST_DIR = BASE_DIR / "test_results"
PLOT_DIR = TEST_DIR / "confusion_matrices"

for d in [TRAIN_DIR, TEST_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout),
                              logging.FileHandler(BASE_DIR / "log_exp_z.txt")], force=True)
logger = logging.getLogger()
logger.info(f"AVVIO ESPERIMENTO Z (Hybrid MLP + L2). Output: {BASE_DIR}")

# --- 1. DEFINIZIONE ARCHITETTURA IBRIDA ---
class ProtoNet_Hybrid_Z(nn.Module):
    def __init__(self, pretrained=True, output_dim=256):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Identity()
        
        # Combinazione: MLP Head...
        self.projection_layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(512, output_dim)
        )
        # ...con Scaling apprendibile
        self.scale = nn.Parameter(torch.tensor(10.0))

    def forward(self, x):
        x = self.backbone(x)
        x = self.projection_layer(x)
        # ...e Normalizzazione L2
        x = F.normalize(x, p=2, dim=1)
        return x * self.scale

# --- 2. CONFIGURAZIONE DATI ---
if 'df_dati' not in locals(): df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")
with open(CONFIG.CARTELLA_INDICI / "folds_manuali.json", "r") as f: dati_manuali = json.load(f)
with open(CONFIG.CARTELLA_INDICI / "mappatura_classi.json", "r") as f: mappa_classi = json.load(f)
id_to_name = {v: k for k, v in mappa_classi.items()}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. FASE DI TRAINING (5 FOLDS) ---
logger.info("\n" + "="*60 + "\nFASE 1: TRAINING (LR Costante, 10 Epoche)\n" + "="*60)

for fold_item in dati_manuali:
    fold_idx = fold_item['fold_idx']
    novel_ids = tuple(fold_item['novel_classes_ids'])
    base_ids = tuple(c for c in range(8) if c not in novel_ids)
    
    print(f"\n>>> Training Fold {fold_idx} (Novel: {novel_ids})...")
    
    # Setup
    imposta_seed(SEED_GLOBALE + fold_idx, modalita_deterministica=True)
    g = crea_generatore_riproducibile(SEED_GLOBALE + fold_idx)
    
    # Dataset (NO CLAHE per coerenza con standard)
    df_base = filtra_dataset(df_dati, base_ids)
    ds_train = LIVECellDataset(df_base[df_base["split"] == "train"], ottieni_trasformazioni_config("train", usa_clahe=False))
    ds_val = LIVECellDataset(df_base[df_base["split"] == "val"], ottieni_trasformazioni_config("val", usa_clahe=False))
    
    loader_train = DataLoader(ds_train, batch_sampler=CampionatoreEpisodico(ds_train.etichette, PROTO_CONFIG.TRAIN_N_WAY, PROTO_CONFIG.TRAIN_K_SHOT, PROTO_CONFIG.TRAIN_Q_QUERY, PROTO_CONFIG.EPISODI_TRAIN), num_workers=2, pin_memory=True, generator=g)
    loader_val = DataLoader(ds_val, batch_sampler=CampionatoreEpisodico(ds_val.etichette, PROTO_CONFIG.VAL_N_WAY, PROTO_CONFIG.VAL_K_SHOT, PROTO_CONFIG.VAL_Q_QUERY, PROTO_CONFIG.EPISODI_VAL), num_workers=2, pin_memory=True, generator=g)
    
    # Init Modello
    modello = ProtoNet_Hybrid_Z(pretrained=True).to(device)
    optimizer = torch.optim.AdamW(modello.parameters(), lr=PROTO_CONFIG.LEARNING_RATE, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler()
    
    best_f1 = 0.0
    path_ckpt = TRAIN_DIR / f"fold_{fold_idx}_best_model.pth"
    
    # Loop Epoche
    for ep in range(PROTO_CONFIG.EPOCHE_PER_FOLD):
        modello.train()
        for bx, by in tqdm(loader_train, desc=f"Fold {fold_idx} Ep {ep+1}", leave=False):
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                emb = modello(bx)
                loss, _, _ = protonet_loss(emb, by, PROTO_CONFIG.TRAIN_N_WAY, PROTO_CONFIG.TRAIN_K_SHOT, PROTO_CONFIG.TRAIN_Q_QUERY)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
        # Validation
        modello.eval()
        acc_sum, f1_sum = 0.0, 0.0
        with torch.no_grad():
            for bx, by in loader_val:
                bx, by = bx.to(device), by.to(device)
                with torch.amp.autocast('cuda'):
                    emb = modello(bx)
                    _, logits, t_loc = protonet_loss(emb, by, PROTO_CONFIG.VAL_N_WAY, PROTO_CONFIG.VAL_K_SHOT, PROTO_CONFIG.VAL_Q_QUERY)
                    acc, _, _, f1, _, _ = calcola_metriche(logits, t_loc, by, PROTO_CONFIG.VAL_N_WAY, PROTO_CONFIG.VAL_K_SHOT, PROTO_CONFIG.VAL_Q_QUERY)
                acc_sum += acc; f1_sum += f1
        
        val_acc = acc_sum / len(loader_val)
        val_f1 = f1_sum / len(loader_val)
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({'model_state_dict': modello.state_dict(), 'val_f1': best_f1}, path_ckpt)
            print(f"  Ep {ep+1}: New Best -> F1: {best_f1:.4f} (Acc: {val_acc:.2%})")
            
    del modello, loader_train, loader_val, optimizer
    gc.collect(); torch.cuda.empty_cache()

# --- 4. FASE DI TESTING (5 RUNS x 600 EP) ---
logger.info("\n" + "="*60 + "\nFASE 2: TESTING (5 Runs x 600 Episodi + TTA)\n" + "="*60)

# Helper TTA
def forward_tta_z(model, x):
    b = x.size(0)
    viste = [x, torch.rot90(x, 1, [2, 3]), torch.rot90(x, 2, [2, 3]), torch.rot90(x, 3, [2, 3])]
    flip = torch.flip(x, [3])
    viste.extend([flip, torch.rot90(flip, 1, [2, 3]), torch.rot90(flip, 2, [2, 3]), torch.rot90(flip, 3, [2, 3])])
    aug = torch.cat(viste, dim=0)
    feat = model(aug).view(8, b, -1)
    # Importante: Normalizzare DOPO aver mediato per mantenere la geometria unitaria
    feat_mean = feat.mean(dim=0)
    return F.normalize(feat_mean, p=2, dim=1) * model.scale

# Plot Function
def plot_cm_z(y_true, y_pred, names, title, path):
    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(divide='ignore', invalid='ignore'): cm_n = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_n = np.nan_to_num(cm_n)
    annot = [[f"{v}\n({p:.1%})" for v, p in zip(r_c, r_p)] for r_c, r_p in zip(cm, cm_n)]
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm_n, annot=np.array(annot), fmt='', cmap='Greens', xticklabels=names, yticklabels=names, cbar=False)
    plt.title(title, fontsize=11, fontweight='bold'); plt.tight_layout()
    plt.savefig(path, transparent=True, dpi=300); plt.close()

results_z = []

for fold_item in dati_manuali:
    fold_idx = fold_item['fold_idx']
    novel_ids = tuple(fold_item['novel_classes_ids'])
    desc = fold_item['descrizione']
    
    path_ckpt = TRAIN_DIR / f"fold_{fold_idx}_best_model.pth"
    if not path_ckpt.exists(): logger.error(f"Pesi mancanti Fold {fold_idx}"); continue
    
    # Load Model (Strictly for testing now)
    modello = ProtoNet_Hybrid_Z(pretrained=False).to(device)
    modello.load_state_dict(torch.load(path_ckpt, map_location=device)['model_state_dict'])
    modello.eval()
    
    ds_test = LIVECellDataset(filtra_dataset(df_dati, novel_ids)[filtra_dataset(df_dati, novel_ids)["split"] == "test"], 
                              ottieni_trasformazioni_config("val", usa_clahe=False))
    
    preds_all, targs_all = [], []
    
    print(f"Testing Fold {fold_idx} ({desc})...")
    
    for run_i in range(5): # 5 Runs
        imposta_seed(42 + run_i, True)
        loader = DataLoader(ds_test, batch_sampler=CampionatoreEpisodico(ds_test.etichette, PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY, 600), num_workers=2)
        
        acc_r, f1_r = 0.0, 0.0
        with torch.no_grad():
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                with torch.amp.autocast('cuda'):
                    # TTA Custom per Exp Z
                    emb = forward_tta_z(modello, bx)
                    _, logits, t_loc = protonet_loss(emb, by, PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY)
                    acc, _, _, f1, _, _ = calcola_metriche(logits, t_loc, by, PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY)
                acc_r += acc; f1_r += f1
                
                # Accumulo per CM
                l_mat = by.view(PROTO_CONFIG.TEST_N_WAY, -1)
                glob_ids = l_mat[:, 0]
                preds_all.extend(glob_ids[logits.argmax(1)].cpu().numpy())
                targs_all.extend(l_mat[:, PROTO_CONFIG.TEST_K_SHOT:].contiguous().view(-1).cpu().numpy())

        results_z.append({"Experiment": EXP_NAME, "Fold": fold_idx, "Run": run_i+1, "Acc": acc_r/len(loader), "F1": f1_r/len(loader)})
        print(f"\r  Run {run_i+1}: Acc={acc_r/len(loader):.2%}", end="")
    
    print("")
    # Plot CM
    plot_cm_z(targs_all, preds_all, [id_to_name[i] for i in sorted(novel_ids)], 
              f"Exp Z (Hybrid) - Fold {fold_idx}", PLOT_DIR / f"fold_{fold_idx}_cm.png")
    
    del modello, ds_test, loader
    gc.collect(); torch.cuda.empty_cache()

# --- 5. REPORT ---
if results_z:
    df = pd.DataFrame(results_z)
    df.to_csv(TEST_DIR / "results_detailed.csv", index=False)
    summary = df.groupby("Experiment")[["Acc", "F1"]].agg(['mean', 'std'])
    print("\nRISULTATI FINALI EXP Z:")
    print(summary)
    summary.to_csv(TEST_DIR / "summary.csv")

# Run dell'esperimento C senza scaling factor per ablation study

## Training senza scaling

In [ ]:
import sys
import json
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import pandas as pd
from pathlib import Path

# 1. CONFIGURAZIONE ESPERIMENTO "NO SCALING" 
TIMESTAMP_EXP = pd.Timestamp.now().strftime("%Y_%m_%d_%H_%M")
EXP_NAME = "EXP_C_VARIANT_NO_SCALE" # Nome specifico per riconoscerlo
OUTPUT_DIR_NOSCALE = CONFIG.PERCORSO_LAVORO / f"RETRAINING_{TIMESTAMP_EXP}_{EXP_NAME}"
OUTPUT_DIR_NOSCALE.mkdir(parents=True, exist_ok=True)

# Setup Logger
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s | %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(OUTPUT_DIR_NOSCALE / "log_no_scale.txt")
    ], 
    force=True
)
logger = logging.getLogger()
logger.info(f"AVVIO ESPERIMENTO VARIANT: L2 NORM SENZA SCALING FACTOR")
logger.info(f"Output salvato in: {OUTPUT_DIR_NOSCALE}")

# 2. DEFINIZIONE MODELLO (L2 NORMALIZATION PURI, NO ALPHA) 
class ProtoNet_L2_NO_SCALE(nn.Module):
    """
    Versione L2 Normalization 'Pura'.
    I vettori hanno lunghezza 1.0. 
    Nessun fattore di scala (alpha) viene applicato prima della loss.
    """
    def __init__(self, pretrained=True, output_dim=256):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, output_dim)
        
        # NOTA: Manca self.scale rispetto alla versione standard

    def forward(self, x):
        out = self.projection_layer(self.backbone(x))
        
        # Applichiamo solo la normalizzazione L2
        # I vettori risultanti giacciono sulla ipersfera unitaria
        out = F.normalize(out, p=2, dim=1)
        
        # Ritorna out SENZA moltiplicazione
        return out

# 3. CARICAMENTO DATI 
if 'df_dati' not in locals():
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")

with open(CONFIG.CARTELLA_INDICI / "folds_manuali.json", "r") as f:
    dati_manuali = json.load(f)

# 4. LOOP DI ADDESTRAMENTO (5 FOLDS) 
# Usiamo la configurazione globale ma forziamo il modello NO_SCALE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_classes = list(range(8))

logger.info("\n" + "="*60)
logger.info("INIZIO TRAINING: PROTO NET L2 - NO SCALING")
logger.info("="*60)

for fold_item in dati_manuali:
    fold_idx = fold_item['fold_idx']
    novel_idx = tuple(fold_item['novel_classes_ids'])
    base_idx = tuple(c for c in all_classes if c not in novel_idx)
    
    logger.info(f"\nTraining Fold {fold_idx} | Novel Classes: {novel_idx}")
    
    # Eseguiamo il training usando la funzione esistente
    # Assicurati di aver aggiornato esegui_training_fold togliendo lo scheduler come discusso!
    esegui_training_fold(
        fold_idx=fold_idx,
        classi_base=base_idx,
        df_dati=df_dati,
        config=PROTO_CONFIG,
        device=device,
        ModelloClasse=ProtoNet_L2_NO_SCALE,  # <--- USIAMO LA CLASSE SENZA SCALING
        cartella_output_exp=OUTPUT_DIR_NOSCALE,
        usa_clahe_fold=False # Manteniamo false per confronto equo con EXP C
    )

logger.info("\n" + "="*60)
logger.info(f"ESPERIMENTO COMPLETATO.")
logger.info(f"Controlla i file 'fold_X_best_model.pth' in {OUTPUT_DIR_NOSCALE}")

## Testing senza scaling

In [ ]:
import sys
import gc
import json
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from pathlib import Path

# 1. CONFIGURAZIONE PATH E LOGGER 
if 'OUTPUT_DIR_NOSCALE' not in locals():
    # OUTPUT_DIR_NOSCALE = Path("/kaggle/working/RETRAINING_YYYY_MM_DD_EXP_C_VARIANT_NO_SCALE")
    raise ValueError("Variabile OUTPUT_DIR_NOSCALE non trovata. Definisci il percorso manualmente.")

TEST_OUTPUT_DIR = OUTPUT_DIR_NOSCALE / "test_results_5runs"
PLOT_DIR = TEST_OUTPUT_DIR / "confusion_matrices"

TEST_OUTPUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s | %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)], 
    force=True
)
logger = logging.getLogger()
logger.info(f"AVVIO TESTING + CM (5 RUNS) SU: {OUTPUT_DIR_NOSCALE}")

# 2. UTILITIES PER PLOTTING 
def plot_save_cm(y_true, y_pred, class_names, title, save_path):
    """Genera e salva una CM con annotazioni (Conteggio + Percentuale)"""
    cm = confusion_matrix(y_true, y_pred)
    
    # Calcolo percentuali per riga (Recall)
    with np.errstate(divide='ignore', invalid='ignore'):
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm)
    
    # Annotazioni complesse
    annot = [[f"{val}\n({pct:.1%})" for val, pct in zip(row_c, row_p)] 
             for row_c, row_p in zip(cm, cm_norm)]
    
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm_norm, annot=np.array(annot), fmt='', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Recall (Row Normalized)'}, 
                square=True, linewidths=1, linecolor='white')
    
    plt.title(title, fontsize=12, fontweight='bold', pad=15)
    plt.ylabel('Ground Truth', fontsize=10, fontweight='bold')
    plt.xlabel('Prediction', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

# 3. RIDEFINIZIONE MODELLO E TTA 
class ProtoNet_L2_NO_SCALE(nn.Module):
    def __init__(self, pretrained=False, output_dim=256):
        super().__init__()
        weights = None 
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, output_dim)

    def forward(self, x):
        out = self.projection_layer(self.backbone(x))
        return F.normalize(out, p=2, dim=1)

def forward_tta_no_scale(model, batch_img):
    b, c, h, w = batch_img.size()
    viste = [batch_img, torch.rot90(batch_img, 1, [2, 3]), torch.rot90(batch_img, 2, [2, 3]), torch.rot90(batch_img, 3, [2, 3])]
    img_flip = torch.flip(batch_img, [3])
    viste.extend([img_flip, torch.rot90(img_flip, 1, [2, 3]), torch.rot90(img_flip, 2, [2, 3]), torch.rot90(img_flip, 3, [2, 3])])
    
    aug_batch = torch.cat(viste, dim=0) 
    feat_aug = model(aug_batch)       
    feat_aug = feat_aug.view(8, b, -1)
    
    # Media e Ri-normalizzazione
    feat_avg = feat_aug.mean(dim=0)
    return F.normalize(feat_avg, p=2, dim=1)

# 4. PREPARAZIONE DATI 
if 'df_dati' not in locals():
    df_dati = pd.read_csv(CONFIG.CARTELLA_INDICI / "indice_livecell.csv")
with open(CONFIG.CARTELLA_INDICI / "folds_manuali.json", "r") as f:
    dati_manuali = json.load(f)
with open(CONFIG.CARTELLA_INDICI / "mappatura_classi.json", "r") as f:
    mappa_classi = json.load(f)
# Invertiamo la mappa per avere ID -> Nome
id_to_name = {v: k for k, v in mappa_classi.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_RUNS = 5
NUM_EPISODI = 600

results_noscale = []

# 5. LOOP DI TEST 
for fold_item in dati_manuali:
    fold_idx = fold_item['fold_idx']
    novel_ids = tuple(fold_item['novel_classes_ids'])
    desc = fold_item['descrizione']
    
    # Nomi classi per il grafico
    fold_class_names = [id_to_name[i] for i in sorted(novel_ids)]

    logger.info(f"--- Processing Fold {fold_idx}: {desc} ---")

    # 1. Carica Modello
    path_ckpt = OUTPUT_DIR_NOSCALE / f"fold_{fold_idx}_best_model.pth"
    if not path_ckpt.exists():
        logger.error(f"Pesi non trovati per Fold {fold_idx}. Skipping.")
        continue

    try:
        modello = ProtoNet_L2_NO_SCALE(pretrained=False).to(device)
        ckpt = torch.load(path_ckpt, map_location=device)
        modello.load_state_dict(ckpt['model_state_dict'])
        modello.eval()
    except Exception as e:
        logger.error(f"Errore caricamento modello Fold {fold_idx}: {e}")
        continue

    # 2. Dataset Test (Solo Novel Classes)
    df_test = filtra_dataset(df_dati, novel_ids)
    df_test = df_test[df_test["split"] == "test"].copy()
    ds_test = LIVECellDataset(df_test, ottieni_trasformazioni_config("val", usa_clahe=False))

    # Liste per accumulare TUTTE le predizioni di questo Fold (attraverso le 5 run)
    all_preds_fold = []
    all_targs_fold = []
    
    fold_accs = []
    fold_f1s = []

    # 3. Loop 5 Runs
    for run_i in range(NUM_RUNS):
        g = crea_generatore_riproducibile(42 + run_i)
        
        loader = DataLoader(
            ds_test, 
            batch_sampler=CampionatoreEpisodico(ds_test.etichette, PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY, NUM_EPISODI), 
            num_workers=2, 
            worker_init_fn=worker_init_fn, 
            generator=g
        )

        acc_sum = 0.0
        f1_sum = 0.0
        pbar = tqdm(loader, desc=f"Run {run_i+1}/{NUM_RUNS}", leave=False)

        with torch.no_grad():
            for bx, by in pbar:
                bx, by = bx.to(device), by.to(device)
                
                with torch.amp.autocast('cuda'):
                    emb = forward_tta_no_scale(modello, bx)
                    
                    _, logits, t_loc = protonet_loss(emb, by, PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY)
                    
                    # Otteniamo metriche e vettori globali
                    acc, _, _, f1, preds_glob, targs_glob = calcola_metriche(logits, t_loc, by, PROTO_CONFIG.TEST_N_WAY, PROTO_CONFIG.TEST_K_SHOT, PROTO_CONFIG.TEST_Q_QUERY)
                
                acc_sum += acc
                f1_sum += f1
                
                # Accumulo per CM
                all_preds_fold.extend(preds_glob.cpu().numpy())
                all_targs_fold.extend(targs_glob.cpu().numpy())

        mean_acc = acc_sum / len(loader)
        mean_f1 = f1_sum / len(loader)
        
        fold_accs.append(mean_acc)
        fold_f1s.append(mean_f1)
        
        results_noscale.append({
            "Experiment": "EXP_NO_SCALE",
            "Fold": fold_idx,
            "Run": run_i + 1,
            "Acc": mean_acc,
            "F1": mean_f1
        })

    # FINE FOLD: CALCOLO STATISTICHE E PLOT CM
    avg_acc_fold = np.mean(fold_accs)
    logger.info(f"  > Fold {fold_idx} Avg: Acc={avg_acc_fold:.2%} | F1={np.mean(fold_f1s):.4f}")
    
    # Generazione Confusion Matrix (Aggregata su 5 Run)
    plot_title = f"Exp No Scale - Fold {fold_idx}\n(Avg 5 Runs - Acc: {avg_acc_fold:.2%})"
    save_path = PLOT_DIR / f"fold_{fold_idx}_cm.png"
    
    plot_save_cm(
        y_true=all_targs_fold,
        y_pred=all_preds_fold,
        class_names=fold_class_names,
        title=plot_title,
        save_path=save_path
    )
    logger.info(f"  > Grafico salvato: {save_path.name}")

    # Cleanup
    del modello, ds_test, loader
    gc.collect()
    torch.cuda.empty_cache()

# 6. SALVATAGGIO CSV FINALE 
if results_noscale:
    df_res = pd.DataFrame(results_noscale)
    df_res.to_csv(TEST_OUTPUT_DIR / "results_noscale_detailed.csv", index=False)
    
    summary = df_res.groupby("Fold")[["Acc", "F1"]].agg(['mean', 'std'])
    summary.to_csv(TEST_OUTPUT_DIR / "summary_folds.csv")
    
    logger.info("\n" + "="*60)
    logger.info("REPORT FINALE (NO SCALE)")
    print(summary)
    logger.info("="*60)

# Visualizzazioni con percorsi locali eseguite in locale per non impegnare le risorse Kaggle: UMAP e KNN

# GRADCAM (Esperimento C)

In [ ]:
'''
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import torchvision.models as models
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image
from glob import glob

# 1. CONFIGURAZIONE E PARAMETRI VISIVI
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

BASE_PATH_MODELS = r"C:\Users\chris\Documents\0_ULTRABACKUP_DELL\UNISA_Magistrale\1_I anno - I semestre\Artificial intelligence and machine learning\Progetto_Tagliaferri\RETRAINING_2026_01_30_14_39\RETRAINING_2026_01_30_14_392026_01_30_14_39_C_L2_SCALE"
BASE_PATH_TEST = r"C:\Users\chris\Documents\livecell-dataset\test"
OUTPUT_FILE = f"GradCAM_RandomSample_Seed{RANDOM_SEED}_224px.jpg"

# PARAMETRI RIDIMENSIONATI PER 224x224
RENDER_SIZE = 224  # Risoluzione del modello
BANNER_H = 60
LABEL_H = 30
FONT_SCALE_H = 0.4
FONT_SCALE_B = 0.9
SPACER_W = 8
BORDER_W = 10

FOLD_MAP = {
    1: ['BT474', 'SkBr3'],
    2: ['A172', 'SHSY5Y'],
    3: ['BT474', 'MCF7'],
    4: ['A172', 'BV2'],
    5: ['MCF7', 'SKOV3']
}

FOLD_COLORS = {
    1: (40, 40, 140),
    2: (120, 60, 20),
    3: (40, 100, 40),
    4: (20, 100, 150),
    5: (100, 40, 100)
}

MEAN = [0.188, 0.188, 0.188]
STD = [0.250, 0.250, 0.250]


# 2. DEFINIZIONE DEL MODELLO
class L2Norm(nn.Module):
    def forward(self, x):
        return x / x.norm(p=2, dim=1, keepdim=True)


class LiveCellProtonet_ExpC(nn.Module):
    def __init__(self):
        super(LiveCellProtonet_ExpC, self).__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, 256)
        self.l2 = L2Norm()
        self.scale = nn.Parameter(torch.tensor(10.0))

    def forward(self, x):
        x = self.backbone(x)
        x = self.projection_layer(x)
        x = self.l2(x)
        x = x * self.scale
        return x


# 3. UTILS
def load_model_for_fold(fold_id, device):
    model = LiveCellProtonet_ExpC()
    path = os.path.join(BASE_PATH_MODELS, f'fold_{fold_id}_best_model.pth')
    if not os.path.exists(path):
        raise FileNotFoundError(f"Modello mancante: {path}")

    checkpoint = torch.load(path, map_location=device)
    clean_state = {k.replace("module.", ""): v for k, v in checkpoint.get('model_state_dict', checkpoint).items()}
    model.load_state_dict(clean_state, strict=True)
    model.to(device).eval()
    return model


def get_random_sample(class_name):
    folder_path = os.path.join(BASE_PATH_TEST, class_name)
    files = glob(os.path.join(folder_path, "*.tif")) + glob(os.path.join(folder_path, "*.png"))
    if not files:
        raise ValueError(f"No files for {class_name}")
    files.sort()  # Ensure deterministic sorting before random choice
    return np.random.choice(files, 1)[0]


def add_header(img, text, bg_color, text_color, font_scale, h):
    header = np.full((h, img.shape[1], 3), bg_color, dtype=np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    t_size = cv2.getTextSize(text, font, font_scale, 1)[0]

    x_pos = (header.shape[1] - t_size[0]) // 2
    y_pos = (h + t_size[1]) // 2

    cv2.putText(header, text, (x_pos, y_pos), font, font_scale, text_color, 1, cv2.LINE_AA)
    return np.vstack([header, img])


# 4. MAIN GENERATION
def create_grid_224():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    fold_columns = []

    print(f"Generating 224x224 Grid (Seed: {RANDOM_SEED})...")

    for fold_id in range(1, 6):
        novel_classes = FOLD_MAP[fold_id]
        model = load_model_for_fold(fold_id, device)
        cam = GradCAM(model=model, target_layers=[model.backbone.layer4[-1]])

        class_rows = []
        for class_name in novel_classes:
            # 1. Load & Preprocess
            img_path = get_random_sample(class_name)
            img_orig = cv2.imread(img_path, 1)

            # Resize a 224 per il modello E per il render (coincidono)
            img_224 = cv2.resize(img_orig, (RENDER_SIZE, RENDER_SIZE))
            img_float = np.float32(img_224[:, :, ::-1]) / 255
            input_tensor = preprocess_image(img_float, mean=MEAN, std=STD).to(device)

            # 2. GradCAM Calculation
            grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]

            # 3. Visualization
            # Resize CAM a 224 usando Cubica per smooth (da 7x7 a 224x224)
            cam_resized = cv2.resize(grayscale_cam, (RENDER_SIZE, RENDER_SIZE), interpolation=cv2.INTER_CUBIC)

            cam_overlay = show_cam_on_image(img_float, cam_resized, use_rgb=True)
            cam_overlay = cv2.cvtColor(cam_overlay, cv2.COLOR_RGB2BGR)

            # 4. Labeling (Scaled fonts)
            lbl_orig = add_header(img_224, f"{class_name}", (0, 0, 0), (255, 255, 255), FONT_SCALE_H, LABEL_H)
            lbl_cam = add_header(cam_overlay, "GradCAM", (0, 0, 0), (255, 255, 255), FONT_SCALE_H, LABEL_H)

            class_rows.append(np.hstack([lbl_orig, lbl_cam]))

        # Stack Fold Column
        col_content = np.vstack(class_rows)

        # Fold Banner
        banner = np.full((BANNER_H, col_content.shape[1], 3), FOLD_COLORS[fold_id], dtype=np.uint8)
        title = f"FOLD {fold_id}"
        font = cv2.FONT_HERSHEY_DUPLEX
        ts = cv2.getTextSize(title, font, FONT_SCALE_B, 2)[0]
        cv2.putText(banner, title, ((banner.shape[1] - ts[0]) // 2, (BANNER_H + ts[1]) // 2), font, FONT_SCALE_B,
                    (255, 255, 255), 2, cv2.LINE_AA)

        fold_columns.append(np.vstack([banner, col_content]))

    # Assembly
    spacer = np.full((fold_columns[0].shape[0], SPACER_W, 3), (255, 255, 255), dtype=np.uint8)
    final_wide = fold_columns[0]
    for i in range(1, 5):
        final_wide = np.hstack([final_wide, spacer, fold_columns[i]])

    final_wide = cv2.copyMakeBorder(final_wide, BORDER_W, BORDER_W, BORDER_W, BORDER_W, cv2.BORDER_CONSTANT,
                                    value=(250, 250, 250))

    cv2.imwrite(OUTPUT_FILE, final_wide, [cv2.IMWRITE_JPEG_QUALITY, 100])
    print(f"Saved: {OUTPUT_FILE} ({final_wide.shape[1]}x{final_wide.shape[0]})")


if __name__ == "__main__":
    create_grid_224()
'''

## K-Nearest Neighbours

In [ ]:
'''
import os
import sys
import json
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision.models import resnet18
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from pathlib import Path
from datetime import datetime

# 1. CONFIGURAZIONE

EXP_PATH = Path("/kaggle/working/esperimenti/2026_01_26_20_44_ProtoNet_TTA_8D4_ResNet18_Lin256_NO_Clahe2.0_NoNorm4W_Val2W_Test2W")
TIMESTAMP = datetime.now().strftime("%Y_%m_%d_%H_%M")
OUTPUT_DIR = Path(f"/kaggle/working/{TIMESTAMP}_EXP_B_Generalization_Analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = Path("/kaggle/working/indici/indice_livecell.csv")
MAP_PATH = Path("/kaggle/working/indici/mappatura_classi.json")

# Parametri ottimizzati
N_WAY = 8           
K_SHOT = 5          
Q_QUERY = 15        
NUM_EPISODES = 600  
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 4     # Massimo per Kaggle
PREFETCH = 2        # Carica i prossimi batch mentre la GPU lavora

MEAN_LC = (0.188, 0.188, 0.188)
STD_LC = (0.250, 0.250, 0.250)

print(f"Analisi Generalizzazione EXP B (SPEED OPTIMIZED)")
print(f"Output: {OUTPUT_DIR}")

# 2. DATASET

def get_transforms():
    return A.Compose([
        A.Resize(height=224, width=224, interpolation=cv2.INTER_CUBIC),
        A.Normalize(mean=MEAN_LC, std=STD_LC, max_pixel_value=255.0),
        ToTensorV2()
    ])

class LIVECellDataset(Dataset):
    def __init__(self, df, transforms):
        self.paths = df['percorso_immagine'].tolist()
        self.labels = df['indice_etichetta'].to_numpy(dtype=np.int64)
        self.transforms = transforms

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        aug = self.transforms(image=img)
        return aug["image"], label

class GlobalTestSampler(Sampler):
    def __init__(self, labels, k_shot, q_query, num_episodes):
        self.num_episodes = num_episodes
        self.unique_classes = np.sort(np.unique(labels))
        self.k_shot = k_shot
        self.q_query = q_query
        self.indices_per_class = {c: np.where(labels == c)[0] for c in self.unique_classes}

    def __len__(self):
        return self.num_episodes

    def __iter__(self):
        for _ in range(self.num_episodes):
            batch = []
            for c in self.unique_classes:
                indices = np.random.choice(self.indices_per_class[c], self.k_shot + self.q_query, replace=False)
                batch.extend(indices)
            yield batch

# 3. MODELLO E TTA

class ProtoResNet_Standard(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, 256) 
    def forward(self, x):
        return self.projection_layer(self.backbone(x))

def forward_tta_all(model, batch_img):
    """
    TTA ottimizzata per AMP.
    Input: [B, C, H, W] -> Output: [B, 256]
    """
    b, c, h, w = batch_img.size()
    views = []
    
    # Rotazioni
    views.append(batch_img)
    views.append(torch.rot90(batch_img, 1, [2, 3]))
    views.append(torch.rot90(batch_img, 2, [2, 3]))
    views.append(torch.rot90(batch_img, 3, [2, 3]))
    
    # Flips + Rotazioni
    img_flip = torch.flip(batch_img, [3])
    views.append(img_flip)
    views.append(torch.rot90(img_flip, 1, [2, 3]))
    views.append(torch.rot90(img_flip, 2, [2, 3]))
    views.append(torch.rot90(img_flip, 3, [2, 3]))
    
    aug_batch = torch.cat(views, dim=0) # [8*B, C, H, W] -> Circa 1280 immagini
    
    # Inferenza (l'autocast esterno gestirà FP16 qui)
    features = model(aug_batch)
    
    features = features.view(8, b, -1)
    return features.mean(dim=0)

def load_model_weights(model, path):
    checkpoint = torch.load(path, map_location=DEVICE)
    if 'model_state_dict' in checkpoint:
        st = checkpoint['model_state_dict']
    else:
        st = checkpoint
    new_st = {k.replace("module.", ""): v for k, v in st.items()}
    model.load_state_dict(new_st)
    model.eval()
    return model

# 4. LOOP DI VALUTAZIONE
def evaluate_dataset(model, loader):
    all_targets = []
    all_preds = []
    
    # Calcolo target statico (identico per ogni batch 8-way)
    # Spostato fuori dal loop per risparmiare micro-secondi
    static_targets = torch.arange(N_WAY, device=DEVICE).repeat_interleave(Q_QUERY)

    with torch.no_grad():
        for batch_img, _ in tqdm(loader, leave=False, desc="Inferenza TTA (AMP)"):
            batch_img = batch_img.to(DEVICE, non_blocking=True)
            
            # --- ACCELERAZIONE: Automatic Mixed Precision (AMP) ---
            with torch.amp.autocast('cuda'):
                # 1. Feature Extraction (TTA)
                embeddings = forward_tta_all(model, batch_img)
                
                # 2. ProtoNet Logic (8-Way)
                z = embeddings.view(N_WAY, K_SHOT + Q_QUERY, -1)
                prototypes = z[:, :K_SHOT].mean(dim=1)
                queries = z[:, K_SHOT:].contiguous().view(N_WAY * Q_QUERY, -1)
                
                # Distanze e Predizioni
                dists = torch.cdist(queries, prototypes)
                preds = dists.argmin(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(static_targets.cpu().numpy())
            
    return np.array(all_targets), np.array(all_preds)

def main():
    df_full = pd.read_csv(INDEX_PATH)
    df_test = df_full[df_full['split'] == 'test'].copy()
    
    with open(MAP_PATH) as f:
        class_map = json.load(f)
    id_to_name = {v: k for k, v in class_map.items()}
    class_names = [id_to_name[i] for i in range(8)]
    
    transforms = get_transforms()
    ds = LIVECellDataset(df_test, transforms)
    
    sampler = GlobalTestSampler(ds.labels, k_shot=K_SHOT, q_query=Q_QUERY, num_episodes=NUM_EPISODES)
    
    # OPTIMIZED DATALOADER
    loader = DataLoader(
        ds, 
        batch_sampler=sampler, 
        num_workers=NUM_WORKERS, 
        pin_memory=True,
        prefetch_factor=PREFETCH,
        persistent_workers=True # Mantiene i worker vivi tra le epoche
    )
    
    fold_accuracies = []
    fold_f1_scores = []
    global_targets = []
    global_preds = []
    
    for fold_idx in range(1, 6):
        weights_path = EXP_PATH / f"fold_{fold_idx}_best_model.pth"
        
        if not weights_path.exists():
            print(f"ERRORE: Pesi non trovati per Fold {fold_idx}")
            continue
            
        print(f"\n--- Valutazione Fold {fold_idx} ---")
        model = ProtoResNet_Standard().to(DEVICE)
        model = load_model_weights(model, weights_path)
        
        targs, preds = evaluate_dataset(model, loader)
        
        acc = accuracy_score(targs, preds)
        f1 = f1_score(targs, preds, average='weighted')
        
        print(f"Fold {fold_idx} -> Accuracy: {acc:.2%} | F1-Score: {f1:.4f}")
        
        fold_accuracies.append(acc)
        fold_f1_scores.append(f1)
        global_targets.extend(targs)
        global_preds.extend(preds)
        
        del model
        torch.cuda.empty_cache()
    
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    mean_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)
    
    print("\n" + "="*60)
    print("RISULTATI FINALI")
    print("="*60)
    print(f"Mean Accuracy: {mean_acc:.2%} (+/- {std_acc:.2%})")
    print(f"Mean F1 Score: {mean_f1:.4f} (+/- {std_f1:.4f})")
    print("-" * 60)
    
    # Report TXT
    with open(OUTPUT_DIR / "metrics_report.txt", "w") as f:
        f.write(f"Esperimento B - Generalizzazione Globale (Optimized)\n")
        f.write(f"Episodi: {NUM_EPISODES} - 8-Way\n\n")
        for i, (a, f1) in enumerate(zip(fold_accuracies, fold_f1_scores)):
            f.write(f"Fold {i+1}: Acc={a:.4f}, F1={f1:.4f}\n")
        f.write(f"\nAGGREGATO:\nMean Acc: {mean_acc:.4f}\nMean F1 : {mean_f1:.4f}\n")

    # Confusion Matrix
    cm = confusion_matrix(global_targets, global_preds)
    with np.errstate(divide='ignore', invalid='ignore'):
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm)
    
    plt.figure(figsize=(12, 10), facecolor='#f5f7f9')
    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    
    plt.title(f"Aggregated CM (EXP B)\nMean Acc: {mean_acc:.2%}", fontsize=15)
    plt.ylabel('Ground Truth', fontsize=12)
    plt.xlabel('Prediction', fontsize=12)
    plt.xticks(rotation=45)
    
    plot_path = OUTPUT_DIR / "Global_Aggregated_CM.png"
    plt.tight_layout()
    plt.savefig(plot_path, dpi=300)
    print(f"Grafico salvato: {plot_path}")

if __name__ == "__main__":
    main()

'''

## GIF UMAP

In [ ]:
'''import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import silhouette_score
import umap
import warnings
import random
import sys

# 1. CONFIGURAZIONE

# SELETTORE ARCHITETTURA
# Opzioni: "STANDARD", "L2_SCALE", "MLP"
MODEL_ARCH = "STANDARD"

# PERCORSI
DATASET_ROOT_DIR = Path(r"C:\Users\chris\Documents\livecell-dataset\test")

MODEL_WEIGHTS_PATH = Path(
    r"C:\Users\chris\Documents\0_ULTRABACKUP_DELL\UNISA_Magistrale\1_I anno - I semestre\Artificial intelligence and machine learning\Progetto_Tagliaferri\RETRAINING_2026_01_30_14_39\RETRAINING_2026_01_30_14_392026_01_30_14_39_B_STANDARD_BEST\fold_4_best_model.pth")

# MODEL_WEIGHTS_PATH = Path(r"C:\Users\chris\Documents\0_ULTRABACKUP_DELL\UNISA_Magistrale\1_I anno - I semestre\Artificial intelligence and machine learning\Progetto_Tagliaferri\RETRAINING_2026_01_30_14_39\RETRAINING_2026_01_30_14_392026_01_30_14_39_C_L2_SCALE\fold_4_best_model.pth")

OUTPUT_DIR = Path(r"C:\Users\chris\Documents\GitHub\livecell_few_shot_resnet18_aiml\0_slide\working_locale")
OUTPUT_GIF = OUTPUT_DIR / f"UMAP_3D_Fold4_{MODEL_ARCH}_BestConfig.gif"

# TARGET
TARGET_CLASSES = ["A172", "BV2"]
SAMPLES_PER_CLASS = 10000  # 10k per classe

# PARAMETRI UMAP
UMAP_N_NEIGHBORS = 30  
UMAP_MIN_DIST = 0.1  
UMAP_SPREAD = 2.0  

# ANIMAZIONE
FPS = 30
FRAMES = 1440 
ELEVATION = 30
BG_COLOR = '#f5f7f9'
TEXT_COLOR = '#4a749d'
GRID_COLOR = '#d1d8dd'
PALETTE = ["#85c1e9", "#f1948a"]  # Azzurro vs Rosso

# NORMALIZZAZIONE
LIVECELL_MEAN = np.array([0.188, 0.188, 0.188])
LIVECELL_STD = np.array([0.250, 0.250, 0.250])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 2. DATASET
class DirectFolderDataset(Dataset):
    def __init__(self, root_dir, classes, samples_limit=None):
        self.samples = []
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

        transform_list = [
            A.Resize(height=224, width=224, interpolation=cv2.INTER_CUBIC),
            A.Normalize(mean=LIVECELL_MEAN, std=LIVECELL_STD, max_pixel_value=255.0),
            ToTensorV2()
        ]
        self.transform = A.Compose(transform_list)

        print(f"Scansione cartelle in: {root_dir}")
        for class_name in classes:
            class_dir = root_dir / class_name
            if not class_dir.exists():
                raise FileNotFoundError(f"❌ Cartella non trovata: {class_dir}")

            valid_exts = ['*.png', '*.tif', '*.tiff', '*.jpg']
            images = []
            for ext in valid_exts:
                images.extend(list(class_dir.glob(ext)))

            print(f"   -> {class_name}: trovate {len(images)} immagini.")

            if samples_limit and len(images) > samples_limit:
                random.seed(42)
                images = random.sample(images, samples_limit)
                print(f"      (Selezionate {len(images)} per il rendering)")

            for img_path in images:
                self.samples.append((str(img_path), self.class_to_idx[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Cannot read: {path}")
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)


        aug = self.transform(image=img_rgb)
        return aug["image"], label


# 3. MODELLO (SELETTORE MULTI-ARCHITETTURA)
class ProtoNet_Standard(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, output_dim)

    def forward(self, x): return self.projection_layer(self.backbone(x))


class ProtoNet_L2Scale(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Linear(512, output_dim)
        self.scale = nn.Parameter(torch.tensor(10.0))

    def forward(self, x):
        out = self.projection_layer(self.backbone(x))
        out = F.normalize(out, p=2, dim=1)
        return out * self.scale


class ProtoNet_MLP(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.projection_layer = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(inplace=True),
            nn.Dropout(p=0.2), nn.Linear(512, output_dim)
        )

    def forward(self, x): return self.projection_layer(self.backbone(x))


def get_model(arch_type, device):
    print(f"Istanziazione architettura: {arch_type}")
    if arch_type == "STANDARD":
        return ProtoNet_Standard().to(device)
    elif arch_type == "L2_SCALE":
        return ProtoNet_L2Scale().to(device)
    elif arch_type == "MLP":
        return ProtoNet_MLP().to(device)
    else:
        raise ValueError(f"Architettura sconosciuta: {arch_type}")


def load_model_strict(model, path):
    checkpoint = torch.load(path, map_location=device)
    state_dict = checkpoint['model_state_dict'] if (
                isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint) else checkpoint
    try:
        model.load_state_dict(state_dict, strict=True)
        print("Pesi caricati con successo (Strict Mode).")
    except RuntimeError as e:
        print(f"\nERRORE: Mismatch Architettura/Pesi.\n{e}")
        exit(1)
    return model


# 4. MAIN
if __name__ == "__main__":
    try:
        print("=" * 80)
        print(f"UMAP 3D GIF GENERATOR | Fold 4 | {MODEL_ARCH} | N={UMAP_N_NEIGHBORS} D={UMAP_MIN_DIST}")
        print("=" * 80)

        # 1. Dataset
        dataset = DirectFolderDataset(DATASET_ROOT_DIR, TARGET_CLASSES, SAMPLES_PER_CLASS)
        dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0)

        # 2. Modello
        model = get_model(MODEL_ARCH, device)
        if not MODEL_WEIGHTS_PATH.exists():
            raise FileNotFoundError(f"Weights not found: {MODEL_WEIGHTS_PATH}")
        model = load_model_strict(model, MODEL_WEIGHTS_PATH)

        # 3. Estrazione Feature
        print("\n⏳ Estrazione Features...")
        model.eval()
        embeddings_list = []
        labels_list = []

        with torch.no_grad():
            for i, (imgs, lbls) in enumerate(dataloader):
                imgs = imgs.to(device)
                features = model(imgs)
                embeddings_list.append(features.cpu().numpy())
                labels_list.append(lbls.numpy())
                print(f"   Batch {i + 1}/{len(dataloader)}", end='\r')

        X = np.concatenate(embeddings_list)
        y = np.concatenate(labels_list)
        print(f"\n   Features estratte: {X.shape}")

        # Debug rapido
        if np.all(X == 0) or np.isnan(X).any():
            raise ValueError("ERRORE: Feature corrotte (tutti zeri o NaN).")

        # 4. UMAP 3D (Metrica Dinamica)
        metric = 'cosine' if MODEL_ARCH == "L2_SCALE" else 'euclidean'
        print(f"\nCalcolo UMAP 3D (Metric: {metric})...")

        reducer = umap.UMAP(
            n_components=3,
            n_neighbors=UMAP_N_NEIGHBORS,  # 30
            min_dist=UMAP_MIN_DIST,  # 0.1
            spread=UMAP_SPREAD,
            metric=metric,
            random_state=42
        )
        X_3d = reducer.fit_transform(X)
        print("   ✓ Proiezione completata.")

        # 5. Silhouette Score 3D
        print("\n📏 Calcolo 3D Silhouette Score...")
        sil_3d = silhouette_score(X_3d, y)
        print(f"   Score: {sil_3d:.4f}")

        # 6. Setup Scena 3D
        print("\nSetup Animazione...")
        fig = plt.figure(figsize=(12, 10))
        fig.patch.set_facecolor(BG_COLOR)
        ax = fig.add_subplot(111, projection='3d')
        ax.set_facecolor(BG_COLOR)

        # Plot punti
        for cls_idx, cls_name in enumerate(TARGET_CLASSES):
            mask = y == cls_idx
            punti = X_3d[mask]
            ax.scatter(punti[:, 0], punti[:, 1], punti[:, 2],
                       label=f"{cls_name} (n={len(punti)})",
                       c=PALETTE[cls_idx],
                       s=3, alpha=0.5, edgecolors='none')

        # Styling
        ax.grid(True, color=GRID_COLOR, linestyle='--', linewidth=0.5, alpha=0.5)
        ax.set_xticklabels([]);
        ax.set_yticklabels([]);
        ax.set_zticklabels([])
        ax.xaxis.pane.fill = False;
        ax.yaxis.pane.fill = False;
        ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor(BG_COLOR);
        ax.yaxis.pane.set_edgecolor(BG_COLOR);
        ax.zaxis.pane.set_edgecolor(BG_COLOR)

        # Titolo Tecnico
        ax.set_title(f"UMAP 3D Space: {MODEL_ARCH}\nN={UMAP_N_NEIGHBORS}, D={UMAP_MIN_DIST} | Sil. 3D: {sil_3d:.3f}",
                     fontsize=14, color=TEXT_COLOR, weight='bold', pad=20)

        # Legenda
        leg = ax.legend(fontsize=12, loc='upper right')
        leg.get_frame().set_facecolor(BG_COLOR)
        leg.get_frame().set_edgecolor(BG_COLOR)
        for text in leg.get_texts(): text.set_color(TEXT_COLOR)


        # 7. Rendering GIF
        def update(frame):
            # Rotazione completa 360 gradi
            angle = frame * (360 / FRAMES)
            ax.view_init(elev=ELEVATION, azim=angle)
            return fig,


        print(f"Generazione GIF ({FRAMES} frames @ {FPS} fps)...")
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

        ani = FuncAnimation(fig, update, frames=FRAMES, interval=1000 / FPS, blit=False)
        writer = PillowWriter(fps=FPS)
        ani.save(OUTPUT_GIF, writer=writer)

        print(f"\nGIF salvata in: {OUTPUT_GIF}")

    except Exception as e:
        print(f"\nERRORE CRITICO: {e}")
        import traceback

        traceback.print_exc()

    finally:
        plt.close('all')

'''